## (old) plk plane tilt of each frame

In [3]:
# -*- coding: utf-8 -*-
"""
Fit a 2-D plane to every perturbation frame in a BLO PKL file.

For each frame, the script fits

    z(x, y) = a*x + b*y + c

where:
    x is positive toward the right/east,
    y is positive upward/north,
    z is the perturbation amplitude.

Outputs:
    1. plane_fit_results.csv
    2. plane_fit_results.npz
    3. plane_angle_vs_frame.png
    4. plane_slope_vs_frame.png
    5. plane_angle_and_slope_vs_frame.png

Angle convention, using the image coordinates of the directional-keogram
script:
    0 degrees   = perturbation increases toward the right
    90 degrees  = perturbation increases toward the bottom of the image
    180 degrees = perturbation increases toward the left
    270 degrees = perturbation increases toward the top of the image

Angles are reported in the range [-180, 180).

Only the direction of maximum perturbation increase is saved, to avoid
redundancy. The opposite direction is simply the wrapped value of
(angle + 180) if needed.

The slope magnitude is sqrt(a^2 + b^2), in amplitude per km.
"""

import csv
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# ==================================================
# Settings
# ==================================================

# macOS Downloads folder
DOWNLOADS_DIR = Path.home() / "Downloads"

# Change only this filename when processing another PKL file.
PKL_FILE = DOWNLOADS_DIR / "processed_data_full_03_25_2025.pkl"

# Output folder inside Downloads
OUT_DIR = DOWNLOADS_DIR / "BLO_plane_fit_outputs_Mar_25_2025"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Spatial pixel sizes
DX_KM = 1.0
DY_KM = 1.0

# Time spacing between frames
DT_SECONDS = 120.0

# Plot settings
DPI = 200

# Optional fitting controls
# Set to an integer larger than 1 to fit every Nth pixel and speed up fitting.
# For example, FIT_PIXEL_STEP = 2 uses every second row and column.
FIT_PIXEL_STEP = 1

# Ignore non-finite pixels automatically.
# Optionally restrict values included in the fit:
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None

# Require at least this many valid pixels in a frame.
MIN_VALID_PIXELS = 100


# ==================================================
# Helper: locate and read perturbation cube
# ==================================================
def load_wave_cube(pkl_file):
    """
    Load the perturbation image cube from the expected BLO PKL structure.

    Expected entry:
        data[5] = ["Perturbation Images", perturbation_array]
    """
    if not pkl_file.exists():
        raise FileNotFoundError(
            f"PKL file not found:\n{pkl_file}\n\n"
            "Update PKL_FILE near the top of the script so that it matches "
            "the filename in your macOS Downloads folder."
        )

    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "Could not read data[5][1] from the PKL file. "
            "The PKL structure is different from the expected BLO format."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected a perturbation array with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")

    return wave_cube, data


# ==================================================
# Helper: fit one frame
# ==================================================
def fit_plane_to_frame(frame, x_grid_km, y_grid_km):
    """
    Fit z = a*x + b*y + c using ordinary least squares.

    Returns
    -------
    a : float
        East-west plane coefficient, amplitude per km.
    b : float
        North-south plane coefficient, amplitude per km.
    c : float
        Plane intercept.
    slope : float
        Gradient magnitude sqrt(a^2 + b^2), amplitude per km.
    angle_deg : float
        Direction of maximum perturbation increase in [-180, 180), using
        image coordinates: 0 degrees is right, +90 degrees is down, ±180
        degrees is left, and -90 degrees is up.
    rmse : float
        Root-mean-square residual of the plane fit.
    valid_count : int
        Number of pixels included in the fit.
    """
    frame_sample = frame[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN

    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))

    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    design_matrix = np.column_stack(
        [
            x_valid,
            y_valid,
            np.ones(valid_count, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_valid,
        rcond=None,
    )

    a, b, c = coefficients

    fitted_values = design_matrix @ coefficients
    residuals = z_valid - fitted_values
    rmse = np.sqrt(np.mean(residuals**2))

    slope = np.hypot(a, b)

    # Direction of maximum perturbation increase:
    #    0 degrees = right
    #   +90 degrees = down
    #   -90 degrees = up
    #   ±180 degrees = left
    #
    # Wrap to [-180, 180).
    angle_deg = ((np.degrees(np.arctan2(b, a)) + 180.0) % 360.0) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


# ==================================================
# Load data
# ==================================================
wave_cube, pkl_data = load_wave_cube(PKL_FILE)

nt, ny, nx = wave_cube.shape

frame_numbers = np.arange(nt, dtype=int)
times_seconds = frame_numbers * DT_SECONDS
times_minutes = times_seconds / 60.0


# ==================================================
# Spatial coordinates
# ==================================================
# Center coordinates on the middle of the image.
#
# x increases toward the right.
# y increases downward with image row number. This matches the coordinate
# convention used by make_directional_keogram() in the pasted script.
x_1d_km = (
    np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
) * DX_KM

y_1d_km = (
    np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
) * DY_KM

x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)


# ==================================================
# Fit every frame
# ==================================================
a_values = np.full(nt, np.nan, dtype=np.float64)
b_values = np.full(nt, np.nan, dtype=np.float64)
c_values = np.full(nt, np.nan, dtype=np.float64)
slope_values = np.full(nt, np.nan, dtype=np.float64)
angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
rmse_values = np.full(nt, np.nan, dtype=np.float64)
valid_pixel_counts = np.zeros(nt, dtype=int)

print("\nFitting a plane to each frame...")

for frame_index in range(nt):
    (
        a_values[frame_index],
        b_values[frame_index],
        c_values[frame_index],
        slope_values[frame_index],
        angle_values_deg[frame_index],
        rmse_values[frame_index],
        valid_pixel_counts[frame_index],
    ) = fit_plane_to_frame(
        wave_cube[frame_index],
        x_grid_km,
        y_grid_km,
    )

    if (
        frame_index == 0
        or frame_index == nt - 1
        or (frame_index + 1) % 25 == 0
    ):
        print(
            f"  Processed frame {frame_index + 1}/{nt}"
        )


# ==================================================
# Save CSV results
# ==================================================
csv_file = OUT_DIR / "plane_fit_results.csv"

with open(csv_file, "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(
        [
            "frame",
            "time_seconds",
            "time_minutes",
            "slope_x_amplitude_per_km",
            "slope_y_amplitude_per_km",
            "plane_intercept",
            "slope_magnitude_amplitude_per_km",
            "increase_direction_deg_wrapped_minus180_to_180_image_coordinates",
            "fit_rmse",
            "valid_pixel_count",
        ]
    )

    for frame_index in range(nt):
        writer.writerow(
            [
                frame_numbers[frame_index],
                times_seconds[frame_index],
                times_minutes[frame_index],
                a_values[frame_index],
                b_values[frame_index],
                c_values[frame_index],
                slope_values[frame_index],
                angle_values_deg[frame_index],
                rmse_values[frame_index],
                valid_pixel_counts[frame_index],
            ]
        )


# ==================================================
# Save NumPy results
# ==================================================
npz_file = OUT_DIR / "plane_fit_results.npz"

np.savez(
    npz_file,
    frame=frame_numbers,
    time_seconds=times_seconds,
    time_minutes=times_minutes,
    slope_x_amplitude_per_km=a_values,
    slope_y_amplitude_per_km=b_values,
    plane_intercept=c_values,
    slope_magnitude_amplitude_per_km=slope_values,
    increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
    fit_rmse=rmse_values,
    valid_pixel_count=valid_pixel_counts,
)


# ==================================================
# Plot angle versus frame
# ==================================================
fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

ax_angle.plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)

ax_angle.set_title("Plane-fit Direction Angle vs Frame")
ax_angle.set_xlabel("Frame")
ax_angle.set_ylabel(
    "Direction of perturbation increase "
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
ax_angle.set_ylim(-180, 180)
ax_angle.set_yticks(np.arange(-180, 181, 45))
ax_angle.grid(True, alpha=0.3)

fig_angle.tight_layout()

angle_plot_file = OUT_DIR / "plane_angle_vs_frame.png"

fig_angle.savefig(
    angle_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_angle)


# ==================================================
# Plot slope magnitude versus frame
# ==================================================
fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

ax_slope.plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)

ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
ax_slope.set_xlabel("Frame")
ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
ax_slope.grid(True, alpha=0.3)

fig_slope.tight_layout()

slope_plot_file = OUT_DIR / "plane_slope_vs_frame.png"

fig_slope.savefig(
    slope_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_slope)


# ==================================================
# Combined plot
# ==================================================
fig_combined, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 8),
    sharex=True,
)

axes[0].plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)
axes[0].set_title("Plane-fit Angle and Slope for Each Frame")
axes[0].set_ylabel(
    "Increase direction\n"
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
axes[0].set_ylim(-180, 180)
axes[0].set_yticks(np.arange(-180, 181, 45))
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
axes[1].grid(True, alpha=0.3)

fig_combined.tight_layout()

combined_plot_file = (
    OUT_DIR / "plane_angle_and_slope_vs_frame.png"
)

fig_combined.savefig(
    combined_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_combined)


# ==================================================
# Summary
# ==================================================
finite_fit_mask = (
    np.isfinite(slope_values)
    & np.isfinite(angle_values_deg)
)

print("\nFinished.")
print(f"Successful fits: {np.count_nonzero(finite_fit_mask)}/{nt}")
print(f"Input file: {PKL_FILE}")
print(f"Output folder: {OUT_DIR}")
print("\nSaved files:")
print(f"  {csv_file}")
print(f"  {npz_file}")
print(f"  {angle_plot_file}")
print(f"  {slope_plot_file}")
print(f"  {combined_plot_file}")

if np.any(finite_fit_mask):
    print(
        "\nMedian slope magnitude: "
        f"{np.nanmedian(slope_values):.6g} amplitude per km"
    )
    print(
        "Median angle: "
        f"{np.nanmedian(angle_values_deg):.3f} degrees"
    )


Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)

Fitting a plane to each frame...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed frame 150/280
  Processed frame 175/280
  Processed frame 200/280
  Processed frame 225/280
  Processed frame 250/280
  Processed frame 275/280
  Processed frame 280/280

Finished.
Successful fits: 280/280
Input file: /Users/anhphan/Downloads/processed_data_full_03_25_2025.pkl
Output folder: /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025

Saved files:
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.csv
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.npz
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_angle_vs_frame.png
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_slope_vs_frame.png
  /Users/anhphan/Downl

## (old) pkl plane tilt and animation

In [4]:
# -*- coding: utf-8 -*-
"""
Fit a 2-D plane to every perturbation frame in a BLO PKL file and create
an animation with the fitted increasing-direction vector overlaid as an arrow.

For each frame, the script fits

    z(x, y) = a*x + b*y + c

where:
    x increases to the right
    y increases downward (to match the image-coordinate convention
      used by the directional-keogram script)
    z is the perturbation amplitude

Outputs:
    1. plane_fit_results.csv
    2. plane_fit_results.npz
    3. plane_angle_vs_frame.png
    4. plane_slope_vs_frame.png
    5. plane_angle_and_slope_vs_frame.png
    6. plane_fit_vector_animation.mp4  (or GIF fallback if ffmpeg is unavailable)

Angle convention:
    0 degrees   = perturbation increases to the right
    +90 degrees = perturbation increases downward
    -90 degrees = perturbation increases upward
    ±180 degrees = perturbation increases to the left

Angles are wrapped to [-180, 180).

Only the direction of maximum perturbation increase is saved. The opposite
direction can be obtained from the wrapped value of (angle + 180) if needed.

The slope magnitude is sqrt(a^2 + b^2), in amplitude per km.

Note:
    The arrow length in the animation is a visual scaling of the fitted slope
    magnitude. It is not drawn in amplitude/km units on the image.
"""

import csv
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter


# ==================================================
# Settings
# ==================================================
DOWNLOADS_DIR = Path.home() / "Downloads"
PKL_FILE = DOWNLOADS_DIR / "processed_data_full_03_25_2025.pkl"

OUT_DIR = DOWNLOADS_DIR / "BLO_plane_fit_outputs_Mar_25_2025"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

# Arrow display settings
ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20

# Arrow length is scaled so that a high-percentile slope corresponds to
# a fraction of the image size.
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35


# ==================================================
# Helper functions
# ==================================================
def load_wave_cube(pkl_file):
    """
    Load the perturbation image cube from the expected BLO PKL structure.

    Expected entry:
        data[5] = ["Perturbation Images", perturbation_array]
    """
    if not pkl_file.exists():
        raise FileNotFoundError(
            f"PKL file not found:\n{pkl_file}\n\n"
            "Update PKL_FILE near the top of the script so that it matches "
            "the filename in your macOS Downloads folder."
        )

    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "Could not read data[5][1] from the PKL file. "
            "The PKL structure is different from the expected BLO format."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected a perturbation array with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")

    return wave_cube, data


def fit_plane_to_frame(frame, x_grid_km, y_grid_km):
    """
    Fit z = a*x + b*y + c using ordinary least squares.

    Returns
    -------
    a : float
        Plane coefficient in x, amplitude per km.
    b : float
        Plane coefficient in y, amplitude per km.
    c : float
        Plane intercept.
    slope : float
        Gradient magnitude sqrt(a^2 + b^2), amplitude per km.
    angle_deg : float
        Direction of maximum perturbation increase in [-180, 180), using
        image coordinates:
            0 degrees = right
           +90 degrees = down
           -90 degrees = up
           ±180 degrees = left
    rmse : float
        Root-mean-square residual of the plane fit.
    valid_count : int
        Number of pixels included in the fit.
    """
    frame_sample = frame[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN

    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))

    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    design_matrix = np.column_stack(
        [
            x_valid,
            y_valid,
            np.ones(valid_count, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_valid,
        rcond=None,
    )

    a, b, c = coefficients

    fitted_values = design_matrix @ coefficients
    residuals = z_valid - fitted_values
    rmse = np.sqrt(np.mean(residuals**2))

    slope = np.hypot(a, b)

    # Direction of maximum perturbation increase, wrapped to [-180, 180).
    angle_deg = ((np.degrees(np.arctan2(b, a)) + 180.0) % 360.0) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


# ==================================================
# Load data
# ==================================================
wave_cube, pkl_data = load_wave_cube(PKL_FILE)

nt, ny, nx = wave_cube.shape

frame_numbers = np.arange(nt, dtype=int)
times_seconds = frame_numbers * DT_SECONDS
times_minutes = times_seconds / 60.0


# ==================================================
# Spatial coordinates
# ==================================================
# x increases to the right
# y increases downward to match image coordinates
x_1d_km = (
    np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
) * DX_KM

y_1d_km = (
    np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
) * DY_KM

x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)


# ==================================================
# Fit every frame
# ==================================================
a_values = np.full(nt, np.nan, dtype=np.float64)
b_values = np.full(nt, np.nan, dtype=np.float64)
c_values = np.full(nt, np.nan, dtype=np.float64)
slope_values = np.full(nt, np.nan, dtype=np.float64)
angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
rmse_values = np.full(nt, np.nan, dtype=np.float64)
valid_pixel_counts = np.zeros(nt, dtype=int)

print("\nFitting a plane to each frame...")

for frame_index in range(nt):
    (
        a_values[frame_index],
        b_values[frame_index],
        c_values[frame_index],
        slope_values[frame_index],
        angle_values_deg[frame_index],
        rmse_values[frame_index],
        valid_pixel_counts[frame_index],
    ) = fit_plane_to_frame(
        wave_cube[frame_index],
        x_grid_km,
        y_grid_km,
    )

    if (
        frame_index == 0
        or frame_index == nt - 1
        or (frame_index + 1) % 25 == 0
    ):
        print(f"  Processed frame {frame_index + 1}/{nt}")


# ==================================================
# Save CSV results
# ==================================================
csv_file = OUT_DIR / "plane_fit_results.csv"

with open(csv_file, "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(
        [
            "frame",
            "time_seconds",
            "time_minutes",
            "slope_x_amplitude_per_km",
            "slope_y_amplitude_per_km",
            "plane_intercept",
            "slope_magnitude_amplitude_per_km",
            "increase_direction_deg_wrapped_minus180_to_180_image_coordinates",
            "fit_rmse",
            "valid_pixel_count",
        ]
    )

    for frame_index in range(nt):
        writer.writerow(
            [
                frame_numbers[frame_index],
                times_seconds[frame_index],
                times_minutes[frame_index],
                a_values[frame_index],
                b_values[frame_index],
                c_values[frame_index],
                slope_values[frame_index],
                angle_values_deg[frame_index],
                rmse_values[frame_index],
                valid_pixel_counts[frame_index],
            ]
        )


# ==================================================
# Save NumPy results
# ==================================================
npz_file = OUT_DIR / "plane_fit_results.npz"

np.savez(
    npz_file,
    frame=frame_numbers,
    time_seconds=times_seconds,
    time_minutes=times_minutes,
    slope_x_amplitude_per_km=a_values,
    slope_y_amplitude_per_km=b_values,
    plane_intercept=c_values,
    slope_magnitude_amplitude_per_km=slope_values,
    increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
    fit_rmse=rmse_values,
    valid_pixel_count=valid_pixel_counts,
)


# ==================================================
# Plot angle versus frame
# ==================================================
fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

ax_angle.plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)

ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
ax_angle.set_xlabel("Frame")
ax_angle.set_ylabel(
    "Direction of perturbation increase "
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
ax_angle.set_ylim(-180, 180)
ax_angle.set_yticks(np.arange(-180, 181, 45))
ax_angle.grid(True, alpha=0.3)

fig_angle.tight_layout()

angle_plot_file = OUT_DIR / "plane_angle_vs_frame.png"

fig_angle.savefig(
    angle_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_angle)


# ==================================================
# Plot slope magnitude versus frame
# ==================================================
fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

ax_slope.plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)

ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
ax_slope.set_xlabel("Frame")
ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
ax_slope.grid(True, alpha=0.3)

fig_slope.tight_layout()

slope_plot_file = OUT_DIR / "plane_slope_vs_frame.png"

fig_slope.savefig(
    slope_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_slope)


# ==================================================
# Combined plot
# ==================================================
fig_combined, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 8),
    sharex=True,
)

axes[0].plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)
axes[0].set_title("Plane-fit Increasing Direction and Slope for Each Frame")
axes[0].set_ylabel(
    "Increase direction\n"
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
axes[0].set_ylim(-180, 180)
axes[0].set_yticks(np.arange(-180, 181, 45))
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
axes[1].grid(True, alpha=0.3)

fig_combined.tight_layout()

combined_plot_file = OUT_DIR / "plane_angle_and_slope_vs_frame.png"

fig_combined.savefig(
    combined_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_combined)


# ==================================================
# Prepare arrow vectors for the animation
# ==================================================
finite_slope_mask = np.isfinite(slope_values)
finite_slopes = slope_values[finite_slope_mask]

if finite_slopes.size > 0:
    slope_ref = np.nanpercentile(
        finite_slopes,
        ARROW_LENGTH_PERCENTILE,
    )
else:
    slope_ref = 1.0

if not np.isfinite(slope_ref) or slope_ref <= 0:
    slope_ref = 1.0

max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

arrow_lengths_pix = np.zeros(nt, dtype=np.float64)
arrow_u = np.zeros(nt, dtype=np.float64)
arrow_v = np.zeros(nt, dtype=np.float64)

for frame_index in range(nt):
    if np.isfinite(slope_values[frame_index]) and np.isfinite(angle_values_deg[frame_index]):
        arrow_lengths_pix[frame_index] = (
            slope_values[frame_index] / slope_ref
        ) * max_arrow_length_pix

        theta_rad = np.deg2rad(angle_values_deg[frame_index])

        # In image coordinates: +x is right, +y is down.
        arrow_u[frame_index] = arrow_lengths_pix[frame_index] * np.cos(theta_rad)
        arrow_v[frame_index] = arrow_lengths_pix[frame_index] * np.sin(theta_rad)
    else:
        arrow_lengths_pix[frame_index] = 0.0
        arrow_u[frame_index] = 0.0
        arrow_v[frame_index] = 0.0


# ==================================================
# Create animation with vector arrow
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0
vmin = -vmax

fig_anim, ax_anim = plt.subplots(figsize=(7, 6))

im = ax_anim.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
)

cbar = fig_anim.colorbar(
    im,
    ax=ax_anim,
    fraction=0.046,
    pad=0.04,
)
cbar.set_label("Perturbation amplitude")

center_x = (nx - 1) / 2.0
center_y = (ny - 1) / 2.0

ax_anim.scatter(
    [center_x],
    [center_y],
    s=CENTER_MARKER_SIZE,
    c=CENTER_MARKER_COLOR,
    marker="o",
)

quiver = ax_anim.quiver(
    [center_x],
    [center_y],
    [arrow_u[0]],
    [arrow_v[0]],
    angles="xy",
    scale_units="xy",
    scale=1,
    color=ARROW_COLOR,
    width=0.008,
    headwidth=4.5,
    headlength=6.0,
    headaxislength=5.5,
)

title = ax_anim.set_title("")

info_text = ax_anim.text(
    0.02,
    0.98,
    "",
    transform=ax_anim.transAxes,
    va="top",
    ha="left",
    fontsize=10,
    color="black",
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.8,
        edgecolor="black",
    ),
)

ax_anim.set_xlabel("X pixel")
ax_anim.set_ylabel("Y pixel")


def update(frame_index):
    im.set_data(wave_cube[frame_index])

    quiver.set_UVC(
        [arrow_u[frame_index]],
        [arrow_v[frame_index]],
    )

    title.set_text(
        f"Plane-fit Vector Animation | Frame {frame_index}/{nt - 1} | "
        f"t = {times_minutes[frame_index]:.1f} min"
    )

    angle_here = angle_values_deg[frame_index]
    slope_here = slope_values[frame_index]
    a_here = a_values[frame_index]
    b_here = b_values[frame_index]

    if np.isfinite(angle_here) and np.isfinite(slope_here):
        info_text.set_text(
            f"Increase direction = {angle_here:+.1f}°\n"
            f"Slope magnitude = {slope_here:.4g} amp/km\n"
            f"a = {a_here:.4g} amp/km\n"
            f"b = {b_here:.4g} amp/km"
        )
    else:
        info_text.set_text("Invalid fit for this frame")

    return im, quiver, title, info_text


ani = FuncAnimation(
    fig_anim,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

animation_mp4_file = OUT_DIR / "plane_fit_vector_animation.mp4"
animation_gif_file = OUT_DIR / "plane_fit_vector_animation.gif"

saved_animation_file = None

try:
    writer = FFMpegWriter(
        fps=FPS,
        bitrate=3000,
    )
    ani.save(
        animation_mp4_file,
        writer=writer,
        dpi=DPI,
    )
    saved_animation_file = animation_mp4_file
except Exception as exc:
    print(
        "\nCould not save MP4 with ffmpeg. "
        "Falling back to GIF."
    )
    print(f"Reason: {exc}")

    gif_writer = PillowWriter(fps=FPS)
    ani.save(
        animation_gif_file,
        writer=gif_writer,
        dpi=DPI,
    )
    saved_animation_file = animation_gif_file

plt.close(fig_anim)


# ==================================================
# Summary
# ==================================================
successful_fits = np.count_nonzero(
    np.isfinite(slope_values) & np.isfinite(angle_values_deg)
)

print("\nFinished.")
print(f"Successful fits: {successful_fits}/{nt}")
print(f"Input file: {PKL_FILE}")
print(f"Output folder: {OUT_DIR}")

print("\nSaved files:")
print(f"  {csv_file}")
print(f"  {npz_file}")
print(f"  {angle_plot_file}")
print(f"  {slope_plot_file}")
print(f"  {combined_plot_file}")

if saved_animation_file is not None:
    print(f"  {saved_animation_file}")

if successful_fits > 0:
    print(
        "\nMedian slope magnitude: "
        f"{np.nanmedian(slope_values):.6g} amplitude per km"
    )
    print(
        "Median increasing direction: "
        f"{np.nanmedian(angle_values_deg):.3f} degrees"
    )


Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)

Fitting a plane to each frame...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed frame 150/280
  Processed frame 175/280
  Processed frame 200/280
  Processed frame 225/280
  Processed frame 250/280
  Processed frame 275/280
  Processed frame 280/280

Finished.
Successful fits: 280/280
Input file: /Users/anhphan/Downloads/processed_data_full_03_25_2025.pkl
Output folder: /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025

Saved files:
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.csv
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.npz
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_angle_vs_frame.png
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_slope_vs_frame.png
  /Users/anhphan/Downl

## Add 3D plot

In [7]:
# -*- coding: utf-8 -*-
"""
Fit a 2-D plane to every perturbation frame in a BLO PKL file and create
an animation with the fitted increasing-direction vector overlaid as an arrow.

For each frame, the script fits

    z(x, y) = a*x + b*y + c

where:
    x increases to the right
    y increases downward (to match the image-coordinate convention
      used by the directional-keogram script)
    z is the perturbation amplitude

Outputs:
    1. plane_fit_results.csv
    2. plane_fit_results.npz
    3. plane_angle_vs_frame.png
    4. plane_slope_vs_frame.png
    5. plane_angle_and_slope_vs_frame.png
    6. plane_tilt_vector_3d_vs_time.png
    7. plane_tilt_vector_2d_colored_by_time.png
    8. plane_fit_vector_animation.mp4  (or GIF fallback if ffmpeg is unavailable)

Angle convention:
    0 degrees   = perturbation increases to the right
    +90 degrees = perturbation increases downward
    -90 degrees = perturbation increases upward
    ±180 degrees = perturbation increases to the left

Angles are wrapped to [-180, 180).

Only the direction of maximum perturbation increase is saved. The opposite
direction can be obtained from the wrapped value of (angle + 180) if needed.

The slope magnitude is sqrt(a^2 + b^2), in amplitude per km.

Note:
    The arrow length in the animation is a visual scaling of the fitted slope
    magnitude. It is not drawn in amplitude/km units on the image.
"""

import csv
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter


# ==================================================
# Settings
# ==================================================
DOWNLOADS_DIR = Path.home() / "Downloads"
PKL_FILE = DOWNLOADS_DIR / "processed_data_full_03_25_2025.pkl"

OUT_DIR = DOWNLOADS_DIR / "BLO_plane_fit_outputs_Mar_25_2025"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

# Arrow display settings
ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20

# Arrow length is scaled so that a high-percentile slope corresponds to
# a fraction of the image size.
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

# Tilt-vector summary plot settings
# Draw one vector from the origin every N frames on the 2-D color-time plot.
# Increase this value if the plot is too crowded.
TILT_VECTOR_PLOT_STEP = 10



# ==================================================
# Helper functions
# ==================================================
def load_wave_cube(pkl_file):
    """
    Load the perturbation image cube from the expected BLO PKL structure.

    Expected entry:
        data[5] = ["Perturbation Images", perturbation_array]
    """
    if not pkl_file.exists():
        raise FileNotFoundError(
            f"PKL file not found:\n{pkl_file}\n\n"
            "Update PKL_FILE near the top of the script so that it matches "
            "the filename in your macOS Downloads folder."
        )

    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "Could not read data[5][1] from the PKL file. "
            "The PKL structure is different from the expected BLO format."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected a perturbation array with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")

    return wave_cube, data


def fit_plane_to_frame(frame, x_grid_km, y_grid_km):
    """
    Fit z = a*x + b*y + c using ordinary least squares.

    Returns
    -------
    a : float
        Plane coefficient in x, amplitude per km.
    b : float
        Plane coefficient in y, amplitude per km.
    c : float
        Plane intercept.
    slope : float
        Gradient magnitude sqrt(a^2 + b^2), amplitude per km.
    angle_deg : float
        Direction of maximum perturbation increase in [-180, 180), using
        image coordinates:
            0 degrees = right
           +90 degrees = down
           -90 degrees = up
           ±180 degrees = left
    rmse : float
        Root-mean-square residual of the plane fit.
    valid_count : int
        Number of pixels included in the fit.
    """
    frame_sample = frame[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN

    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))

    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    design_matrix = np.column_stack(
        [
            x_valid,
            y_valid,
            np.ones(valid_count, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_valid,
        rcond=None,
    )

    a, b, c = coefficients

    fitted_values = design_matrix @ coefficients
    residuals = z_valid - fitted_values
    rmse = np.sqrt(np.mean(residuals**2))

    slope = np.hypot(a, b)

    # Direction of maximum perturbation increase, wrapped to [-180, 180).
    angle_deg = ((np.degrees(np.arctan2(b, a)) + 180.0) % 360.0) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


# ==================================================
# Load data
# ==================================================
wave_cube, pkl_data = load_wave_cube(PKL_FILE)

nt, ny, nx = wave_cube.shape

frame_numbers = np.arange(nt, dtype=int)
times_seconds = frame_numbers * DT_SECONDS
times_minutes = times_seconds / 60.0


# ==================================================
# Spatial coordinates
# ==================================================
# x increases to the right
# y increases downward to match image coordinates
x_1d_km = (
    np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
) * DX_KM

y_1d_km = (
    np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
) * DY_KM

x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)


# ==================================================
# Fit every frame
# ==================================================
a_values = np.full(nt, np.nan, dtype=np.float64)
b_values = np.full(nt, np.nan, dtype=np.float64)
c_values = np.full(nt, np.nan, dtype=np.float64)
slope_values = np.full(nt, np.nan, dtype=np.float64)
angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
rmse_values = np.full(nt, np.nan, dtype=np.float64)
valid_pixel_counts = np.zeros(nt, dtype=int)

print("\nFitting a plane to each frame...")

for frame_index in range(nt):
    (
        a_values[frame_index],
        b_values[frame_index],
        c_values[frame_index],
        slope_values[frame_index],
        angle_values_deg[frame_index],
        rmse_values[frame_index],
        valid_pixel_counts[frame_index],
    ) = fit_plane_to_frame(
        wave_cube[frame_index],
        x_grid_km,
        y_grid_km,
    )

    if (
        frame_index == 0
        or frame_index == nt - 1
        or (frame_index + 1) % 25 == 0
    ):
        print(f"  Processed frame {frame_index + 1}/{nt}")


# ==================================================
# Save CSV results
# ==================================================
csv_file = OUT_DIR / "plane_fit_results.csv"

with open(csv_file, "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(
        [
            "frame",
            "time_seconds",
            "time_minutes",
            "slope_x_amplitude_per_km",
            "slope_y_amplitude_per_km",
            "plane_intercept",
            "slope_magnitude_amplitude_per_km",
            "increase_direction_deg_wrapped_minus180_to_180_image_coordinates",
            "fit_rmse",
            "valid_pixel_count",
        ]
    )

    for frame_index in range(nt):
        writer.writerow(
            [
                frame_numbers[frame_index],
                times_seconds[frame_index],
                times_minutes[frame_index],
                a_values[frame_index],
                b_values[frame_index],
                c_values[frame_index],
                slope_values[frame_index],
                angle_values_deg[frame_index],
                rmse_values[frame_index],
                valid_pixel_counts[frame_index],
            ]
        )


# ==================================================
# Save NumPy results
# ==================================================
npz_file = OUT_DIR / "plane_fit_results.npz"

np.savez(
    npz_file,
    frame=frame_numbers,
    time_seconds=times_seconds,
    time_minutes=times_minutes,
    slope_x_amplitude_per_km=a_values,
    slope_y_amplitude_per_km=b_values,
    plane_intercept=c_values,
    slope_magnitude_amplitude_per_km=slope_values,
    increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
    fit_rmse=rmse_values,
    valid_pixel_count=valid_pixel_counts,
)


# ==================================================
# Plot angle versus frame
# ==================================================
fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

ax_angle.plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)

ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
ax_angle.set_xlabel("Frame")
ax_angle.set_ylabel(
    "Direction of perturbation increase "
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
ax_angle.set_ylim(-180, 180)
ax_angle.set_yticks(np.arange(-180, 181, 45))
ax_angle.grid(True, alpha=0.3)

fig_angle.tight_layout()

angle_plot_file = OUT_DIR / "plane_angle_vs_frame.png"

fig_angle.savefig(
    angle_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_angle)


# ==================================================
# Plot slope magnitude versus frame
# ==================================================
fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

ax_slope.plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)

ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
ax_slope.set_xlabel("Frame")
ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
ax_slope.grid(True, alpha=0.3)

fig_slope.tight_layout()

slope_plot_file = OUT_DIR / "plane_slope_vs_frame.png"

fig_slope.savefig(
    slope_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_slope)


# ==================================================
# Combined plot
# ==================================================
fig_combined, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 8),
    sharex=True,
)

axes[0].plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)
axes[0].set_title("Plane-fit Increasing Direction and Slope for Each Frame")
axes[0].set_ylabel(
    "Increase direction\n"
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
axes[0].set_ylim(-180, 180)
axes[0].set_yticks(np.arange(-180, 181, 45))
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
axes[1].grid(True, alpha=0.3)

fig_combined.tight_layout()

combined_plot_file = OUT_DIR / "plane_angle_and_slope_vs_frame.png"

fig_combined.savefig(
    combined_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_combined)


# ==================================================
# Prepare the shared display vectors
# ==================================================
# These are the exact visually scaled vector components used in:
#   1. the frame animation,
#   2. the 2-D time-colored vector plot, and
#   3. the 3-D vector-versus-time plot.
#
# Therefore, all three outputs use the same direction convention and
# the same relative vector lengths.
finite_slope_mask = np.isfinite(slope_values)
finite_slopes = slope_values[finite_slope_mask]

if finite_slopes.size > 0:
    slope_ref = np.nanpercentile(
        finite_slopes,
        ARROW_LENGTH_PERCENTILE,
    )
else:
    slope_ref = 1.0

if not np.isfinite(slope_ref) or slope_ref <= 0:
    slope_ref = 1.0

max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

arrow_lengths_pix = np.zeros(nt, dtype=np.float64)
arrow_u = np.zeros(nt, dtype=np.float64)
arrow_v = np.zeros(nt, dtype=np.float64)

for frame_index in range(nt):
    if (
        np.isfinite(slope_values[frame_index])
        and np.isfinite(angle_values_deg[frame_index])
    ):
        arrow_lengths_pix[frame_index] = (
            slope_values[frame_index] / slope_ref
        ) * max_arrow_length_pix

        theta_rad = np.deg2rad(angle_values_deg[frame_index])

        # Image-coordinate convention used by the animation:
        #   +u = right
        #   +v = down
        arrow_u[frame_index] = (
            arrow_lengths_pix[frame_index] * np.cos(theta_rad)
        )
        arrow_v[frame_index] = (
            arrow_lengths_pix[frame_index] * np.sin(theta_rad)
        )


# Common x-y limits for both tilt-vector plots.
# Equal symmetric limits guarantee a 1:1 x-y aspect ratio and prevent
# vector directions from being visually distorted.
finite_display_vector = (
    np.isfinite(arrow_u)
    & np.isfinite(arrow_v)
    & np.isfinite(times_minutes)
)

if np.any(finite_display_vector):
    vector_limit = np.nanmax(
        np.abs(
            np.concatenate(
                [
                    arrow_u[finite_display_vector],
                    arrow_v[finite_display_vector],
                ]
            )
        )
    )
else:
    vector_limit = 1.0

if not np.isfinite(vector_limit) or vector_limit <= 0:
    vector_limit = 1.0

vector_limit *= 1.10


# ==================================================
# 3-D displayed tilt vector versus time
# ==================================================
# X and Y are the same displayed vector components used in the animation.
# The y-axis is inverted so +Y points downward, matching imshow(origin="upper").
fig_tilt_3d = plt.figure(figsize=(8, 8))
ax_tilt_3d = fig_tilt_3d.add_subplot(111, projection="3d")

if np.any(finite_display_vector):
    u_finite = arrow_u[finite_display_vector]
    v_finite = arrow_v[finite_display_vector]
    time_finite = times_minutes[finite_display_vector]

    ax_tilt_3d.plot(
        u_finite,
        v_finite,
        time_finite,
        linewidth=1.0,
        alpha=0.55,
    )

    scatter_3d = ax_tilt_3d.scatter(
        u_finite,
        v_finite,
        time_finite,
        c=time_finite,
        cmap="viridis",
        s=18,
        depthshade=False,
    )

    cbar_3d = fig_tilt_3d.colorbar(
        scatter_3d,
        ax=ax_tilt_3d,
        pad=0.10,
        shrink=0.72,
    )
    cbar_3d.set_label("Time (minutes)")

ax_tilt_3d.set_xlim(-vector_limit, vector_limit)
ax_tilt_3d.set_ylim(vector_limit, -vector_limit)
ax_tilt_3d.set_box_aspect((1, 1, 1))

ax_tilt_3d.set_title("Displayed Plane-fit Vector vs Time")
ax_tilt_3d.set_xlabel("Displayed X component, u (pixels; right positive)")
ax_tilt_3d.set_ylabel("Displayed Y component, v (pixels; down positive)")
ax_tilt_3d.set_zlabel("Time (minutes)")
ax_tilt_3d.grid(True, alpha=0.3)

fig_tilt_3d.tight_layout()

tilt_vector_3d_file = OUT_DIR / "plane_tilt_vector_3d_vs_time.png"

fig_tilt_3d.savefig(
    tilt_vector_3d_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_tilt_3d)


# ==================================================
# 2-D displayed tilt-vector endpoints colored by time
# ==================================================
# This plot uses exactly the same arrow endpoints as the animation:
#   endpoint = (arrow_u, arrow_v)
#
# The y-axis increases downward, just as it does in the image animation.
fig_tilt_2d, ax_tilt_2d = plt.subplots(figsize=(8, 8))

if np.any(finite_display_vector):
    u_finite = arrow_u[finite_display_vector]
    v_finite = arrow_v[finite_display_vector]
    time_finite = times_minutes[finite_display_vector]

    ax_tilt_2d.plot(
        u_finite,
        v_finite,
        linewidth=0.8,
        alpha=0.30,
    )

    scatter_2d = ax_tilt_2d.scatter(
        u_finite,
        v_finite,
        c=time_finite,
        cmap="viridis",
        s=26,
        zorder=3,
    )

    cbar_2d = fig_tilt_2d.colorbar(
        scatter_2d,
        ax=ax_tilt_2d,
        pad=0.02,
    )
    cbar_2d.set_label("Time (minutes)")

    selected_indices = np.flatnonzero(
        finite_display_vector
    )[::TILT_VECTOR_PLOT_STEP]

    if selected_indices.size > 0:
        ax_tilt_2d.quiver(
            np.zeros(selected_indices.size),
            np.zeros(selected_indices.size),
            arrow_u[selected_indices],
            arrow_v[selected_indices],
            angles="xy",
            scale_units="xy",
            scale=1.0,
            color="0.35",
            width=0.003,
            alpha=0.45,
            zorder=2,
        )

ax_tilt_2d.axhline(
    0.0,
    color="0.5",
    linewidth=0.8,
    alpha=0.6,
)
ax_tilt_2d.axvline(
    0.0,
    color="0.5",
    linewidth=0.8,
    alpha=0.6,
)
ax_tilt_2d.scatter(
    [0.0],
    [0.0],
    s=30,
    c="black",
    marker="o",
    zorder=4,
)

ax_tilt_2d.set_xlim(-vector_limit, vector_limit)
ax_tilt_2d.set_ylim(vector_limit, -vector_limit)
ax_tilt_2d.set_aspect("equal", adjustable="box")

ax_tilt_2d.set_title(
    "Displayed Plane-fit Vector Endpoints Colored by Time"
)
ax_tilt_2d.set_xlabel(
    "Displayed X component, u (pixels; right positive)"
)
ax_tilt_2d.set_ylabel(
    "Displayed Y component, v (pixels; down positive)"
)
ax_tilt_2d.grid(True, alpha=0.3)

fig_tilt_2d.tight_layout()

tilt_vector_2d_file = (
    OUT_DIR / "plane_tilt_vector_2d_colored_by_time.png"
)

fig_tilt_2d.savefig(
    tilt_vector_2d_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_tilt_2d)


# ==================================================
# Create animation with vector arrow
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0
vmin = -vmax

fig_anim, ax_anim = plt.subplots(figsize=(7, 6))

im = ax_anim.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
)

cbar = fig_anim.colorbar(
    im,
    ax=ax_anim,
    fraction=0.046,
    pad=0.04,
)
cbar.set_label("Perturbation amplitude")

center_x = (nx - 1) / 2.0
center_y = (ny - 1) / 2.0

ax_anim.scatter(
    [center_x],
    [center_y],
    s=CENTER_MARKER_SIZE,
    c=CENTER_MARKER_COLOR,
    marker="o",
)

quiver = ax_anim.quiver(
    [center_x],
    [center_y],
    [arrow_u[0]],
    [arrow_v[0]],
    angles="xy",
    scale_units="xy",
    scale=1,
    color=ARROW_COLOR,
    width=0.008,
    headwidth=4.5,
    headlength=6.0,
    headaxislength=5.5,
)

title = ax_anim.set_title("")

info_text = ax_anim.text(
    0.02,
    0.98,
    "",
    transform=ax_anim.transAxes,
    va="top",
    ha="left",
    fontsize=10,
    color="black",
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.8,
        edgecolor="black",
    ),
)

ax_anim.set_xlabel("X pixel")
ax_anim.set_ylabel("Y pixel")


def update(frame_index):
    im.set_data(wave_cube[frame_index])

    quiver.set_UVC(
        [arrow_u[frame_index]],
        [arrow_v[frame_index]],
    )

    title.set_text(
        f"Plane-fit Vector Animation | Frame {frame_index}/{nt - 1} | "
        f"t = {times_minutes[frame_index]:.1f} min"
    )

    angle_here = angle_values_deg[frame_index]
    slope_here = slope_values[frame_index]
    a_here = a_values[frame_index]
    b_here = b_values[frame_index]

    if np.isfinite(angle_here) and np.isfinite(slope_here):
        info_text.set_text(
            f"Increase direction = {angle_here:+.1f}°\n"
            f"Slope magnitude = {slope_here:.4g} amp/km\n"
            f"a = {a_here:.4g} amp/km\n"
            f"b = {b_here:.4g} amp/km"
        )
    else:
        info_text.set_text("Invalid fit for this frame")

    return im, quiver, title, info_text


ani = FuncAnimation(
    fig_anim,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

animation_mp4_file = OUT_DIR / "plane_fit_vector_animation.mp4"
animation_gif_file = OUT_DIR / "plane_fit_vector_animation.gif"

saved_animation_file = None

try:
    writer = FFMpegWriter(
        fps=FPS,
        bitrate=3000,
    )
    ani.save(
        animation_mp4_file,
        writer=writer,
        dpi=DPI,
    )
    saved_animation_file = animation_mp4_file
except Exception as exc:
    print(
        "\nCould not save MP4 with ffmpeg. "
        "Falling back to GIF."
    )
    print(f"Reason: {exc}")

    gif_writer = PillowWriter(fps=FPS)
    ani.save(
        animation_gif_file,
        writer=gif_writer,
        dpi=DPI,
    )
    saved_animation_file = animation_gif_file

plt.close(fig_anim)


# ==================================================
# Summary
# ==================================================
successful_fits = np.count_nonzero(
    np.isfinite(slope_values) & np.isfinite(angle_values_deg)
)

print("\nFinished.")
print(f"Successful fits: {successful_fits}/{nt}")
print(f"Input file: {PKL_FILE}")
print(f"Output folder: {OUT_DIR}")

print("\nSaved files:")
print(f"  {csv_file}")
print(f"  {npz_file}")
print(f"  {angle_plot_file}")
print(f"  {slope_plot_file}")
print(f"  {combined_plot_file}")
print(f"  {tilt_vector_3d_file}")
print(f"  {tilt_vector_2d_file}")

if saved_animation_file is not None:
    print(f"  {saved_animation_file}")

if successful_fits > 0:
    print(
        "\nMedian slope magnitude: "
        f"{np.nanmedian(slope_values):.6g} amplitude per km"
    )
    print(
        "Median increasing direction: "
        f"{np.nanmedian(angle_values_deg):.3f} degrees"
    )


Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)

Fitting a plane to each frame...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed frame 150/280
  Processed frame 175/280
  Processed frame 200/280
  Processed frame 225/280
  Processed frame 250/280
  Processed frame 275/280
  Processed frame 280/280

Finished.
Successful fits: 280/280
Input file: /Users/anhphan/Downloads/processed_data_full_03_25_2025.pkl
Output folder: /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025

Saved files:
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.csv
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.npz
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_angle_vs_frame.png
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_slope_vs_frame.png
  /Users/anhphan/Downl

## Filter and fit the 3D curve

In [19]:
# -*- coding: utf-8 -*-
"""
Fit a 2-D plane to every perturbation frame in a BLO PKL file and create
an animation with the fitted increasing-direction vector overlaid as an arrow.

For each frame, the script fits

    z(x, y) = a*x + b*y + c

where:
    x increases to the right
    y increases downward (to match the image-coordinate convention
      used by the directional-keogram script)
    z is the perturbation amplitude

Outputs:
    1. plane_fit_results.csv
    2. plane_fit_results.npz
    3. plane_angle_vs_frame.png
    4. plane_slope_vs_frame.png
    5. plane_angle_and_slope_vs_frame.png
    6. plane_tilt_vector_3d_vs_time.png
    7. plane_tilt_vector_2d_whole_time_colored_by_time.png
    8. plane_tilt_vector_2d_zoom_100_to_500_min_colored_by_time.png
    9. plane_fit_spline_anchor_points.csv
    10. plane_fit_spline_anchor_points.npz
    11. plane_fit_vector_animation.mp4  (or GIF fallback if ffmpeg is unavailable)

Angle convention:
    0 degrees   = perturbation increases to the right
    +90 degrees = perturbation increases downward
    -90 degrees = perturbation increases upward
    ±180 degrees = perturbation increases to the left

Angles are wrapped to [-180, 180).

Only the direction of maximum perturbation increase is saved. The opposite
direction can be obtained from the wrapped value of (angle + 180) if needed.

The slope magnitude is sqrt(a^2 + b^2), in amplitude per km.

Note:
    The arrow length in the animation is a visual scaling of the fitted slope
    magnitude. It is not drawn in amplitude/km units on the image.
"""

import csv
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from scipy.signal import savgol_filter


# ==================================================
# Settings
# ==================================================
DOWNLOADS_DIR = Path.home() / "Downloads"
PKL_FILE = DOWNLOADS_DIR / "processed_data_full_03_25_2025.pkl"

OUT_DIR = DOWNLOADS_DIR / "BLO_plane_fit_outputs_Mar_25_2025"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

# Arrow display settings
ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20

# Arrow length is scaled so that a high-percentile slope corresponds to
# a fraction of the image size.
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

# Tilt-vector summary plot settings
# Draw one vector from the origin every N frames on the 2-D color-time plot.
# Increase this value if the plot is too crowded.
TILT_VECTOR_PLOT_STEP = 10

# Smooth the displayed vector components before selecting spline control points.
# The window must be odd. It is automatically reduced if there are too few frames.
CURVE_FILTER_WINDOW_FRAMES = 31
CURVE_FILTER_POLYORDER = 3

CURVE_CONTROL_POINT_COUNT = 6

# Rounded-polyline settings. Most of each connection remains straight, while
# a short quadratic Bezier section rounds each interior anchor.
# Smaller CORNER_FRACTION values make the path straighter.
CURVE_CORNER_FRACTION = 0.10
CURVE_POINTS_PER_STRAIGHT = 180
CURVE_POINTS_PER_CORNER = 80

# Exact requested rounded-path anchor times. Each time is mapped to the nearest
# available fitted frame inside the zoom interval.
CURVE_ANCHOR_TIMES_MIN = np.array([155.0, 225.0, 295.0, 365.0, 435.0, 500.0])

# Fractional padding around the spline-fit vector range in the summary plots.
CURVE_PLOT_PADDING_FRACTION = 0.12

# Zoomed time range used for filtering, anchor-point selection, and spline fitting.
CURVE_START_TIME_MIN = 120.0
CURVE_END_TIME_MIN = 500.0


# ==================================================
# Helper functions
# ==================================================
def make_rounded_polyline(
    times,
    x_values,
    y_values,
    corner_fraction=0.10,
    points_per_straight=180,
    points_per_corner=80,
):
    """
    Connect anchor points with mostly straight segments and rounded joints.

    The straight portions end shortly before each interior anchor. A quadratic
    Bezier curve then rounds the joint and continues onto the next straight
    portion. Decrease corner_fraction to make the path straighter.

    Parameters
    ----------
    times : array-like
        Anchor times in minutes.
    x_values, y_values : array-like
        Anchor vector components in amplitude per 100 km.
    corner_fraction : float
        Fraction of each adjacent segment used to round an interior joint.
        Must lie between 0 and 0.49.
    points_per_straight : int
        Number of samples along each straight portion.
    points_per_corner : int
        Number of samples along each rounded joint.

    Returns
    -------
    time_curve, x_curve, y_curve : ndarray
        Coordinates of the rounded path.
    """
    times = np.asarray(times, dtype=np.float64)
    x_values = np.asarray(x_values, dtype=np.float64)
    y_values = np.asarray(y_values, dtype=np.float64)

    if not (times.size == x_values.size == y_values.size):
        raise ValueError("times, x_values, and y_values must have equal lengths.")

    if times.size < 2:
        return times.copy(), x_values.copy(), y_values.copy()

    if np.any(np.diff(times) <= 0):
        raise ValueError("Anchor times must be strictly increasing.")

    corner_fraction = float(np.clip(corner_fraction, 0.0, 0.49))
    points_per_straight = max(int(points_per_straight), 2)
    points_per_corner = max(int(points_per_corner), 2)

    # Work in three dimensions: [time, x slope, y slope]. This keeps time
    # monotonic while the vector endpoint follows the rounded 2-D path.
    points = np.column_stack([times, x_values, y_values])
    curve_parts = []
    current_point = points[0]

    for index in range(1, len(points) - 1):
        previous_anchor = points[index - 1]
        corner_anchor = points[index]
        next_anchor = points[index + 1]

        corner_start = corner_anchor - corner_fraction * (
            corner_anchor - previous_anchor
        )
        corner_end = corner_anchor + corner_fraction * (
            next_anchor - corner_anchor
        )

        straight_parameter = np.linspace(
            0.0,
            1.0,
            points_per_straight,
            endpoint=False,
        )
        straight_section = (
            current_point[None, :]
            + straight_parameter[:, None]
            * (corner_start - current_point)[None, :]
        )
        curve_parts.append(straight_section)

        corner_parameter = np.linspace(
            0.0,
            1.0,
            points_per_corner,
            endpoint=False,
        )
        one_minus = 1.0 - corner_parameter
        rounded_corner = (
            one_minus[:, None] ** 2 * corner_start[None, :]
            + 2.0
            * one_minus[:, None]
            * corner_parameter[:, None]
            * corner_anchor[None, :]
            + corner_parameter[:, None] ** 2 * corner_end[None, :]
        )
        curve_parts.append(rounded_corner)
        current_point = corner_end

    final_parameter = np.linspace(
        0.0,
        1.0,
        points_per_straight,
    )
    final_section = (
        current_point[None, :]
        + final_parameter[:, None]
        * (points[-1] - current_point)[None, :]
    )
    curve_parts.append(final_section)

    full_curve = np.vstack(curve_parts)
    return full_curve[:, 0], full_curve[:, 1], full_curve[:, 2]


def load_wave_cube(pkl_file):
    """
    Load the perturbation image cube from the expected BLO PKL structure.

    Expected entry:
        data[5] = ["Perturbation Images", perturbation_array]
    """
    if not pkl_file.exists():
        raise FileNotFoundError(
            f"PKL file not found:\n{pkl_file}\n\n"
            "Update PKL_FILE near the top of the script so that it matches "
            "the filename in your macOS Downloads folder."
        )

    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "Could not read data[5][1] from the PKL file. "
            "The PKL structure is different from the expected BLO format."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected a perturbation array with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")

    return wave_cube, data


def fit_plane_to_frame(frame, x_grid_km, y_grid_km):
    """
    Fit z = a*x + b*y + c using ordinary least squares.

    Returns
    -------
    a : float
        Plane coefficient in x, amplitude per km.
    b : float
        Plane coefficient in y, amplitude per km.
    c : float
        Plane intercept.
    slope : float
        Gradient magnitude sqrt(a^2 + b^2), amplitude per km.
    angle_deg : float
        Direction of maximum perturbation increase in [-180, 180), using
        image coordinates:
            0 degrees = right
           +90 degrees = down
           -90 degrees = up
           ±180 degrees = left
    rmse : float
        Root-mean-square residual of the plane fit.
    valid_count : int
        Number of pixels included in the fit.
    """
    frame_sample = frame[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN

    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))

    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    design_matrix = np.column_stack(
        [
            x_valid,
            y_valid,
            np.ones(valid_count, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_valid,
        rcond=None,
    )

    a, b, c = coefficients

    fitted_values = design_matrix @ coefficients
    residuals = z_valid - fitted_values
    rmse = np.sqrt(np.mean(residuals**2))

    slope = np.hypot(a, b)

    # Direction of maximum perturbation increase, wrapped to [-180, 180).
    angle_deg = ((np.degrees(np.arctan2(b, a)) + 180.0) % 360.0) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


# ==================================================
# Load data
# ==================================================
wave_cube, pkl_data = load_wave_cube(PKL_FILE)

nt, ny, nx = wave_cube.shape

frame_numbers = np.arange(nt, dtype=int)
times_seconds = frame_numbers * DT_SECONDS
times_minutes = times_seconds / 60.0


# ==================================================
# Spatial coordinates
# ==================================================
# x increases to the right
# y increases downward to match image coordinates
x_1d_km = (
    np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
) * DX_KM

y_1d_km = (
    np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
) * DY_KM

x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)


# ==================================================
# Fit every frame
# ==================================================
a_values = np.full(nt, np.nan, dtype=np.float64)
b_values = np.full(nt, np.nan, dtype=np.float64)
c_values = np.full(nt, np.nan, dtype=np.float64)
slope_values = np.full(nt, np.nan, dtype=np.float64)
angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
rmse_values = np.full(nt, np.nan, dtype=np.float64)
valid_pixel_counts = np.zeros(nt, dtype=int)

print("\nFitting a plane to each frame...")

for frame_index in range(nt):
    (
        a_values[frame_index],
        b_values[frame_index],
        c_values[frame_index],
        slope_values[frame_index],
        angle_values_deg[frame_index],
        rmse_values[frame_index],
        valid_pixel_counts[frame_index],
    ) = fit_plane_to_frame(
        wave_cube[frame_index],
        x_grid_km,
        y_grid_km,
    )

    if (
        frame_index == 0
        or frame_index == nt - 1
        or (frame_index + 1) % 25 == 0
    ):
        print(f"  Processed frame {frame_index + 1}/{nt}")


# ==================================================
# Save CSV results
# ==================================================
csv_file = OUT_DIR / "plane_fit_results.csv"

with open(csv_file, "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(
        [
            "frame",
            "time_seconds",
            "time_minutes",
            "slope_x_amplitude_per_km",
            "slope_y_amplitude_per_km",
            "plane_intercept",
            "slope_magnitude_amplitude_per_km",
            "increase_direction_deg_wrapped_minus180_to_180_image_coordinates",
            "fit_rmse",
            "valid_pixel_count",
        ]
    )

    for frame_index in range(nt):
        writer.writerow(
            [
                frame_numbers[frame_index],
                times_seconds[frame_index],
                times_minutes[frame_index],
                a_values[frame_index],
                b_values[frame_index],
                c_values[frame_index],
                slope_values[frame_index],
                angle_values_deg[frame_index],
                rmse_values[frame_index],
                valid_pixel_counts[frame_index],
            ]
        )


# ==================================================
# Save NumPy results
# ==================================================
npz_file = OUT_DIR / "plane_fit_results.npz"

np.savez(
    npz_file,
    frame=frame_numbers,
    time_seconds=times_seconds,
    time_minutes=times_minutes,
    slope_x_amplitude_per_km=a_values,
    slope_y_amplitude_per_km=b_values,
    plane_intercept=c_values,
    slope_magnitude_amplitude_per_km=slope_values,
    increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
    fit_rmse=rmse_values,
    valid_pixel_count=valid_pixel_counts,
)


# ==================================================
# Plot angle versus frame
# ==================================================
fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

ax_angle.plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)

ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
ax_angle.set_xlabel("Frame")
ax_angle.set_ylabel(
    "Direction of perturbation increase "
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
ax_angle.set_ylim(-180, 180)
ax_angle.set_yticks(np.arange(-180, 181, 45))
ax_angle.grid(True, alpha=0.3)

fig_angle.tight_layout()

angle_plot_file = OUT_DIR / "plane_angle_vs_frame.png"

fig_angle.savefig(
    angle_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_angle)


# ==================================================
# Plot slope magnitude versus frame
# ==================================================
fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

ax_slope.plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)

ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
ax_slope.set_xlabel("Frame")
ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
ax_slope.grid(True, alpha=0.3)

fig_slope.tight_layout()

slope_plot_file = OUT_DIR / "plane_slope_vs_frame.png"

fig_slope.savefig(
    slope_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_slope)


# ==================================================
# Combined plot
# ==================================================
fig_combined, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 8),
    sharex=True,
)

axes[0].plot(
    frame_numbers,
    angle_values_deg,
    linewidth=1.3,
)
axes[0].set_title("Plane-fit Increasing Direction and Slope for Each Frame")
axes[0].set_ylabel(
    "Increase direction\n"
    "(-180° to 180°; 0° right, 90° down, -90° up)"
)
axes[0].set_ylim(-180, 180)
axes[0].set_yticks(np.arange(-180, 181, 45))
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    frame_numbers,
    slope_values,
    linewidth=1.3,
)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
axes[1].grid(True, alpha=0.3)

fig_combined.tight_layout()

combined_plot_file = OUT_DIR / "plane_angle_and_slope_vs_frame.png"

fig_combined.savefig(
    combined_plot_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_combined)


# ==================================================
# Prepare animation vectors and physical tilt vectors
# ==================================================
# Physical plane-gradient components used by the 2-D and 3-D tilt-vector plots.
# Units are perturbation amplitude per 100 km.
x_slope_per_100km = 100.0 * a_values
y_slope_per_100km = 100.0 * b_values
slope_magnitude_per_100km = 100.0 * slope_values

finite_physical_vector = (
    np.isfinite(x_slope_per_100km)
    & np.isfinite(y_slope_per_100km)
    & np.isfinite(times_minutes)
)

# The animation still needs pixel-scaled arrows because it is drawn on an image.
# Only this animation-specific block uses displayed pixel lengths.
finite_slope_mask = np.isfinite(slope_values)
finite_slopes = slope_values[finite_slope_mask]

if finite_slopes.size > 0:
    slope_ref = np.nanpercentile(
        finite_slopes,
        ARROW_LENGTH_PERCENTILE,
    )
else:
    slope_ref = 1.0

if not np.isfinite(slope_ref) or slope_ref <= 0:
    slope_ref = 1.0

max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

arrow_u = np.zeros(nt, dtype=np.float64)
arrow_v = np.zeros(nt, dtype=np.float64)

for frame_index in range(nt):
    if (
        np.isfinite(a_values[frame_index])
        and np.isfinite(b_values[frame_index])
    ):
        arrow_u[frame_index] = (
            a_values[frame_index] / slope_ref
        ) * max_arrow_length_pix
        arrow_v[frame_index] = (
            b_values[frame_index] / slope_ref
        ) * max_arrow_length_pix


# ==================================================
# Remove high-frequency fluctuations and build a rounded path
# ==================================================
curve_time_mask = (
    finite_physical_vector
    & (times_minutes >= CURVE_START_TIME_MIN)
    & (times_minutes <= CURVE_END_TIME_MIN)
)

valid_indices = np.flatnonzero(curve_time_mask)

curve_available = False
control_indices = np.array([], dtype=int)
control_times = np.array([], dtype=np.float64)
control_u = np.array([], dtype=np.float64)
control_v = np.array([], dtype=np.float64)
time_curve = np.array([], dtype=np.float64)
u_curve = np.array([], dtype=np.float64)
v_curve = np.array([], dtype=np.float64)

if valid_indices.size >= CURVE_CONTROL_POINT_COUNT:
    time_valid = times_minutes[valid_indices]
    u_valid = x_slope_per_100km[valid_indices]
    v_valid = y_slope_per_100km[valid_indices]

    filter_window = min(
        CURVE_FILTER_WINDOW_FRAMES,
        valid_indices.size,
    )

    if filter_window % 2 == 0:
        filter_window -= 1

    minimum_window = CURVE_FILTER_POLYORDER + 2
    if minimum_window % 2 == 0:
        minimum_window += 1

    if filter_window >= minimum_window:
        u_filtered = savgol_filter(
            u_valid,
            window_length=filter_window,
            polyorder=CURVE_FILTER_POLYORDER,
            mode="interp",
        )
        v_filtered = savgol_filter(
            v_valid,
            window_length=filter_window,
            polyorder=CURVE_FILTER_POLYORDER,
            mode="interp",
        )
    else:
        u_filtered = u_valid.copy()
        v_filtered = v_valid.copy()

    if CURVE_ANCHOR_TIMES_MIN.size != CURVE_CONTROL_POINT_COUNT:
        raise ValueError(
            "CURVE_ANCHOR_TIMES_MIN must contain exactly "
            f"{CURVE_CONTROL_POINT_COUNT} times."
        )

    if np.any(np.diff(CURVE_ANCHOR_TIMES_MIN) <= 0):
        raise ValueError(
            "CURVE_ANCHOR_TIMES_MIN must be strictly increasing."
        )

    if (
        CURVE_ANCHOR_TIMES_MIN[0] < CURVE_START_TIME_MIN
        or CURVE_ANCHOR_TIMES_MIN[-1] > CURVE_END_TIME_MIN
    ):
        raise ValueError(
            "All CURVE_ANCHOR_TIMES_MIN values must lie inside "
            "CURVE_START_TIME_MIN and CURVE_END_TIME_MIN."
        )

    # Select the nearest valid fitted frame for each requested anchor time.
    control_positions = np.array(
        [
            int(np.argmin(np.abs(time_valid - requested_time)))
            for requested_time in CURVE_ANCHOR_TIMES_MIN
        ],
        dtype=int,
    )

    if np.unique(control_positions).size != CURVE_CONTROL_POINT_COUNT:
        raise ValueError(
            "Two requested anchor times mapped to the same frame. "
            "Separate the anchor times or decrease DT_SECONDS."
        )

    control_indices = valid_indices[control_positions]
    control_times = times_minutes[control_indices]
    control_u = u_filtered[control_positions]
    control_v = v_filtered[control_positions]

    time_curve, u_curve, v_curve = make_rounded_polyline(
        control_times,
        control_u,
        control_v,
        corner_fraction=CURVE_CORNER_FRACTION,
        points_per_straight=CURVE_POINTS_PER_STRAIGHT,
        points_per_corner=CURVE_POINTS_PER_CORNER,
    )

    curve_available = True


# ==================================================
# Save the rounded-path anchor points in physical gradient units
# ==================================================
anchor_csv_file = OUT_DIR / "plane_fit_spline_anchor_points.csv"
anchor_npz_file = OUT_DIR / "plane_fit_spline_anchor_points.npz"

if curve_available:
    with open(anchor_csv_file, "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow(
            [
                "anchor_number",
                "frame",
                "time_minutes",
                "slope_x_amplitude_per_km",
                "slope_y_amplitude_per_km",
                "slope_magnitude_amplitude_per_km",
                "slope_x_amplitude_per_100km",
                "slope_y_amplitude_per_100km",
                "slope_magnitude_amplitude_per_100km",
                "increase_direction_deg_image_coordinates",
                "displayed_u_pixels",
                "displayed_v_pixels",
            ]
        )

        for anchor_number, frame_index in enumerate(
            control_indices,
            start=1,
        ):
            u_here = control_u[anchor_number - 1]
            v_here = control_v[anchor_number - 1]

            # The spline is fitted directly in amplitude per 100 km.
            slope_x_here = u_here / 100.0
            slope_y_here = v_here / 100.0
            slope_magnitude_here = np.hypot(
                slope_x_here,
                slope_y_here,
            )

            direction_here = (
                (
                    np.degrees(
                        np.arctan2(
                            slope_y_here,
                            slope_x_here,
                        )
                    )
                    + 180.0
                )
                % 360.0
            ) - 180.0

            writer.writerow(
                [
                    anchor_number,
                    frame_index,
                    times_minutes[frame_index],
                    slope_x_here,
                    slope_y_here,
                    slope_magnitude_here,
                    100.0 * slope_x_here,
                    100.0 * slope_y_here,
                    100.0 * slope_magnitude_here,
                    direction_here,
                    arrow_u[frame_index],
                    arrow_v[frame_index],
                ]
            )

    np.savez(
        anchor_npz_file,
        anchor_number=np.arange(
            1,
            CURVE_CONTROL_POINT_COUNT + 1,
            dtype=int,
        ),
        frame=control_indices,
        time_minutes=control_times,
        slope_x_amplitude_per_km=control_u / 100.0,
        slope_y_amplitude_per_km=control_v / 100.0,
        slope_magnitude_amplitude_per_km=(
            np.hypot(control_u, control_v) / 100.0
        ),
        slope_x_amplitude_per_100km=control_u,
        slope_y_amplitude_per_100km=control_v,
        slope_magnitude_amplitude_per_100km=np.hypot(
            control_u,
            control_v,
        ),
        increase_direction_deg_image_coordinates=(
            (
                np.degrees(
                    np.arctan2(
                        control_v,
                        control_u,
                    )
                )
                + 180.0
            )
            % 360.0
        )
        - 180.0,
        displayed_u_pixels=(control_u / 100.0 / slope_ref) * max_arrow_length_pix,
        displayed_v_pixels=(control_v / 100.0 / slope_ref) * max_arrow_length_pix,
    )


if not curve_available:
    print(
        "\nWarning: fewer than "
        f"{CURVE_CONTROL_POINT_COUNT} valid frames were found between "
        f"{CURVE_START_TIME_MIN:.1f} and "
        f"{CURVE_END_TIME_MIN:.1f} minutes. "
        "The rounded path and anchor-point files were not created."
    )


# ==================================================
# Determine vector-axis limits
# ==================================================
def calculate_vector_limits(x_values, y_values, padding_fraction):
    """Return x limits and image-style reversed y limits."""
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = np.asarray(x_values)[valid]
    y_values = np.asarray(y_values)[valid]

    if x_values.size == 0:
        return (-1.0, 1.0), (1.0, -1.0)

    x_min = np.nanmin(x_values)
    x_max = np.nanmax(x_values)
    y_min = np.nanmin(y_values)
    y_max = np.nanmax(y_values)

    x_span = x_max - x_min
    y_span = y_max - y_min

    if not np.isfinite(x_span) or x_span <= 0:
        x_span = max(abs(x_min), abs(x_max), 1.0e-6)
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = max(abs(y_min), abs(y_max), 1.0e-6)

    x_padding = padding_fraction * x_span
    y_padding = padding_fraction * y_span

    x_limits = (x_min - x_padding, x_max + x_padding)
    y_limits = (y_max + y_padding, y_min - y_padding)
    return x_limits, y_limits


whole_time_mask = finite_physical_vector

whole_vector_x_limits, whole_vector_y_limits = calculate_vector_limits(
    x_slope_per_100km[whole_time_mask],
    y_slope_per_100km[whole_time_mask],
    CURVE_PLOT_PADDING_FRACTION,
)

if curve_available:
    zoom_x_for_limits = np.concatenate(
        [x_slope_per_100km[curve_time_mask], control_u, u_curve]
    )
    zoom_y_for_limits = np.concatenate(
        [y_slope_per_100km[curve_time_mask], control_v, v_curve]
    )
else:
    zoom_x_for_limits = x_slope_per_100km[curve_time_mask]
    zoom_y_for_limits = y_slope_per_100km[curve_time_mask]

zoom_vector_x_limits, zoom_vector_y_limits = calculate_vector_limits(
    zoom_x_for_limits,
    zoom_y_for_limits,
    CURVE_PLOT_PADDING_FRACTION,
)


# ==================================================
# 3-D physical tilt vector versus time

# ==================================================
# X and Y are the same displayed vector components used in the animation.
# The y-axis is inverted so +Y points downward, matching imshow(origin="upper").
fig_tilt_3d = plt.figure(figsize=(8, 8))
ax_tilt_3d = fig_tilt_3d.add_subplot(111, projection="3d")

if np.any(curve_time_mask):
    u_finite = x_slope_per_100km[curve_time_mask]
    v_finite = y_slope_per_100km[curve_time_mask]
    time_finite = times_minutes[curve_time_mask]

    ax_tilt_3d.plot(
        u_finite,
        v_finite,
        time_finite,
        linewidth=1.0,
        alpha=0.55,
    )

    scatter_3d = ax_tilt_3d.scatter(
        u_finite,
        v_finite,
        time_finite,
        c=time_finite,
        cmap="viridis",
        s=18,
        depthshade=False,
    )

    cbar_3d = fig_tilt_3d.colorbar(
        scatter_3d,
        ax=ax_tilt_3d,
        pad=0.10,
        shrink=0.72,
    )
    cbar_3d.set_label("Time (minutes)")

if curve_available:
    ax_tilt_3d.plot(
        u_curve,
        v_curve,
        time_curve,
        color="black",
        linewidth=2.2,
        label=f"Rounded path through {CURVE_CONTROL_POINT_COUNT} filtered points",
        zorder=5,
    )

    ax_tilt_3d.scatter(
        control_u,
        control_v,
        control_times,
        color="black",
        marker="x",
        s=70,
        linewidths=2.0,
        label=f"{CURVE_CONTROL_POINT_COUNT} anchor points",
        depthshade=False,
        zorder=6,
    )

    ax_tilt_3d.legend()

ax_tilt_3d.set_xlim(*zoom_vector_x_limits)
ax_tilt_3d.set_ylim(*zoom_vector_y_limits)
ax_tilt_3d.set_box_aspect((1, 1, 1))

ax_tilt_3d.set_title("Plane-fit Tilt Vector vs Time")
ax_tilt_3d.set_xlabel("X slope (amplitude per 100 km; right positive)")
ax_tilt_3d.set_ylabel("Y slope (amplitude per 100 km; down positive)")
ax_tilt_3d.set_zlabel("Time (minutes)")
ax_tilt_3d.grid(True, alpha=0.3)

fig_tilt_3d.tight_layout()

tilt_vector_3d_file = OUT_DIR / "plane_tilt_vector_3d_vs_time.png"

fig_tilt_3d.savefig(
    tilt_vector_3d_file,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_tilt_3d)


# ==================================================
# 2-D physical tilt-vector endpoints colored by time
# ==================================================
def save_tilt_vector_2d_plot(
    time_mask,
    output_file,
    title_text,
    x_limits,
    y_limits,
    include_spline=False,
):
    """Save one physical 2-D tilt-vector endpoint plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    selected_all = np.flatnonzero(time_mask)

    if selected_all.size > 0:
        u_plot = x_slope_per_100km[selected_all]
        v_plot = y_slope_per_100km[selected_all]
        time_plot = times_minutes[selected_all]

        ax.plot(
            u_plot,
            v_plot,
            linewidth=0.8,
            alpha=0.30,
        )

        scatter = ax.scatter(
            u_plot,
            v_plot,
            c=time_plot,
            cmap="viridis",
            s=26,
            zorder=3,
        )

        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=0.02,
        )
        cbar.set_label("Time (minutes)")

        vector_indices = selected_all[::TILT_VECTOR_PLOT_STEP]

        if vector_indices.size > 0:
            ax.quiver(
                np.zeros(vector_indices.size),
                np.zeros(vector_indices.size),
                x_slope_per_100km[vector_indices],
                y_slope_per_100km[vector_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

    if include_spline and curve_available:
        ax.plot(
            u_curve,
            v_curve,
            color="black",
            linewidth=2.2,
            label="Rounded path through selected anchor times",
            zorder=5,
        )

        ax.scatter(
            control_u,
            control_v,
            color="black",
            marker="x",
            s=70,
            linewidths=2.0,
            label="Selected anchor times",
            zorder=6,
        )

        for anchor_time, anchor_u, anchor_v in zip(
            control_times,
            control_u,
            control_v,
        ):
            ax.annotate(
                f"{anchor_time:.0f} min",
                xy=(anchor_u, anchor_v),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )

        ax.legend()

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(title_text)
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


tilt_vector_2d_whole_file = (
    OUT_DIR / "plane_tilt_vector_2d_whole_time_colored_by_time.png"
)

tilt_vector_2d_zoom_file = (
    OUT_DIR
    / f"plane_tilt_vector_2d_zoom_{CURVE_START_TIME_MIN:.0f}_to_"
      f"{CURVE_END_TIME_MIN:.0f}_min_colored_by_time.png"
)

save_tilt_vector_2d_plot(
    time_mask=whole_time_mask,
    output_file=tilt_vector_2d_whole_file,
    title_text="Plane-fit Tilt Vector Endpoints: Whole Time Range",
    x_limits=whole_vector_x_limits,
    y_limits=whole_vector_y_limits,
    include_spline=False,
)

save_tilt_vector_2d_plot(
    time_mask=curve_time_mask,
    output_file=tilt_vector_2d_zoom_file,
    title_text=(
        "Plane-fit Tilt Vector Endpoints: "
        f"{CURVE_START_TIME_MIN:.0f}–{CURVE_END_TIME_MIN:.0f} Minutes"
    ),
    x_limits=zoom_vector_x_limits,
    y_limits=zoom_vector_y_limits,
    include_spline=True,
)


# ==================================================
# Create animation with vector arrow
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0
vmin = -vmax

fig_anim, ax_anim = plt.subplots(figsize=(7, 6))

im = ax_anim.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
)

cbar = fig_anim.colorbar(
    im,
    ax=ax_anim,
    fraction=0.046,
    pad=0.04,
)
cbar.set_label("Perturbation amplitude")

center_x = (nx - 1) / 2.0
center_y = (ny - 1) / 2.0

ax_anim.scatter(
    [center_x],
    [center_y],
    s=CENTER_MARKER_SIZE,
    c=CENTER_MARKER_COLOR,
    marker="o",
)

quiver = ax_anim.quiver(
    [center_x],
    [center_y],
    [arrow_u[0]],
    [arrow_v[0]],
    angles="xy",
    scale_units="xy",
    scale=1,
    color=ARROW_COLOR,
    width=0.008,
    headwidth=4.5,
    headlength=6.0,
    headaxislength=5.5,
)

title = ax_anim.set_title("")

info_text = ax_anim.text(
    0.02,
    0.98,
    "",
    transform=ax_anim.transAxes,
    va="top",
    ha="left",
    fontsize=10,
    color="black",
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.8,
        edgecolor="black",
    ),
)

ax_anim.set_xlabel("X pixel")
ax_anim.set_ylabel("Y pixel")


def update(frame_index):
    im.set_data(wave_cube[frame_index])

    quiver.set_UVC(
        [arrow_u[frame_index]],
        [arrow_v[frame_index]],
    )

    title.set_text(
        f"Plane-fit Vector Animation | Frame {frame_index}/{nt - 1} | "
        f"t = {times_minutes[frame_index]:.1f} min"
    )

    angle_here = angle_values_deg[frame_index]
    slope_here = slope_values[frame_index]
    a_here = a_values[frame_index]
    b_here = b_values[frame_index]

    if np.isfinite(angle_here) and np.isfinite(slope_here):
        info_text.set_text(
            f"Increase direction = {angle_here:+.1f}°\n"
            f"Slope magnitude = {slope_here:.4g} amp/km\n"
            f"a = {a_here:.4g} amp/km\n"
            f"b = {b_here:.4g} amp/km"
        )
    else:
        info_text.set_text("Invalid fit for this frame")

    return im, quiver, title, info_text


ani = FuncAnimation(
    fig_anim,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

animation_mp4_file = OUT_DIR / "plane_fit_vector_animation.mp4"
animation_gif_file = OUT_DIR / "plane_fit_vector_animation.gif"

saved_animation_file = None

try:
    writer = FFMpegWriter(
        fps=FPS,
        bitrate=3000,
    )
    ani.save(
        animation_mp4_file,
        writer=writer,
        dpi=DPI,
    )
    saved_animation_file = animation_mp4_file
except Exception as exc:
    print(
        "\nCould not save MP4 with ffmpeg. "
        "Falling back to GIF."
    )
    print(f"Reason: {exc}")

    gif_writer = PillowWriter(fps=FPS)
    ani.save(
        animation_gif_file,
        writer=gif_writer,
        dpi=DPI,
    )
    saved_animation_file = animation_gif_file

plt.close(fig_anim)


# ==================================================
# Summary
# ==================================================
successful_fits = np.count_nonzero(
    np.isfinite(slope_values) & np.isfinite(angle_values_deg)
)

print("\nFinished.")
print(f"Successful fits: {successful_fits}/{nt}")
print(f"Input file: {PKL_FILE}")
print(f"Output folder: {OUT_DIR}")

print("\nSaved files:")
print(f"  {csv_file}")
print(f"  {npz_file}")
print(f"  {angle_plot_file}")
print(f"  {slope_plot_file}")
print(f"  {combined_plot_file}")
print(f"  {tilt_vector_3d_file}")
print(f"  {tilt_vector_2d_whole_file}")
print(f"  {tilt_vector_2d_zoom_file}")

if curve_available:
    print(f"  {anchor_csv_file}")
    print(f"  {anchor_npz_file}")

if saved_animation_file is not None:
    print(f"  {saved_animation_file}")

if curve_available:
    print(
        f"\nRounded-path anchor points "
        f"({CURVE_START_TIME_MIN:.1f} to "
        f"{CURVE_END_TIME_MIN:.1f} minutes):"
    )

    for point_number, frame_index in enumerate(
        control_indices,
        start=1,
    ):
        u_here = control_u[point_number - 1]
        v_here = control_v[point_number - 1]
        slope_x_here = u_here / 100.0
        slope_y_here = v_here / 100.0
        slope_magnitude_here = np.hypot(
            slope_x_here,
            slope_y_here,
        )

        direction_here = (
            (
                np.degrees(
                    np.arctan2(
                        slope_y_here,
                        slope_x_here,
                    )
                )
                + 180.0
            )
            % 360.0
        ) - 180.0

        print(
            f"  Anchor {point_number}: "
            f"frame = {frame_index}, "
            f"time = {times_minutes[frame_index]:.1f} min\n"
            f"    x slope = {slope_x_here:.8f} amp/km "
            f"({100.0 * slope_x_here:.6f} amp/100 km)\n"
            f"    y slope = {slope_y_here:.8f} amp/km "
            f"({100.0 * slope_y_here:.6f} amp/100 km)\n"
            f"    magnitude = {slope_magnitude_here:.8f} amp/km "
            f"({100.0 * slope_magnitude_here:.6f} amp/100 km)\n"
            f"    direction = {direction_here:+.3f} deg"
        )

if successful_fits > 0:
    print(
        "\nMedian slope magnitude: "
        f"{np.nanmedian(slope_values):.6g} amplitude per km"
    )
    print(
        "Median increasing direction: "
        f"{np.nanmedian(angle_values_deg):.3f} degrees"
    )


Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)

Fitting a plane to each frame...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed frame 150/280
  Processed frame 175/280
  Processed frame 200/280
  Processed frame 225/280
  Processed frame 250/280
  Processed frame 275/280
  Processed frame 280/280

Finished.
Successful fits: 280/280
Input file: /Users/anhphan/Downloads/processed_data_full_03_25_2025.pkl
Output folder: /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025

Saved files:
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.csv
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_fit_results.npz
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_angle_vs_frame.png
  /Users/anhphan/Downloads/BLO_plane_fit_outputs_Mar_25_2025/plane_slope_vs_frame.png
  /Users/anhphan/Downl

## Add local time, no anchor

In [34]:
# -*- coding: utf-8 -*-
"""
Batch-process all BLO PKL files from 2024 and 2025.

The script recursively searches INPUT_ROOT for .pkl files. For each file, it:

1. Reads the perturbation cube from data[5][1].
2. Reads the local date/start time from data[7][1].
3. Processes the file only when the internal start year is in TARGET_YEARS.
4. Fits a 2-D plane to every perturbation frame.
5. Saves CSV/NPZ results, angle/slope plots, 2-D/3-D tilt plots,
   and optionally an animation.
6. Writes each PKL's outputs into its own folder.
7. Creates one batch_processing_summary.csv file.

Time convention:
    data[7][1] is treated as the beginning of the first frame interval.
    Each fitted point is assigned to the center of its interval. Therefore,
    with DT_SECONDS = 120, frame 0 occurs one minute after the PKL start time.

Angle convention:
    0 degrees    = perturbation increases to the right
    +90 degrees  = perturbation increases downward
    -90 degrees  = perturbation increases upward
    +/-180       = perturbation increases to the left
"""

import csv
import gc
import pickle
import traceback
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter


# ==================================================
# Batch settings
# ==================================================
INPUT_ROOT = Path.home() / "Downloads" / "pkl"
OUTPUT_ROOT = INPUT_ROOT / "BLO_plane_fit_batch_outputs"

TARGET_YEARS = {2024, 2025, 2026}
RECURSIVE_SEARCH = True

# Animations can make the full batch substantially slower and use much more
# disk space. Set this to False when only the numerical results and plots
# are needed.
MAKE_ANIMATION = True

# If plane_fit_results.npz already exists in a PKL's output folder, skip it.
SKIP_IF_RESULTS_EXIST = True

# Continue processing the remaining PKLs after one file fails.
CONTINUE_ON_ERROR = True


# ==================================================
# Analysis settings
# ==================================================
DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

LOCAL_TIME_LABEL_FORMAT = "%H:%M"
LOCAL_TIME_EXACT_FORMAT = "%H:%M:%S"
LOCAL_DATETIME_PRINT_FORMAT = "%Y-%m-%d %H:%M:%S"

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

TILT_VECTOR_PLOT_STEP = 10
VECTOR_PLOT_PADDING_FRACTION = 0.12

TILT_PLOT_START_TIME_MIN = 90.0
TILT_PLOT_END_TIME_MIN = 500.0

HOURLY_MARKER_SIZE = 55
MIDNIGHT_MARKER_SIZE = 130
HOURLY_LABEL_OFFSET_POINTS = (6, 6)

# On the whole-time 2-D plot, every hour is marked, but fewer labels are drawn.
WHOLE_TIME_LABEL_EVERY_N_HOURS = 8
WHOLE_TIME_COLORBAR_EVERY_N_HOURS = 2


# ==================================================
# Time helpers
# ==================================================
def parse_pkl_start_datetime(date_start):
    """Parse data[7][1] = [date, start_time] as a local datetime."""
    if (
        not isinstance(date_start, (list, tuple, np.ndarray))
        or len(date_start) < 2
    ):
        raise ValueError(
            "Expected data[7][1] to contain [date, start_time], "
            f"but got: {date_start!r}"
        )

    combined_text = f"{str(date_start[0]).strip()} {str(date_start[1]).strip()}"

    accepted_formats = (
        "%m_%d_%Y %H:%M:%S",
        "%m_%d_%Y %H:%M",
        "%m-%d-%Y %H:%M:%S",
        "%m-%d-%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in accepted_formats:
        try:
            return datetime.strptime(combined_text, date_format)
        except ValueError:
            pass

    raise ValueError(
        "Could not parse the PKL date/start time: "
        f"{date_start!r}. Add its format to accepted_formats."
    )


def local_datetime_from_elapsed_minutes(start_datetime, elapsed_minutes):
    return start_datetime + timedelta(minutes=float(elapsed_minutes))


def make_hour_boundaries(start_datetime, time_values):
    """Return exact whole local hours within the elapsed-time range."""
    time_values = np.asarray(time_values, dtype=np.float64)
    finite_values = time_values[np.isfinite(time_values)]

    if finite_values.size == 0:
        return np.array([], dtype=object), np.array([], dtype=np.float64)

    minimum_elapsed = float(np.nanmin(finite_values))
    maximum_elapsed = float(np.nanmax(finite_values))

    minimum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, minimum_elapsed
    )
    maximum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, maximum_elapsed
    )

    first_hour = minimum_datetime.replace(
        minute=0, second=0, microsecond=0
    )
    if first_hour < minimum_datetime:
        first_hour += timedelta(hours=1)

    hour_datetimes = []
    current_hour = first_hour
    while current_hour <= maximum_datetime:
        hour_datetimes.append(current_hour)
        current_hour += timedelta(hours=1)

    elapsed_hour_minutes = np.array(
        [
            (hour_datetime - start_datetime).total_seconds() / 60.0
            for hour_datetime in hour_datetimes
        ],
        dtype=np.float64,
    )

    return np.array(hour_datetimes, dtype=object), elapsed_hour_minutes


def format_hour_tick(start_datetime, elapsed_minutes):
    local_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, elapsed_minutes
    )
    if local_datetime.hour == 0:
        local_label = "Midnight\n00:00"
    else:
        local_label = local_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)

    return f"{elapsed_minutes:.0f} min\n{local_label}"


def apply_hourly_elapsed_local_labels_to_colorbar(
    colorbar,
    start_datetime,
    time_values,
    label_every_n_hours=1,
):
    hour_datetimes, hour_elapsed_minutes = make_hour_boundaries(
        start_datetime, time_values
    )

    if hour_elapsed_minutes.size == 0:
        finite_values = np.asarray(time_values, dtype=np.float64)
        finite_values = finite_values[np.isfinite(finite_values)]
        if finite_values.size == 0:
            return

        hour_elapsed_minutes = np.unique(
            np.array(
                [
                    float(np.nanmin(finite_values)),
                    float(np.nanmax(finite_values)),
                ]
            )
        )
        hour_datetimes = np.array(
            [
                local_datetime_from_elapsed_minutes(start_datetime, value)
                for value in hour_elapsed_minutes
            ],
            dtype=object,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    keep = np.array(
        [
            (hour_datetime.hour == 0)
            or (hour_datetime.hour % label_every_n_hours == 0)
            for hour_datetime in hour_datetimes
        ],
        dtype=bool,
    )

    if np.any(keep):
        hour_elapsed_minutes = hour_elapsed_minutes[keep]

    colorbar.set_ticks(hour_elapsed_minutes)
    colorbar.set_ticklabels(
        [
            format_hour_tick(start_datetime, elapsed_minutes)
            for elapsed_minutes in hour_elapsed_minutes
        ]
    )
    colorbar.set_label("Elapsed time / local time")


def find_nearest_frame_indices_for_hours(
    selected_indices,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    target_hour_datetimes, target_hour_minutes = make_hour_boundaries(
        start_datetime, times_minutes[selected_indices]
    )

    if target_hour_minutes.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    marker_indices = []

    for target_datetime in target_hour_datetimes:
        differences = np.array(
            [
                abs(
                    (
                        local_datetimes[frame_index] - target_datetime
                    ).total_seconds()
                )
                for frame_index in selected_indices
            ],
            dtype=np.float64,
        )
        marker_indices.append(selected_indices[int(np.argmin(differences))])

    marker_indices = np.asarray(marker_indices, dtype=int)

    keep = np.concatenate(
        [
            np.array([True]),
            marker_indices[1:] != marker_indices[:-1],
        ]
    )

    return marker_indices[keep], target_hour_datetimes[keep]


def add_hourly_markers_2d(
    ax,
    selected_indices,
    x_values,
    y_values,
    local_datetimes,
    start_datetime,
    times_minutes,
    label_every_n_hours=1,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.2,
            label="1-hour marks",
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            zorder=8,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        is_midnight = target_datetime.hour == 0
        should_label = (
            is_midnight
            or target_datetime.hour % label_every_n_hours == 0
        )

        if not should_label:
            continue

        label_text = (
            "Midnight\n00:00"
            if is_midnight
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.annotate(
            label_text,
            xy=(x_values[frame_index], y_values[frame_index]),
            xytext=HOURLY_LABEL_OFFSET_POINTS,
            textcoords="offset points",
            fontsize=8,
            fontweight="bold" if is_midnight else "normal",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=9,
        )


def add_hourly_markers_3d(
    ax,
    selected_indices,
    x_values,
    y_values,
    z_values,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            z_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.0,
            label="1-hour marks",
            depthshade=False,
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            z_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            depthshade=False,
            zorder=8,
        )

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        label_text = (
            "Midnight\n00:00"
            if target_datetime.hour == 0
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.text(
            x_values[frame_index],
            y_values[frame_index],
            z_values[frame_index],
            f"  {label_text}",
            fontsize=8,
            zorder=9,
        )


# ==================================================
# Data and fitting helpers
# ==================================================
def load_wave_cube(pkl_file):
    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
        date_start = data[7][1]
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "The PKL does not contain the expected data[5][1] perturbation "
            "cube and data[7][1] date/start time."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected perturbation data with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    return label, wave_cube, date_start


def fit_plane_to_frame(frame, x_grid_km, y_grid_km):
    frame_sample = frame[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN
    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))

    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    design_matrix = np.column_stack(
        [x_valid, y_valid, np.ones(valid_count, dtype=np.float64)]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix, z_valid, rcond=None
    )

    a, b, c = coefficients
    residuals = z_valid - design_matrix @ coefficients
    rmse = np.sqrt(np.mean(residuals**2))
    slope = np.hypot(a, b)
    angle_deg = ((np.degrees(np.arctan2(b, a)) + 180.0) % 360.0) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


def calculate_vector_limits(x_values, y_values, padding_fraction):
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = np.asarray(x_values)[valid]
    y_values = np.asarray(y_values)[valid]

    if x_values.size == 0:
        return (-1.0, 1.0), (1.0, -1.0)

    x_min = np.nanmin(x_values)
    x_max = np.nanmax(x_values)
    y_min = np.nanmin(y_values)
    y_max = np.nanmax(y_values)

    x_span = x_max - x_min
    y_span = y_max - y_min

    if not np.isfinite(x_span) or x_span <= 0:
        x_span = max(abs(x_min), abs(x_max), 1.0e-6)
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = max(abs(y_min), abs(y_max), 1.0e-6)

    x_padding = padding_fraction * x_span
    y_padding = padding_fraction * y_span

    return (
        (x_min - x_padding, x_max + x_padding),
        (y_max + y_padding, y_min - y_padding),
    )


def safe_relative_output_name(pkl_file):
    """Create a unique folder name from the PKL's path relative to INPUT_ROOT."""
    try:
        relative_path = pkl_file.relative_to(INPUT_ROOT).with_suffix("")
        parts = relative_path.parts
    except ValueError:
        parts = (pkl_file.stem,)

    return "__".join(parts)


# ==================================================
# Plot helper
# ==================================================
def save_tilt_vector_2d_plot(
    time_mask,
    output_file,
    title_text,
    x_limits,
    y_limits,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    label_every_n_hours=1,
    colorbar_every_n_hours=1,
):
    fig, ax = plt.subplots(figsize=(9, 8))

    selected_all = np.flatnonzero(time_mask)

    if selected_all.size > 0:
        u_plot = x_slope_per_100km[selected_all]
        v_plot = y_slope_per_100km[selected_all]
        time_plot = times_minutes[selected_all]

        ax.plot(u_plot, v_plot, linewidth=0.8, alpha=0.30)

        scatter = ax.scatter(
            u_plot,
            v_plot,
            c=time_plot,
            cmap="viridis",
            s=26,
            zorder=3,
        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            time_plot,
            label_every_n_hours=colorbar_every_n_hours,
        )

        vector_indices = selected_all[::TILT_VECTOR_PLOT_STEP]
        if vector_indices.size > 0:
            ax.quiver(
                np.zeros(vector_indices.size),
                np.zeros(vector_indices.size),
                x_slope_per_100km[vector_indices],
                y_slope_per_100km[vector_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        add_hourly_markers_2d(
            ax=ax,
            selected_indices=selected_all,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
            label_every_n_hours=label_every_n_hours,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    if label_every_n_hours <= 1:
        subtitle = (
            "Every whole local hour is marked and labeled; "
            "midnight is highlighted"
        )
    else:
        subtitle = (
            "Every whole local hour is marked; labels are shown every "
            f"{label_every_n_hours} hours; midnight is highlighted"
        )

    ax.set_title(f"{title_text}\n{subtitle}")
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Process one PKL
# ==================================================
def process_one_pkl(pkl_file):
    print("\n" + "=" * 80)
    print(f"Reading: {pkl_file}")

    label, wave_cube, date_start = load_wave_cube(pkl_file)
    start_local_datetime = parse_pkl_start_datetime(date_start)

    if start_local_datetime.year not in TARGET_YEARS:
        print(
            f"Skipped: internal year {start_local_datetime.year} "
            f"is not in {sorted(TARGET_YEARS)}"
        )
        return {
            "input_file": str(pkl_file),
            "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
            "status": "skipped_year",
            "output_folder": "",
            "frame_count": "",
            "successful_fits": "",
            "message": f"Year {start_local_datetime.year} not selected",
        }

    output_name = safe_relative_output_name(pkl_file)
    out_dir = (
        OUTPUT_ROOT
        / str(start_local_datetime.year)
        / output_name
    )
    out_dir.mkdir(parents=True, exist_ok=True)

    npz_file = out_dir / "plane_fit_results.npz"

    if SKIP_IF_RESULTS_EXIST and npz_file.exists():
        print(f"Skipped existing result: {npz_file}")
        return {
            "input_file": str(pkl_file),
            "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
            "status": "skipped_existing",
            "output_folder": str(out_dir),
            "frame_count": wave_cube.shape[0],
            "successful_fits": "",
            "message": "plane_fit_results.npz already exists",
        }

    nt, ny, nx = wave_cube.shape
    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")
    print(
        "PKL interval start datetime: "
        f"{start_local_datetime.strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    frame_numbers = np.arange(nt, dtype=int)

    # Frame-center time: frame 0 is half one frame duration after PKL start.
    times_seconds = (frame_numbers + 0.5) * DT_SECONDS
    times_minutes = times_seconds / 60.0

    local_datetimes = np.array(
        [
            local_datetime_from_elapsed_minutes(
                start_local_datetime, elapsed_minutes
            )
            for elapsed_minutes in times_minutes
        ],
        dtype=object,
    )

    local_datetime_iso = np.array(
        [
            local_datetime.isoformat(sep=" ")
            for local_datetime in local_datetimes
        ],
        dtype=str,
    )

    print(
        "First frame-center datetime: "
        f"{local_datetimes[0].strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    x_1d_km = (
        np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
    ) * DX_KM
    y_1d_km = (
        np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
    ) * DY_KM
    x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)

    a_values = np.full(nt, np.nan, dtype=np.float64)
    b_values = np.full(nt, np.nan, dtype=np.float64)
    c_values = np.full(nt, np.nan, dtype=np.float64)
    slope_values = np.full(nt, np.nan, dtype=np.float64)
    angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    rmse_values = np.full(nt, np.nan, dtype=np.float64)
    valid_pixel_counts = np.zeros(nt, dtype=int)

    print("Fitting a plane to each frame...")

    for frame_index in range(nt):
        (
            a_values[frame_index],
            b_values[frame_index],
            c_values[frame_index],
            slope_values[frame_index],
            angle_values_deg[frame_index],
            rmse_values[frame_index],
            valid_pixel_counts[frame_index],
        ) = fit_plane_to_frame(
            wave_cube[frame_index],
            x_grid_km,
            y_grid_km,
        )

        if (
            frame_index == 0
            or frame_index == nt - 1
            or (frame_index + 1) % 25 == 0
        ):
            print(f"  Processed frame {frame_index + 1}/{nt}")

    successful_fits = int(
        np.count_nonzero(
            np.isfinite(slope_values) & np.isfinite(angle_values_deg)
        )
    )

    # ----------------------------------------------
    # Save CSV and NPZ
    # ----------------------------------------------
    csv_file = out_dir / "plane_fit_results.csv"

    with open(csv_file, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            [
                "frame",
                "frame_center_time_seconds",
                "frame_center_time_minutes",
                "local_datetime",
                "local_time",
                "slope_x_amplitude_per_km",
                "slope_y_amplitude_per_km",
                "plane_intercept",
                "slope_magnitude_amplitude_per_km",
                "increase_direction_deg_wrapped_minus180_to_180_image_coordinates",
                "fit_rmse",
                "valid_pixel_count",
            ]
        )

        for frame_index in range(nt):
            writer.writerow(
                [
                    frame_numbers[frame_index],
                    times_seconds[frame_index],
                    times_minutes[frame_index],
                    local_datetime_iso[frame_index],
                    local_datetimes[frame_index].strftime(
                        LOCAL_TIME_EXACT_FORMAT
                    ),
                    a_values[frame_index],
                    b_values[frame_index],
                    c_values[frame_index],
                    slope_values[frame_index],
                    angle_values_deg[frame_index],
                    rmse_values[frame_index],
                    valid_pixel_counts[frame_index],
                ]
            )

    np.savez(
        npz_file,
        input_file=str(pkl_file),
        frame=frame_numbers,
        time_seconds=times_seconds,
        time_minutes=times_minutes,
        time_reference="frame_center",
        frame_duration_seconds=DT_SECONDS,
        local_datetime=local_datetime_iso,
        start_local_datetime=start_local_datetime.isoformat(sep=" "),
        slope_x_amplitude_per_km=a_values,
        slope_y_amplitude_per_km=b_values,
        plane_intercept=c_values,
        slope_magnitude_amplitude_per_km=slope_values,
        increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
        fit_rmse=rmse_values,
        valid_pixel_count=valid_pixel_counts,
    )

    # ----------------------------------------------
    # Angle plot
    # ----------------------------------------------
    fig_angle, ax_angle = plt.subplots(figsize=(11, 5))
    ax_angle.plot(frame_numbers, angle_values_deg, linewidth=1.3)
    ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
    ax_angle.set_xlabel("Frame")
    ax_angle.set_ylabel(
        "Direction of perturbation increase "
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks(np.arange(-180, 181, 45))
    ax_angle.grid(True, alpha=0.3)
    fig_angle.tight_layout()
    angle_plot_file = out_dir / "plane_angle_vs_frame.png"
    fig_angle.savefig(angle_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_angle)

    # ----------------------------------------------
    # Slope plot
    # ----------------------------------------------
    fig_slope, ax_slope = plt.subplots(figsize=(11, 5))
    ax_slope.plot(frame_numbers, slope_values, linewidth=1.3)
    ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
    ax_slope.grid(True, alpha=0.3)
    fig_slope.tight_layout()
    slope_plot_file = out_dir / "plane_slope_vs_frame.png"
    fig_slope.savefig(slope_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_slope)

    # ----------------------------------------------
    # Combined plot
    # ----------------------------------------------
    fig_combined, axes = plt.subplots(
        nrows=2, ncols=1, figsize=(11, 8), sharex=True
    )

    axes[0].plot(frame_numbers, angle_values_deg, linewidth=1.3)
    axes[0].set_title(
        "Plane-fit Increasing Direction and Slope for Each Frame"
    )
    axes[0].set_ylabel(
        "Increase direction\n"
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    axes[0].set_ylim(-180, 180)
    axes[0].set_yticks(np.arange(-180, 181, 45))
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(frame_numbers, slope_values, linewidth=1.3)
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
    axes[1].grid(True, alpha=0.3)

    fig_combined.tight_layout()
    combined_plot_file = (
        out_dir / "plane_angle_and_slope_vs_frame.png"
    )
    fig_combined.savefig(
        combined_plot_file, dpi=DPI, bbox_inches="tight"
    )
    plt.close(fig_combined)

    # ----------------------------------------------
    # Physical tilt vectors
    # ----------------------------------------------
    x_slope_per_100km = 100.0 * a_values
    y_slope_per_100km = 100.0 * b_values

    finite_physical_vector = (
        np.isfinite(x_slope_per_100km)
        & np.isfinite(y_slope_per_100km)
        & np.isfinite(times_minutes)
    )

    tilt_plot_time_mask = (
        finite_physical_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )

    whole_time_mask = finite_physical_vector

    whole_vector_x_limits, whole_vector_y_limits = (
        calculate_vector_limits(
            x_slope_per_100km[whole_time_mask],
            y_slope_per_100km[whole_time_mask],
            VECTOR_PLOT_PADDING_FRACTION,
        )
    )

    zoom_vector_x_limits, zoom_vector_y_limits = (
        calculate_vector_limits(
            x_slope_per_100km[tilt_plot_time_mask],
            y_slope_per_100km[tilt_plot_time_mask],
            VECTOR_PLOT_PADDING_FRACTION,
        )
    )

    # 3-D tilt plot
    fig_tilt_3d = plt.figure(figsize=(9, 8))
    ax_tilt_3d = fig_tilt_3d.add_subplot(111, projection="3d")

    selected_3d_indices = np.flatnonzero(tilt_plot_time_mask)

    if selected_3d_indices.size > 0:
        u_finite = x_slope_per_100km[selected_3d_indices]
        v_finite = y_slope_per_100km[selected_3d_indices]
        time_finite = times_minutes[selected_3d_indices]

        ax_tilt_3d.plot(
            u_finite,
            v_finite,
            time_finite,
            linewidth=1.0,
            alpha=0.55,
        )

        scatter_3d = ax_tilt_3d.scatter(
            u_finite,
            v_finite,
            time_finite,
            c=time_finite,
            cmap="viridis",
            s=18,
            depthshade=False,
        )

        cbar_3d = fig_tilt_3d.colorbar(
            scatter_3d,
            ax=ax_tilt_3d,
            pad=0.10,
            shrink=0.72,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar_3d,
            start_local_datetime,
            time_finite,
        )

        add_hourly_markers_3d(
            ax=ax_tilt_3d,
            selected_indices=selected_3d_indices,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            z_values=times_minutes,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
        )

    ax_tilt_3d.set_xlim(*zoom_vector_x_limits)
    ax_tilt_3d.set_ylim(*zoom_vector_y_limits)
    ax_tilt_3d.set_box_aspect((1, 1, 1))
    ax_tilt_3d.set_title(
        "Plane-fit Tilt Vector vs Time\n"
        "Every whole local hour is marked; midnight is highlighted"
    )
    ax_tilt_3d.set_xlabel(
        "X slope (amplitude per 100 km; right positive)"
    )
    ax_tilt_3d.set_ylabel(
        "Y slope (amplitude per 100 km; down positive)"
    )
    ax_tilt_3d.set_zlabel("Elapsed time / local time")

    if selected_3d_indices.size > 0:
        hour_datetimes_3d, hour_elapsed_3d = make_hour_boundaries(
            start_local_datetime,
            times_minutes[selected_3d_indices],
        )

        if hour_elapsed_3d.size > 0:
            ax_tilt_3d.set_zticks(hour_elapsed_3d)
            ax_tilt_3d.set_zticklabels(
                [
                    (
                        "Midnight\n00:00"
                        if hour_datetime.hour == 0
                        else hour_datetime.strftime(
                            LOCAL_TIME_LABEL_FORMAT
                        )
                    )
                    + f"\n({elapsed_minutes:.0f} min)"
                    for hour_datetime, elapsed_minutes in zip(
                        hour_datetimes_3d,
                        hour_elapsed_3d,
                    )
                ]
            )

    handles_3d, _ = ax_tilt_3d.get_legend_handles_labels()
    if handles_3d:
        ax_tilt_3d.legend()

    ax_tilt_3d.grid(True, alpha=0.3)
    fig_tilt_3d.tight_layout()

    tilt_vector_3d_file = out_dir / "plane_tilt_vector_3d_vs_time.png"
    fig_tilt_3d.savefig(
        tilt_vector_3d_file, dpi=DPI, bbox_inches="tight"
    )
    plt.close(fig_tilt_3d)

    # 2-D whole and zoom plots
    tilt_vector_2d_whole_file = (
        out_dir
        / "plane_tilt_vector_2d_whole_time_colored_by_time.png"
    )

    tilt_vector_2d_zoom_file = (
        out_dir
        / f"plane_tilt_vector_2d_zoom_{TILT_PLOT_START_TIME_MIN:.0f}_to_"
          f"{TILT_PLOT_END_TIME_MIN:.0f}_min_colored_by_time.png"
    )

    save_tilt_vector_2d_plot(
        time_mask=whole_time_mask,
        output_file=tilt_vector_2d_whole_file,
        title_text="Plane-fit Tilt Vector Endpoints: Whole Time Range",
        x_limits=whole_vector_x_limits,
        y_limits=whole_vector_y_limits,
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=WHOLE_TIME_LABEL_EVERY_N_HOURS,
        colorbar_every_n_hours=WHOLE_TIME_COLORBAR_EVERY_N_HOURS,
    )

    save_tilt_vector_2d_plot(
        time_mask=tilt_plot_time_mask,
        output_file=tilt_vector_2d_zoom_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: "
            f"{TILT_PLOT_START_TIME_MIN:.0f}–"
            f"{TILT_PLOT_END_TIME_MIN:.0f} Minutes"
        ),
        x_limits=zoom_vector_x_limits,
        y_limits=zoom_vector_y_limits,
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=1,
        colorbar_every_n_hours=1,
    )

    # ----------------------------------------------
    # Animation
    # ----------------------------------------------
    saved_animation_file = ""

    if MAKE_ANIMATION:
        finite_slopes = slope_values[np.isfinite(slope_values)]
        if finite_slopes.size > 0:
            slope_ref = np.nanpercentile(
                finite_slopes, ARROW_LENGTH_PERCENTILE
            )
        else:
            slope_ref = 1.0

        if not np.isfinite(slope_ref) or slope_ref <= 0:
            slope_ref = 1.0

        max_arrow_length_pix = (
            ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)
        )

        arrow_u = np.zeros(nt, dtype=np.float64)
        arrow_v = np.zeros(nt, dtype=np.float64)

        finite_ab = np.isfinite(a_values) & np.isfinite(b_values)
        arrow_u[finite_ab] = (
            a_values[finite_ab] / slope_ref
        ) * max_arrow_length_pix
        arrow_v[finite_ab] = (
            b_values[finite_ab] / slope_ref
        ) * max_arrow_length_pix

        vmax = np.nanpercentile(np.abs(wave_cube), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        vmin = -vmax

        fig_anim, ax_anim = plt.subplots(figsize=(7, 6))
        im = ax_anim.imshow(
            wave_cube[0],
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )

        cbar = fig_anim.colorbar(
            im, ax=ax_anim, fraction=0.046, pad=0.04
        )
        cbar.set_label("Perturbation amplitude")

        center_x = (nx - 1) / 2.0
        center_y = (ny - 1) / 2.0

        ax_anim.scatter(
            [center_x],
            [center_y],
            s=CENTER_MARKER_SIZE,
            c=CENTER_MARKER_COLOR,
            marker="o",
        )

        quiver = ax_anim.quiver(
            [center_x],
            [center_y],
            [arrow_u[0]],
            [arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        title = ax_anim.set_title("")

        info_text = ax_anim.text(
            0.02,
            0.98,
            "",
            transform=ax_anim.transAxes,
            va="top",
            ha="left",
            fontsize=10,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        ax_anim.set_xlabel("X pixel")
        ax_anim.set_ylabel("Y pixel")

        def update(frame_index):
            im.set_data(wave_cube[frame_index])
            quiver.set_UVC(
                [arrow_u[frame_index]],
                [arrow_v[frame_index]],
            )

            local_time_text = local_datetimes[frame_index].strftime(
                LOCAL_TIME_EXACT_FORMAT
            )

            title.set_text(
                f"Plane-fit Vector Animation | "
                f"Frame {frame_index}/{nt - 1}\n"
                f"t = {times_minutes[frame_index]:.1f} min | "
                f"Local time = {local_time_text}"
            )

            if (
                np.isfinite(angle_values_deg[frame_index])
                and np.isfinite(slope_values[frame_index])
            ):
                info_text.set_text(
                    f"Increase direction = "
                    f"{angle_values_deg[frame_index]:+.1f}°\n"
                    f"Slope magnitude = "
                    f"{slope_values[frame_index]:.4g} amp/km\n"
                    f"a = {a_values[frame_index]:.4g} amp/km\n"
                    f"b = {b_values[frame_index]:.4g} amp/km"
                )
            else:
                info_text.set_text("Invalid fit for this frame")

            return im, quiver, title, info_text

        ani = FuncAnimation(
            fig_anim,
            update,
            frames=nt,
            interval=1000 / FPS,
            blit=False,
        )

        animation_mp4_file = (
            out_dir / "plane_fit_vector_animation.mp4"
        )
        animation_gif_file = (
            out_dir / "plane_fit_vector_animation.gif"
        )

        try:
            writer = FFMpegWriter(fps=FPS, bitrate=3000)
            ani.save(
                animation_mp4_file,
                writer=writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_mp4_file)
        except Exception as exc:
            print(
                "Could not save MP4 with ffmpeg. "
                "Falling back to GIF."
            )
            print(f"Reason: {exc}")

            gif_writer = PillowWriter(fps=FPS)
            ani.save(
                animation_gif_file,
                writer=gif_writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_gif_file)

        plt.close(fig_anim)

    print(f"Finished: {pkl_file.name}")
    print(f"Successful fits: {successful_fits}/{nt}")
    print(f"Output folder: {out_dir}")

    return {
        "input_file": str(pkl_file),
        "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
        "status": "processed",
        "output_folder": str(out_dir),
        "frame_count": nt,
        "successful_fits": successful_fits,
        "message": (
            f"Animation: {saved_animation_file}"
            if MAKE_ANIMATION
            else "Animation disabled"
        ),
    }


# ==================================================
# Batch driver
# ==================================================
def discover_pkl_files():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(
            f"Input folder does not exist:\n{INPUT_ROOT}"
        )

    iterator = (
        INPUT_ROOT.rglob("*.pkl")
        if RECURSIVE_SEARCH
        else INPUT_ROOT.glob("*.pkl")
    )

    files = []
    output_root_resolved = OUTPUT_ROOT.resolve()

    for file_path in iterator:
        try:
            file_resolved = file_path.resolve()
            if (
                file_resolved == output_root_resolved
                or output_root_resolved in file_resolved.parents
            ):
                continue
        except OSError:
            pass

        files.append(file_path)

    return sorted(files)


def write_batch_summary(summary_rows):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary_file = OUTPUT_ROOT / "batch_processing_summary.csv"

    fieldnames = [
        "input_file",
        "internal_start_datetime",
        "status",
        "output_folder",
        "frame_count",
        "successful_fits",
        "message",
    ]

    with open(summary_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    return summary_file


def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    pkl_files = discover_pkl_files()

    print(f"Input root: {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Target years: {sorted(TARGET_YEARS)}")
    print(f"PKL files found: {len(pkl_files)}")
    print(f"Animations enabled: {MAKE_ANIMATION}")

    if not pkl_files:
        print("No PKL files were found.")
        return

    summary_rows = []

    for file_number, pkl_file in enumerate(pkl_files, start=1):
        print(
            f"\nBatch file {file_number}/{len(pkl_files)}: "
            f"{pkl_file.name}"
        )

        try:
            summary_rows.append(process_one_pkl(pkl_file))
        except Exception as exc:
            print(f"FAILED: {pkl_file}")
            print(f"{type(exc).__name__}: {exc}")
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(pkl_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "message": f"{type(exc).__name__}: {exc}",
                }
            )

            if not CONTINUE_ON_ERROR:
                summary_file = write_batch_summary(summary_rows)
                print(f"Partial summary saved: {summary_file}")
                raise
        finally:
            plt.close("all")
            gc.collect()

        # Save an updated summary after every file so progress is retained.
        write_batch_summary(summary_rows)

    summary_file = write_batch_summary(summary_rows)

    processed_count = sum(
        row["status"] == "processed" for row in summary_rows
    )
    skipped_count = sum(
        str(row["status"]).startswith("skipped")
        for row in summary_rows
    )
    failed_count = sum(
        row["status"] == "failed" for row in summary_rows
    )

    print("\n" + "=" * 80)
    print("Batch finished.")
    print(f"Processed: {processed_count}")
    print(f"Skipped:   {skipped_count}")
    print(f"Failed:    {failed_count}")
    print(f"Summary:   {summary_file}")


if __name__ == "__main__":
    main()


Input root: /Users/anhphan/Downloads/pkl
Output root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs
Target years: [2024, 2025, 2026]
PKL files found: 11
Animations enabled: True

Batch file 1/11: processed_data_full_01_16_2026.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_01_16_2026.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (271, 1000, 1000)
PKL interval start datetime: 2026-01-16 20:25:53
First frame-center datetime: 2026-01-16 20:26:53
Fitting a plane to each frame...
  Processed frame 1/271
  Processed frame 25/271
  Processed frame 50/271
  Processed frame 75/271
  Processed frame 100/271
  Processed frame 125/271
  Processed frame 150/271
  Processed frame 175/271
  Processed frame 200/271
  Processed frame 225/271
  Processed frame 250/271
  Processed frame 271/271


/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py:1078: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig_tilt_3d.tight_layout()


Finished: processed_data_full_01_16_2026.pkl
Successful fits: 271/271
Output folder: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2026/processed_data_full_01_16_2026

Batch file 2/11: processed_data_full_03_11_2026.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_03_11_2026.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (271, 1000, 1000)
PKL interval start datetime: 2026-03-11 20:21:32
First frame-center datetime: 2026-03-11 20:22:32
Fitting a plane to each frame...
  Processed frame 1/271
  Processed frame 25/271
  Processed frame 50/271
  Processed frame 75/271
  Processed frame 100/271
  Processed frame 125/271
  Processed frame 150/271
  Processed frame 175/271
  Processed frame 200/271
  Processed frame 225/271
  Processed frame 250/271
  Processed frame 271/271


/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py:1078: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig_tilt_3d.tight_layout()


Finished: processed_data_full_03_11_2026.pkl
Successful fits: 271/271
Output folder: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2026/processed_data_full_03_11_2026

Batch file 3/11: processed_data_full_03_24_2025.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_03_24_2025.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (285, 1000, 1000)
PKL interval start datetime: 2025-03-24 20:32:37
First frame-center datetime: 2025-03-24 20:33:37
Fitting a plane to each frame...
  Processed frame 1/285
  Processed frame 25/285
  Processed frame 50/285
  Processed frame 75/285
  Processed frame 100/285
  Processed frame 125/285
  Processed frame 150/285
  Processed frame 175/285
  Processed frame 200/285
  Processed frame 225/285
  Processed frame 250/285
  Processed frame 275/285
  Processed frame 285/285


/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py:1078: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig_tilt_3d.tight_layout()


... (12 outputs left out)

Finished: processed_data_full_09_30_2024.pkl
Successful fits: 250/250
Output folder: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2024/processed_data_full_09_30_2024

Batch file 10/11: processed_data_full_10_27_2025.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_10_27_2025.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (301, 1000, 1000)
PKL interval start datetime: 2025-10-27 20:27:07
First frame-center datetime: 2025-10-27 20:28:07
Fitting a plane to each frame...
  Processed frame 1/301
  Processed frame 25/301
  Processed frame 50/301
  Processed frame 75/301
  Processed frame 100/301
  Processed frame 125/301
  Processed frame 150/301
  Processed frame 175/301
  Processed frame 200/301
  Processed frame 225/301
  Processed frame 250/301
  Processed frame 275/301
  Processed frame 300/301
  Processed frame 301/301


/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py:1078: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig_tilt_3d.tight_layout()


Finished: processed_data_full_10_27_2025.pkl
Successful fits: 301/301
Output folder: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/2025/processed_data_full_10_27_2025

Batch file 11/11: processed_data_full_11_30_2024.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_11_30_2024.pkl
FAILED: /Users/anhphan/Downloads/pkl/processed_data_full_11_30_2024.pkl
UnpicklingError: pickle data was truncated

Batch finished.
Processed: 10
Skipped:   0
Failed:    1
Summary:   /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/batch_processing_summary.csv


Traceback (most recent call last):
  File "/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py", line 1405, in main
    summary_rows.append(process_one_pkl(pkl_file))
  File "/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py", line 693, in process_one_pkl
    label, wave_cube, date_start = load_wave_cube(pkl_file)
  File "/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_14248/1090333991.py", line 477, in load_wave_cube
    data = pickle.load(file)
_pickle.UnpicklingError: pickle data was truncated


## Add hourly average

In [4]:
# -*- coding: utf-8 -*-
"""
Batch-process all BLO PKL files from 2024 and 2025.

The script recursively searches INPUT_ROOT for .pkl files. For each file, it:

1. Reads the perturbation cube from data[5][1].
2. Reads the local date/start time from data[7][1].
3. Processes the file only when the internal start year is in TARGET_YEARS.
4. Fits a 2-D plane to every perturbation frame.
5. Saves CSV/NPZ results, angle/slope plots, 2-D/3-D tilt plots,
   and optionally an animation.
6. Writes each PKL's outputs into its own folder.
7. Creates one batch_processing_summary.csv file.

Time convention:
    data[7][1] is treated as the beginning of the first frame interval.
    Each fitted point is assigned to the center of its interval. Therefore,
    with DT_SECONDS = 120, frame 0 occurs one minute after the PKL start time.

Angle convention:
    0 degrees    = perturbation increases to the right
    +90 degrees  = perturbation increases downward
    -90 degrees  = perturbation increases upward
    +/-180       = perturbation increases to the left
"""

import csv
import gc
import pickle
import traceback
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter


# ==================================================
# Batch settings
# ==================================================
INPUT_ROOT = Path.home() / "Downloads" / "pkl"
OUTPUT_ROOT = INPUT_ROOT / "BLO_plane_fit_batch_outputs"

TARGET_YEARS = {2024, 2025, 2026}
RECURSIVE_SEARCH = True

# Animations can make the full batch substantially slower and use much more
# disk space. Set this to False when only the numerical results and plots
# are needed.
MAKE_ANIMATION = True

# If plane_fit_results.npz already exists in a PKL's output folder, skip it.
SKIP_IF_RESULTS_EXIST = True

# Continue processing the remaining PKLs after one file fails.
CONTINUE_ON_ERROR = True


# ==================================================
# Analysis settings
# ==================================================
DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

LOCAL_TIME_LABEL_FORMAT = "%H:%M"
LOCAL_TIME_EXACT_FORMAT = "%H:%M:%S"
LOCAL_DATETIME_PRINT_FORMAT = "%Y-%m-%d %H:%M:%S"

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

TILT_VECTOR_PLOT_STEP = 10
VECTOR_PLOT_PADDING_FRACTION = 0.12

TILT_PLOT_START_TIME_MIN = 90.0
TILT_PLOT_END_TIME_MIN = 500.0

HOURLY_MARKER_SIZE = 55
MIDNIGHT_MARKER_SIZE = 130
HOURLY_LABEL_OFFSET_POINTS = (6, 6)

# On the whole-time 2-D plot, every hour is marked, but fewer labels are drawn.
WHOLE_TIME_LABEL_EVERY_N_HOURS = 8
WHOLE_TIME_COLORBAR_EVERY_N_HOURS = 2

# Separate sizes for the whole-range and zoomed 2-D tilt plots.
WHOLE_TIME_TILT_FIGSIZE = (14, 12)
ZOOM_TIME_TILT_FIGSIZE = (9, 8)

# For the hourly-average tilt plots, only include local clock hours from
# 21:00 through 03:00 (inclusive), i.e. 9 PM to 3 AM.
HOURLY_AVERAGE_START_HOUR = 21
HOURLY_AVERAGE_END_HOUR = 3


# ==================================================
# Time helpers
# ==================================================
def parse_pkl_start_datetime(date_start):
    """Parse data[7][1] = [date, start_time] as a local datetime."""
    if (
        not isinstance(date_start, (list, tuple, np.ndarray))
        or len(date_start) < 2
    ):
        raise ValueError(
            "Expected data[7][1] to contain [date, start_time], "
            f"but got: {date_start!r}"
        )

    combined_text = f"{str(date_start[0]).strip()} {str(date_start[1]).strip()}"

    accepted_formats = (
        "%m_%d_%Y %H:%M:%S",
        "%m_%d_%Y %H:%M",
        "%m-%d-%Y %H:%M:%S",
        "%m-%d-%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in accepted_formats:
        try:
            return datetime.strptime(combined_text, date_format)
        except ValueError:
            pass

    raise ValueError(
        "Could not parse the PKL date/start time: "
        f"{date_start!r}. Add its format to accepted_formats."
    )


def local_datetime_from_elapsed_minutes(start_datetime, elapsed_minutes):
    return start_datetime + timedelta(minutes=float(elapsed_minutes))


def make_hour_boundaries(start_datetime, time_values):
    """Return exact whole local hours within the elapsed-time range."""
    time_values = np.asarray(time_values, dtype=np.float64)
    finite_values = time_values[np.isfinite(time_values)]

    if finite_values.size == 0:
        return np.array([], dtype=object), np.array([], dtype=np.float64)

    minimum_elapsed = float(np.nanmin(finite_values))
    maximum_elapsed = float(np.nanmax(finite_values))

    minimum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, minimum_elapsed
    )
    maximum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, maximum_elapsed
    )

    first_hour = minimum_datetime.replace(
        minute=0, second=0, microsecond=0
    )
    if first_hour < minimum_datetime:
        first_hour += timedelta(hours=1)

    hour_datetimes = []
    current_hour = first_hour
    while current_hour <= maximum_datetime:
        hour_datetimes.append(current_hour)
        current_hour += timedelta(hours=1)

    elapsed_hour_minutes = np.array(
        [
            (hour_datetime - start_datetime).total_seconds() / 60.0
            for hour_datetime in hour_datetimes
        ],
        dtype=np.float64,
    )

    return np.array(hour_datetimes, dtype=object), elapsed_hour_minutes


def format_hour_tick(start_datetime, elapsed_minutes):
    local_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, elapsed_minutes
    )
    if local_datetime.hour == 0:
        local_label = "Midnight\n00:00"
    else:
        local_label = local_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)

    return f"{elapsed_minutes:.0f} min\n{local_label}"


def apply_hourly_elapsed_local_labels_to_colorbar(
    colorbar,
    start_datetime,
    time_values,
    label_every_n_hours=1,
):
    hour_datetimes, hour_elapsed_minutes = make_hour_boundaries(
        start_datetime, time_values
    )

    if hour_elapsed_minutes.size == 0:
        finite_values = np.asarray(time_values, dtype=np.float64)
        finite_values = finite_values[np.isfinite(finite_values)]
        if finite_values.size == 0:
            return

        hour_elapsed_minutes = np.unique(
            np.array(
                [
                    float(np.nanmin(finite_values)),
                    float(np.nanmax(finite_values)),
                ]
            )
        )
        hour_datetimes = np.array(
            [
                local_datetime_from_elapsed_minutes(start_datetime, value)
                for value in hour_elapsed_minutes
            ],
            dtype=object,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    keep = np.array(
        [
            (hour_datetime.hour == 0)
            or (hour_datetime.hour % label_every_n_hours == 0)
            for hour_datetime in hour_datetimes
        ],
        dtype=bool,
    )

    if np.any(keep):
        hour_elapsed_minutes = hour_elapsed_minutes[keep]

    colorbar.set_ticks(hour_elapsed_minutes)
    colorbar.set_ticklabels(
        [
            format_hour_tick(start_datetime, elapsed_minutes)
            for elapsed_minutes in hour_elapsed_minutes
        ]
    )
    colorbar.set_label("Elapsed time / local time")


def find_nearest_frame_indices_for_hours(
    selected_indices,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    target_hour_datetimes, target_hour_minutes = make_hour_boundaries(
        start_datetime, times_minutes[selected_indices]
    )

    if target_hour_minutes.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    marker_indices = []

    for target_datetime in target_hour_datetimes:
        differences = np.array(
            [
                abs(
                    (
                        local_datetimes[frame_index] - target_datetime
                    ).total_seconds()
                )
                for frame_index in selected_indices
            ],
            dtype=np.float64,
        )
        marker_indices.append(selected_indices[int(np.argmin(differences))])

    marker_indices = np.asarray(marker_indices, dtype=int)

    keep = np.concatenate(
        [
            np.array([True]),
            marker_indices[1:] != marker_indices[:-1],
        ]
    )

    return marker_indices[keep], target_hour_datetimes[keep]


def add_hourly_markers_2d(
    ax,
    selected_indices,
    x_values,
    y_values,
    local_datetimes,
    start_datetime,
    times_minutes,
    label_every_n_hours=1,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.2,
            label="1-hour marks",
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            zorder=8,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        is_midnight = target_datetime.hour == 0
        should_label = (
            is_midnight
            or target_datetime.hour % label_every_n_hours == 0
        )

        if not should_label:
            continue

        label_text = (
            "Midnight\n00:00"
            if is_midnight
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.annotate(
            label_text,
            xy=(x_values[frame_index], y_values[frame_index]),
            xytext=HOURLY_LABEL_OFFSET_POINTS,
            textcoords="offset points",
            fontsize=8,
            fontweight="bold" if is_midnight else "normal",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=9,
        )


def add_hourly_markers_3d(
    ax,
    selected_indices,
    x_values,
    y_values,
    z_values,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            z_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.0,
            label="1-hour marks",
            depthshade=False,
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            z_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            depthshade=False,
            zorder=8,
        )

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        label_text = (
            "Midnight\n00:00"
            if target_datetime.hour == 0
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.text(
            x_values[frame_index],
            y_values[frame_index],
            z_values[frame_index],
            f"  {label_text}",
            fontsize=8,
            zorder=9,
        )


# ==================================================
# Data and fitting helpers
# ==================================================
def load_wave_cube(pkl_file):
    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
        date_start = data[7][1]
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "The PKL does not contain the expected data[5][1] perturbation "
            "cube and data[7][1] date/start time."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected perturbation data with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    return label, wave_cube, date_start


def fit_plane_to_frame(frame, x_grid_km, y_grid_km):
    frame_sample = frame[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN
    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))

    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    design_matrix = np.column_stack(
        [x_valid, y_valid, np.ones(valid_count, dtype=np.float64)]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix, z_valid, rcond=None
    )

    a, b, c = coefficients
    residuals = z_valid - design_matrix @ coefficients
    rmse = np.sqrt(np.mean(residuals**2))
    slope = np.hypot(a, b)
    angle_deg = ((np.degrees(np.arctan2(b, a)) + 180.0) % 360.0) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


def calculate_vector_limits(x_values, y_values, padding_fraction):
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = np.asarray(x_values)[valid]
    y_values = np.asarray(y_values)[valid]

    if x_values.size == 0:
        return (-1.0, 1.0), (1.0, -1.0)

    x_min = np.nanmin(x_values)
    x_max = np.nanmax(x_values)
    y_min = np.nanmin(y_values)
    y_max = np.nanmax(y_values)

    x_span = x_max - x_min
    y_span = y_max - y_min

    if not np.isfinite(x_span) or x_span <= 0:
        x_span = max(abs(x_min), abs(x_max), 1.0e-6)
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = max(abs(y_min), abs(y_max), 1.0e-6)

    x_padding = padding_fraction * x_span
    y_padding = padding_fraction * y_span

    return (
        (x_min - x_padding, x_max + x_padding),
        (y_max + y_padding, y_min - y_padding),
    )


def safe_relative_output_name(pkl_file):
    """Create a unique folder name from the PKL's path relative to INPUT_ROOT."""
    try:
        relative_path = pkl_file.relative_to(INPUT_ROOT).with_suffix("")
        parts = relative_path.parts
    except ValueError:
        parts = (pkl_file.stem,)

    return "__".join(parts)


# ==================================================
# Plot helper
# ==================================================

# ==================================================
# Plot helpers
# ==================================================
def save_tilt_vector_2d_plot(
    time_mask,
    output_file,
    title_text,
    x_limits,
    y_limits,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    label_every_n_hours=1,
    colorbar_every_n_hours=1,
    figure_size=(9, 8),
):
    fig, ax = plt.subplots(figsize=figure_size)

    selected_all = np.flatnonzero(time_mask)

    if selected_all.size > 0:
        u_plot = x_slope_per_100km[selected_all]
        v_plot = y_slope_per_100km[selected_all]
        time_plot = times_minutes[selected_all]

        ax.plot(u_plot, v_plot, linewidth=0.8, alpha=0.30)

        scatter = ax.scatter(
            u_plot,
            v_plot,
            c=time_plot,
            cmap="viridis",
            s=26,
            zorder=3,
        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            time_plot,
            label_every_n_hours=colorbar_every_n_hours,
        )

        vector_indices = selected_all[::TILT_VECTOR_PLOT_STEP]
        if vector_indices.size > 0:
            ax.quiver(
                np.zeros(vector_indices.size),
                np.zeros(vector_indices.size),
                x_slope_per_100km[vector_indices],
                y_slope_per_100km[vector_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        add_hourly_markers_2d(
            ax=ax,
            selected_indices=selected_all,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
            label_every_n_hours=label_every_n_hours,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    if label_every_n_hours <= 1:
        subtitle = (
            "Every whole local hour is marked and labeled; "
            "midnight is highlighted"
        )
    else:
        subtitle = (
            "Every whole local hour is marked; labels are shown every "
            f"{label_every_n_hours} hours; midnight is highlighted"
        )

    ax.set_title(f"{title_text}\n{subtitle}")
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def local_hour_is_in_hourly_average_window(local_hour):
    """
    Return True when the local clock hour falls inside the requested nightly
    window. This supports windows that cross midnight, such as 21:00 to 03:00.
    """
    start_hour = int(HOURLY_AVERAGE_START_HOUR) % 24
    end_hour = int(HOURLY_AVERAGE_END_HOUR) % 24
    local_hour = int(local_hour) % 24

    if start_hour <= end_hour:
        return start_hour <= local_hour <= end_hour

    return (local_hour >= start_hour) or (local_hour <= end_hour)


def compute_hourly_average_rows(
    selected_indices,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    input_file,
    output_folder,
    start_local_datetime,
):
    """
    Group the selected frame-center points by local clock hour and average
    the tilt-vector endpoint for each hour.
    """
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return []

    hour_groups = {}

    for frame_index in selected_indices:
        if not local_hour_is_in_hourly_average_window(
            local_datetimes[frame_index].hour
        ):
            continue

        hour_start = local_datetimes[frame_index].replace(
            minute=0,
            second=0,
            microsecond=0,
        )
        hour_groups.setdefault(hour_start, []).append(int(frame_index))

    rows = []

    night_label = start_local_datetime.strftime("%Y-%m-%d")

    for hour_start in sorted(hour_groups):
        frame_indices = np.asarray(hour_groups[hour_start], dtype=int)

        mean_x = float(np.nanmean(x_slope_per_100km[frame_indices]))
        mean_y = float(np.nanmean(y_slope_per_100km[frame_indices]))
        mean_elapsed_minutes = float(np.nanmean(times_minutes[frame_indices]))

        rows.append(
            {
                "input_file": str(input_file),
                "output_folder": str(output_folder),
                "start_local_datetime": start_local_datetime.isoformat(sep=" "),
                "night_label": night_label,
                "local_date": hour_start.strftime("%Y-%m-%d"),
                "hour_start_local_datetime": hour_start.isoformat(sep=" "),
                "hour_label": hour_start.strftime(LOCAL_TIME_LABEL_FORMAT),
                "is_midnight": int(hour_start.hour == 0),
                "mean_elapsed_minutes": mean_elapsed_minutes,
                "mean_x_slope_per_100km": mean_x,
                "mean_y_slope_per_100km": mean_y,
                "mean_slope_magnitude_per_100km": float(np.hypot(mean_x, mean_y)),
                "frame_count_in_average": int(frame_indices.size),
                "first_frame": int(frame_indices[0]),
                "last_frame": int(frame_indices[-1]),
            }
        )

    return rows


def save_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def load_hourly_average_csv(input_file):
    if not Path(input_file).exists():
        return []

    with open(input_file, "r", newline="") as file:
        return list(csv.DictReader(file))


def save_hourly_average_tilt_plot(
    hourly_rows,
    output_file,
    title_text,
    x_limits,
    y_limits,
    start_local_datetime,
    figure_size=(9, 8),
):
    """
    Save a 2-D tilt plot using one averaged point per selected local hour.
    """
    fig, ax = plt.subplots(figsize=figure_size)

    if hourly_rows:
        mean_x = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_y = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_elapsed = np.array(
            [float(row["mean_elapsed_minutes"]) for row in hourly_rows],
            dtype=np.float64,
        )

        ax.plot(
            mean_x,
            mean_y,
            linewidth=1.2,
            alpha=0.55,
            color="0.35",
            zorder=2,
        )

        scatter = ax.scatter(
            mean_x,
            mean_y,
            c=mean_elapsed,
            cmap="viridis",
            s=95,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            mean_elapsed,
            label_every_n_hours=1,
        )

        midnight_mask = np.array(
            [int(row["is_midnight"]) == 1 for row in hourly_rows],
            dtype=bool,
        )

        if np.any(midnight_mask):
            ax.scatter(
                mean_x[midnight_mask],
                mean_y[midnight_mask],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color="black",
                edgecolors="white",
                linewidths=0.8,
                label="Midnight",
                zorder=4,
            )

        for row, x_here, y_here in zip(hourly_rows, mean_x, mean_y):
            if int(row["is_midnight"]) == 1:
                label_text = "Midnight\n00:00"
                font_weight = "bold"
            else:
                label_text = row["hour_label"]
                font_weight = "normal"

            ax.annotate(
                label_text,
                xy=(x_here, y_here),
                xytext=HOURLY_LABEL_OFFSET_POINTS,
                textcoords="offset points",
                fontsize=8,
                fontweight=font_weight,
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white",
                    edgecolor="0.5",
                    alpha=0.85,
                ),
                zorder=5,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"{title_text}\n"
        "One averaged tilt-vector endpoint is shown for each local hour "
        f"from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_all_days_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def save_all_days_hourly_average_plot(hourly_rows, output_file):
    """
    Plot the hourly averaged tilt points from all processed PKLs together.

    Each night/day is assigned its own color. The rows should already be filtered
    to the requested local-hour window.
    """
    if not hourly_rows:
        return

    x_values = np.array(
        [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )
    y_values = np.array(
        [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )

    x_limits, y_limits = calculate_vector_limits(
        x_values,
        y_values,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    rows_by_night = {}
    for row in hourly_rows:
        night_key = row.get("night_label", row["start_local_datetime"][:10])
        rows_by_night.setdefault(night_key, []).append(row)

    night_keys = sorted(rows_by_night.keys())

    if len(night_keys) <= 20:
        cmap = plt.get_cmap("tab20", max(len(night_keys), 1))
        colors = [cmap(i) for i in range(len(night_keys))]
    else:
        cmap = plt.get_cmap("nipy_spectral", max(len(night_keys), 1))
        colors = [cmap(i / max(len(night_keys) - 1, 1)) for i in range(len(night_keys))]

    for color, night_key in zip(colors, night_keys):
        night_rows = sorted(
            rows_by_night[night_key],
            key=lambda row: row["hour_start_local_datetime"],
        )

        x_here = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        y_here = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        midnight_mask_here = np.array(
            [int(row["is_midnight"]) == 1 for row in night_rows],
            dtype=bool,
        )

        ax.plot(
            x_here,
            y_here,
            linewidth=1.4,
            alpha=0.85,
            color=color,
            zorder=2,
            label=night_key,
        )

        ax.scatter(
            x_here,
            y_here,
            s=75,
            color=color,
            edgecolors="black",
            linewidths=0.5,
            alpha=0.95,
            zorder=3,
        )

        if np.any(midnight_mask_here):
            ax.scatter(
                x_here[midnight_mask_here],
                y_here[midnight_mask_here],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color=color,
                edgecolors="black",
                linewidths=0.8,
                zorder=4,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        "Hourly Averaged Tilt Points: All Days\n"
        f"Local-hour averages from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00; each day has a different color"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=8,
        framealpha=0.9,
        title="Day",
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Process one PKL
# ==================================================
def process_one_pkl(pkl_file):
    print("\n" + "=" * 80)
    print(f"Reading: {pkl_file}")

    label, wave_cube, date_start = load_wave_cube(pkl_file)
    start_local_datetime = parse_pkl_start_datetime(date_start)

    if start_local_datetime.year not in TARGET_YEARS:
        print(
            f"Skipped: internal year {start_local_datetime.year} "
            f"is not in {sorted(TARGET_YEARS)}"
        )
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_year",
                "output_folder": "",
                "frame_count": "",
                "successful_fits": "",
                "hourly_average_csv": "",
                "message": f"Year {start_local_datetime.year} not selected",
            },
            [],
        )

    output_name = safe_relative_output_name(pkl_file)
    out_dir = OUTPUT_ROOT / str(start_local_datetime.year) / output_name
    out_dir.mkdir(parents=True, exist_ok=True)

    npz_file = out_dir / "plane_fit_results.npz"
    hourly_average_csv_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )
    hourly_average_plot_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    required_existing_files = [
        npz_file,
        hourly_average_csv_file,
        hourly_average_plot_file,
    ]

    if SKIP_IF_RESULTS_EXIST and all(file.exists() for file in required_existing_files):
        print(f"Skipped existing results in: {out_dir}")
        hourly_rows = load_hourly_average_csv(hourly_average_csv_file)
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_existing",
                "output_folder": str(out_dir),
                "frame_count": wave_cube.shape[0],
                "successful_fits": "",
                "hourly_average_csv": str(hourly_average_csv_file),
                "message": "All required result files already exist",
            },
            hourly_rows,
        )

    nt, ny, nx = wave_cube.shape
    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")
    print(
        "PKL interval start datetime: "
        f"{start_local_datetime.strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    frame_numbers = np.arange(nt, dtype=int)

    # data[7][1] is the beginning of the first frame interval.
    # Each fitted point is assigned to the center of its interval.
    times_seconds = (frame_numbers + 0.5) * DT_SECONDS
    times_minutes = times_seconds / 60.0

    local_datetimes = np.array(
        [
            local_datetime_from_elapsed_minutes(
                start_local_datetime,
                elapsed_minutes,
            )
            for elapsed_minutes in times_minutes
        ],
        dtype=object,
    )

    local_datetime_iso = np.array(
        [
            local_datetime.isoformat(sep=" ")
            for local_datetime in local_datetimes
        ],
        dtype=str,
    )

    print(
        "First frame-center datetime: "
        f"{local_datetimes[0].strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    x_1d_km = (
        np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
    ) * DX_KM
    y_1d_km = (
        np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
    ) * DY_KM
    x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)

    a_values = np.full(nt, np.nan, dtype=np.float64)
    b_values = np.full(nt, np.nan, dtype=np.float64)
    c_values = np.full(nt, np.nan, dtype=np.float64)
    slope_values = np.full(nt, np.nan, dtype=np.float64)
    angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    rmse_values = np.full(nt, np.nan, dtype=np.float64)
    valid_pixel_counts = np.zeros(nt, dtype=int)

    print("Fitting a plane to each frame...")

    for frame_index in range(nt):
        (
            a_values[frame_index],
            b_values[frame_index],
            c_values[frame_index],
            slope_values[frame_index],
            angle_values_deg[frame_index],
            rmse_values[frame_index],
            valid_pixel_counts[frame_index],
        ) = fit_plane_to_frame(
            wave_cube[frame_index],
            x_grid_km,
            y_grid_km,
        )

        if (
            frame_index == 0
            or frame_index == nt - 1
            or (frame_index + 1) % 25 == 0
        ):
            print(f"  Processed frame {frame_index + 1}/{nt}")

    successful_fits = int(
        np.count_nonzero(
            np.isfinite(slope_values) & np.isfinite(angle_values_deg)
        )
    )

    # ----------------------------------------------
    # Save CSV and NPZ
    # ----------------------------------------------
    csv_file = out_dir / "plane_fit_results.csv"

    with open(csv_file, "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow(
            [
                "frame",
                "frame_center_time_seconds",
                "frame_center_time_minutes",
                "local_datetime",
                "local_time",
                "slope_x_amplitude_per_km",
                "slope_y_amplitude_per_km",
                "plane_intercept",
                "slope_magnitude_amplitude_per_km",
                "increase_direction_deg_wrapped_minus180_to_180_image_coordinates",
                "fit_rmse",
                "valid_pixel_count",
            ]
        )

        for frame_index in range(nt):
            writer.writerow(
                [
                    frame_numbers[frame_index],
                    times_seconds[frame_index],
                    times_minutes[frame_index],
                    local_datetime_iso[frame_index],
                    local_datetimes[frame_index].strftime(
                        LOCAL_TIME_EXACT_FORMAT
                    ),
                    a_values[frame_index],
                    b_values[frame_index],
                    c_values[frame_index],
                    slope_values[frame_index],
                    angle_values_deg[frame_index],
                    rmse_values[frame_index],
                    valid_pixel_counts[frame_index],
                ]
            )

    np.savez(
        npz_file,
        input_file=str(pkl_file),
        frame=frame_numbers,
        time_seconds=times_seconds,
        time_minutes=times_minutes,
        time_reference="frame_center",
        frame_duration_seconds=DT_SECONDS,
        local_datetime=local_datetime_iso,
        start_local_datetime=start_local_datetime.isoformat(sep=" "),
        slope_x_amplitude_per_km=a_values,
        slope_y_amplitude_per_km=b_values,
        plane_intercept=c_values,
        slope_magnitude_amplitude_per_km=slope_values,
        increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
        fit_rmse=rmse_values,
        valid_pixel_count=valid_pixel_counts,
    )

    # ----------------------------------------------
    # Angle plot
    # ----------------------------------------------
    fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

    ax_angle.plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
    )

    ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
    ax_angle.set_xlabel("Frame")
    ax_angle.set_ylabel(
        "Direction of perturbation increase "
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks(np.arange(-180, 181, 45))
    ax_angle.grid(True, alpha=0.3)

    fig_angle.tight_layout()
    angle_plot_file = out_dir / "plane_angle_vs_frame.png"
    fig_angle.savefig(angle_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_angle)

    # ----------------------------------------------
    # Slope plot
    # ----------------------------------------------
    fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

    ax_slope.plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
    )

    ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
    ax_slope.grid(True, alpha=0.3)

    fig_slope.tight_layout()
    slope_plot_file = out_dir / "plane_slope_vs_frame.png"
    fig_slope.savefig(slope_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_slope)

    # ----------------------------------------------
    # Combined plot
    # ----------------------------------------------
    fig_combined, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 8),
        sharex=True,
    )

    axes[0].plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
    )
    axes[0].set_title("Plane-fit Increasing Direction and Slope for Each Frame")
    axes[0].set_ylabel(
        "Increase direction\n"
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    axes[0].set_ylim(-180, 180)
    axes[0].set_yticks(np.arange(-180, 181, 45))
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
    )
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
    axes[1].grid(True, alpha=0.3)

    fig_combined.tight_layout()
    combined_plot_file = out_dir / "plane_angle_and_slope_vs_frame.png"
    fig_combined.savefig(
        combined_plot_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_combined)

    # ----------------------------------------------
    # Physical tilt vectors
    # ----------------------------------------------
    x_slope_per_100km = 100.0 * a_values
    y_slope_per_100km = 100.0 * b_values

    finite_physical_vector = (
        np.isfinite(x_slope_per_100km)
        & np.isfinite(y_slope_per_100km)
        & np.isfinite(times_minutes)
    )

    tilt_plot_time_mask = (
        finite_physical_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )

    whole_time_mask = finite_physical_vector

    whole_vector_x_limits, whole_vector_y_limits = calculate_vector_limits(
        x_slope_per_100km[whole_time_mask],
        y_slope_per_100km[whole_time_mask],
        VECTOR_PLOT_PADDING_FRACTION,
    )

    zoom_vector_x_limits, zoom_vector_y_limits = calculate_vector_limits(
        x_slope_per_100km[tilt_plot_time_mask],
        y_slope_per_100km[tilt_plot_time_mask],
        VECTOR_PLOT_PADDING_FRACTION,
    )

    # 3-D tilt plot
    fig_tilt_3d = plt.figure(figsize=(9, 8))
    ax_tilt_3d = fig_tilt_3d.add_subplot(111, projection="3d")

    selected_3d_indices = np.flatnonzero(tilt_plot_time_mask)

    if selected_3d_indices.size > 0:
        u_finite = x_slope_per_100km[selected_3d_indices]
        v_finite = y_slope_per_100km[selected_3d_indices]
        time_finite = times_minutes[selected_3d_indices]

        ax_tilt_3d.plot(
            u_finite,
            v_finite,
            time_finite,
            linewidth=1.0,
            alpha=0.55,
        )

        scatter_3d = ax_tilt_3d.scatter(
            u_finite,
            v_finite,
            time_finite,
            c=time_finite,
            cmap="viridis",
            s=18,
            depthshade=False,
        )

        cbar_3d = fig_tilt_3d.colorbar(
            scatter_3d,
            ax=ax_tilt_3d,
            pad=0.10,
            shrink=0.72,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar_3d,
            start_local_datetime,
            time_finite,
        )

        add_hourly_markers_3d(
            ax=ax_tilt_3d,
            selected_indices=selected_3d_indices,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            z_values=times_minutes,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
        )

    ax_tilt_3d.set_xlim(*zoom_vector_x_limits)
    ax_tilt_3d.set_ylim(*zoom_vector_y_limits)
    ax_tilt_3d.set_box_aspect((1, 1, 1))

    ax_tilt_3d.set_title(
        "Plane-fit Tilt Vector vs Time\n"
        "Every whole local hour is marked; midnight is highlighted"
    )
    ax_tilt_3d.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax_tilt_3d.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax_tilt_3d.set_zlabel("Elapsed time / local time")

    if selected_3d_indices.size > 0:
        hour_datetimes_3d, hour_elapsed_3d = make_hour_boundaries(
            start_local_datetime,
            times_minutes[selected_3d_indices],
        )

        if hour_elapsed_3d.size > 0:
            ax_tilt_3d.set_zticks(hour_elapsed_3d)
            ax_tilt_3d.set_zticklabels(
                [
                    (
                        "Midnight\n00:00"
                        if hour_datetime.hour == 0
                        else hour_datetime.strftime(
                            LOCAL_TIME_LABEL_FORMAT
                        )
                    )
                    + f"\n({elapsed_minutes:.0f} min)"
                    for hour_datetime, elapsed_minutes in zip(
                        hour_datetimes_3d,
                        hour_elapsed_3d,
                    )
                ]
            )

    handles_3d, _ = ax_tilt_3d.get_legend_handles_labels()
    if handles_3d:
        ax_tilt_3d.legend()

    ax_tilt_3d.grid(True, alpha=0.3)
    fig_tilt_3d.tight_layout()

    tilt_vector_3d_file = out_dir / "plane_tilt_vector_3d_vs_time.png"
    fig_tilt_3d.savefig(
        tilt_vector_3d_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_tilt_3d)

    # 2-D whole and zoom plots
    tilt_vector_2d_whole_file = (
        out_dir / "plane_tilt_vector_2d_whole_time_colored_by_time.png"
    )

    tilt_vector_2d_zoom_file = (
        out_dir
        / f"plane_tilt_vector_2d_zoom_{TILT_PLOT_START_TIME_MIN:.0f}_to_"
          f"{TILT_PLOT_END_TIME_MIN:.0f}_min_colored_by_time.png"
    )

    save_tilt_vector_2d_plot(
        time_mask=whole_time_mask,
        output_file=tilt_vector_2d_whole_file,
        title_text="Plane-fit Tilt Vector Endpoints: Whole Time Range",
        x_limits=whole_vector_x_limits,
        y_limits=whole_vector_y_limits,
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=WHOLE_TIME_LABEL_EVERY_N_HOURS,
        colorbar_every_n_hours=WHOLE_TIME_COLORBAR_EVERY_N_HOURS,
        figure_size=WHOLE_TIME_TILT_FIGSIZE,
    )

    save_tilt_vector_2d_plot(
        time_mask=tilt_plot_time_mask,
        output_file=tilt_vector_2d_zoom_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: "
            f"{TILT_PLOT_START_TIME_MIN:.0f}–"
            f"{TILT_PLOT_END_TIME_MIN:.0f} Minutes"
        ),
        x_limits=zoom_vector_x_limits,
        y_limits=zoom_vector_y_limits,
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=1,
        colorbar_every_n_hours=1,
        figure_size=ZOOM_TIME_TILT_FIGSIZE,
    )

    # New: hourly-averaged zoomed tilt points
    hourly_average_rows = compute_hourly_average_rows(
        selected_indices=np.flatnonzero(finite_physical_vector),
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        input_file=pkl_file,
        output_folder=out_dir,
        start_local_datetime=start_local_datetime,
    )

    save_hourly_average_csv(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_csv_file,
    )

    if hourly_average_rows:
        hourly_average_x = np.array(
            [
                float(row["mean_x_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )
        hourly_average_y = np.array(
            [
                float(row["mean_y_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )

        hourly_average_x_limits, hourly_average_y_limits = (
            calculate_vector_limits(
                hourly_average_x,
                hourly_average_y,
                VECTOR_PLOT_PADDING_FRACTION,
            )
        )
    else:
        hourly_average_x_limits = (-1.0, 1.0)
        hourly_average_y_limits = (1.0, -1.0)

    save_hourly_average_tilt_plot(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_plot_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: Hourly Averages\n"
            f"{HOURLY_AVERAGE_START_HOUR:02d}:00–"
            f"{HOURLY_AVERAGE_END_HOUR:02d}:00 Local Time"
        ),
        x_limits=hourly_average_x_limits,
        y_limits=hourly_average_y_limits,
        start_local_datetime=start_local_datetime,
        figure_size=ZOOM_TIME_TILT_FIGSIZE,
    )

    # ----------------------------------------------
    # Animation
    # ----------------------------------------------
    saved_animation_file = ""

    if MAKE_ANIMATION:
        finite_slopes = slope_values[np.isfinite(slope_values)]

        if finite_slopes.size > 0:
            slope_ref = np.nanpercentile(
                finite_slopes,
                ARROW_LENGTH_PERCENTILE,
            )
        else:
            slope_ref = 1.0

        if not np.isfinite(slope_ref) or slope_ref <= 0:
            slope_ref = 1.0

        max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

        arrow_u = np.zeros(nt, dtype=np.float64)
        arrow_v = np.zeros(nt, dtype=np.float64)

        finite_ab = np.isfinite(a_values) & np.isfinite(b_values)
        arrow_u[finite_ab] = (
            a_values[finite_ab] / slope_ref
        ) * max_arrow_length_pix
        arrow_v[finite_ab] = (
            b_values[finite_ab] / slope_ref
        ) * max_arrow_length_pix

        vmax = np.nanpercentile(np.abs(wave_cube), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        vmin = -vmax

        fig_anim, ax_anim = plt.subplots(figsize=(7, 6))

        im = ax_anim.imshow(
            wave_cube[0],
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )

        cbar = fig_anim.colorbar(
            im,
            ax=ax_anim,
            fraction=0.046,
            pad=0.04,
        )
        cbar.set_label("Perturbation amplitude")

        center_x = (nx - 1) / 2.0
        center_y = (ny - 1) / 2.0

        ax_anim.scatter(
            [center_x],
            [center_y],
            s=CENTER_MARKER_SIZE,
            c=CENTER_MARKER_COLOR,
            marker="o",
        )

        quiver = ax_anim.quiver(
            [center_x],
            [center_y],
            [arrow_u[0]],
            [arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        title = ax_anim.set_title("")

        info_text = ax_anim.text(
            0.02,
            0.98,
            "",
            transform=ax_anim.transAxes,
            va="top",
            ha="left",
            fontsize=10,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        ax_anim.set_xlabel("X pixel")
        ax_anim.set_ylabel("Y pixel")

        def update(frame_index):
            im.set_data(wave_cube[frame_index])

            quiver.set_UVC(
                [arrow_u[frame_index]],
                [arrow_v[frame_index]],
            )

            local_time_text = local_datetimes[frame_index].strftime(
                LOCAL_TIME_EXACT_FORMAT
            )

            title.set_text(
                f"Plane-fit Vector Animation | Frame {frame_index}/{nt - 1}\n"
                f"t = {times_minutes[frame_index]:.1f} min | "
                f"Local time = {local_time_text}"
            )

            angle_here = angle_values_deg[frame_index]
            slope_here = slope_values[frame_index]
            a_here = a_values[frame_index]
            b_here = b_values[frame_index]

            if np.isfinite(angle_here) and np.isfinite(slope_here):
                info_text.set_text(
                    f"Increase direction = {angle_here:+.1f}°\n"
                    f"Slope magnitude = {slope_here:.4g} amp/km\n"
                    f"a = {a_here:.4g} amp/km\n"
                    f"b = {b_here:.4g} amp/km"
                )
            else:
                info_text.set_text("Invalid fit for this frame")

            return im, quiver, title, info_text

        ani = FuncAnimation(
            fig_anim,
            update,
            frames=nt,
            interval=1000 / FPS,
            blit=False,
        )

        animation_mp4_file = out_dir / "plane_fit_vector_animation.mp4"
        animation_gif_file = out_dir / "plane_fit_vector_animation.gif"

        try:
            writer = FFMpegWriter(
                fps=FPS,
                bitrate=3000,
            )
            ani.save(
                animation_mp4_file,
                writer=writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_mp4_file)
        except Exception as exc:
            print(
                "Could not save MP4 with ffmpeg. "
                "Falling back to GIF."
            )
            print(f"Reason: {exc}")

            gif_writer = PillowWriter(fps=FPS)
            ani.save(
                animation_gif_file,
                writer=gif_writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_gif_file)

        plt.close(fig_anim)

    print(f"Finished: {pkl_file.name}")
    print(f"Successful fits: {successful_fits}/{nt}")
    print(f"Output folder: {out_dir}")

    return (
        {
            "input_file": str(pkl_file),
            "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
            "status": "processed",
            "output_folder": str(out_dir),
            "frame_count": nt,
            "successful_fits": successful_fits,
            "hourly_average_csv": str(hourly_average_csv_file),
            "message": (
                f"Animation: {saved_animation_file}"
                if MAKE_ANIMATION
                else "Animation disabled"
            ),
        },
        hourly_average_rows,
    )


# ==================================================
# Batch driver
# ==================================================
def discover_pkl_files():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(
            f"Input folder does not exist:\n{INPUT_ROOT}"
        )

    iterator = (
        INPUT_ROOT.rglob("*.pkl")
        if RECURSIVE_SEARCH
        else INPUT_ROOT.glob("*.pkl")
    )

    files = []
    output_root_resolved = OUTPUT_ROOT.resolve()

    for file_path in iterator:
        try:
            file_resolved = file_path.resolve()
            if (
                file_resolved == output_root_resolved
                or output_root_resolved in file_resolved.parents
            ):
                continue
        except OSError:
            pass

        files.append(file_path)

    return sorted(files)


def write_batch_summary(summary_rows):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary_file = OUTPUT_ROOT / "batch_processing_summary.csv"

    fieldnames = [
        "input_file",
        "internal_start_datetime",
        "status",
        "output_folder",
        "frame_count",
        "successful_fits",
        "hourly_average_csv",
        "message",
    ]

    with open(summary_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    return summary_file


def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    pkl_files = discover_pkl_files()

    print(f"Input root: {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Target years: {sorted(TARGET_YEARS)}")
    print(f"PKL files found: {len(pkl_files)}")
    print(f"Animations enabled: {MAKE_ANIMATION}")

    if not pkl_files:
        print("No PKL files were found.")
        return

    summary_rows = []
    all_days_hourly_average_rows = []

    for file_number, pkl_file in enumerate(pkl_files, start=1):
        print(
            f"\nBatch file {file_number}/{len(pkl_files)}: "
            f"{pkl_file.name}"
        )

        try:
            summary_row, hourly_rows = process_one_pkl(pkl_file)
            summary_rows.append(summary_row)
            all_days_hourly_average_rows.extend(hourly_rows)
        except Exception as exc:
            print(f"FAILED: {pkl_file}")
            print(f"{type(exc).__name__}: {exc}")
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(pkl_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "message": f"{type(exc).__name__}: {exc}",
                }
            )

            if not CONTINUE_ON_ERROR:
                summary_file = write_batch_summary(summary_rows)
                print(f"Partial summary saved: {summary_file}")
                raise
        finally:
            plt.close("all")
            gc.collect()

        # Save an updated summary after every file so progress is retained.
        write_batch_summary(summary_rows)

    summary_file = write_batch_summary(summary_rows)

    all_days_hourly_average_csv_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )
    all_days_hourly_average_plot_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    save_all_days_hourly_average_csv(
        all_days_hourly_average_rows,
        all_days_hourly_average_csv_file,
    )
    save_all_days_hourly_average_plot(
        all_days_hourly_average_rows,
        all_days_hourly_average_plot_file,
    )

    processed_count = sum(
        row["status"] == "processed" for row in summary_rows
    )
    skipped_count = sum(
        str(row["status"]).startswith("skipped")
        for row in summary_rows
    )
    failed_count = sum(
        row["status"] == "failed" for row in summary_rows
    )

    print("\n" + "=" * 80)
    print("Batch finished.")
    print(f"Processed: {processed_count}")
    print(f"Skipped:   {skipped_count}")
    print(f"Failed:    {failed_count}")
    print(f"Summary:   {summary_file}")
    print(f"All-days hourly-average CSV:  {all_days_hourly_average_csv_file}")
    print(f"All-days hourly-average plot: {all_days_hourly_average_plot_file}")
    print(
        "Hourly-average local-hour window: "
        f"{HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )


if __name__ == "__main__":
    main()


Input root: /Users/anhphan/Downloads/pkl
Output root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs
Target years: [2024, 2025, 2026]
Animations enabled: True
Preprocessing diagnostics enabled: True

RUN MODE: SINGLE FILE
Selected PKL: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)
PKL interval start datetime: 2025-03-25 20:29:27
First frame-center datetime: 2025-03-25 20:30:27
Plane-fit region: radius <= 250.0 km (196364 pixels before other validity filters)
Spatial preprocessing: robust clip at 4 MAD-sigma, then Gaussian sigma = 40 km
Robust plane refit: enabled, residual threshold = 2 MAD-sigma, max iterations = 3
Fitting a plane to each frame...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed f

/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_40354/553231762.py:2155: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig_tilt_3d.tight_layout()


FAILED: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
UnboundLocalError: local variable 'removed_component_vmax' referenced before assignment
Summary saved: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs/batch_processing_summary.csv


Traceback (most recent call last):
  File "/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_40354/553231762.py", line 2715, in main
    ) = process_one_pkl(
  File "/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_40354/553231762.py", line 2362, in process_one_pkl
    if not np.isfinite(removed_component_vmax) or removed_component_vmax <= 0:
UnboundLocalError: local variable 'removed_component_vmax' referenced before assignment


## (300) km radius, smoothing

In [20]:
# -*- coding: utf-8 -*-
"""
Batch-process all BLO PKL files from 2024 and 2025.

The script recursively searches INPUT_ROOT for .pkl files. For each file, it:

1. Reads the perturbation cube from data[5][1].
2. Reads the local date/start time from data[7][1].
3. Processes the file only when the internal start year is in TARGET_YEARS.
4. Fits a 2-D plane to both the raw frame and a Gaussian-background-outlier-cleaned frame, using only pixels within FIT_RADIUS_KM of the image center.
5. Saves CSV/NPZ results, angle/slope plots, 2-D/3-D tilt plots,
   and optionally an animation.
6. Writes each PKL's outputs into its own folder.
7. Creates one batch_processing_summary.csv file.

Time convention:
    data[7][1] is treated as the beginning of the first frame interval.
    Each fitted point is assigned to the center of its interval. Therefore,
    with DT_SECONDS = 120, frame 0 occurs one minute after the PKL start time.

Angle convention:
    0 degrees    = perturbation increases to the right
    +90 degrees  = perturbation increases downward
    -90 degrees  = perturbation increases upward
    +/-180       = perturbation increases to the left
"""

import csv
import gc
import pickle
import traceback
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.patches import Ellipse
from scipy.ndimage import binary_dilation, gaussian_filter


# ==================================================
# Batch settings
# ==================================================
INPUT_ROOT = Path.home() / "Downloads" / "pkl"
OUTPUT_ROOT = INPUT_ROOT / "BLO_plane_fit_batch_outputs"

# ==================================================
# Run mode
# ==================================================
# True:
#     Process only SINGLE_PKL_FILE. Use this while tuning one night/day.
#
# False:
#     Discover and process all PKL files under INPUT_ROOT.
PROCESS_SINGLE_FILE = True

# Change this to the one PKL you want to examine first.
# An absolute path is also fine.
SINGLE_PKL_FILE = INPUT_ROOT / "processed_data_full_03_25_2025.pkl"

TARGET_YEARS = {2024, 2025, 2026}
RECURSIVE_SEARCH = True

# Animations can make the full batch substantially slower and use much more
# disk space. Set this to False when only the numerical results and plots
# are needed.
MAKE_ANIMATION = True

# If plane_fit_results.npz already exists in a PKL's output folder, skip it.
# Keep this False after changing the fit radius/filter settings so old results
# are recalculated with the new method.
SKIP_IF_RESULTS_EXIST = False

# Continue processing the remaining PKLs after one file fails.
CONTINUE_ON_ERROR = True


# ==================================================
# Analysis settings
# ==================================================
DX_KM = 1.0
DY_KM = 1.0
DT_SECONDS = 120.0

LOCAL_TIME_LABEL_FORMAT = "%H:%M"
LOCAL_TIME_EXACT_FORMAT = "%H:%M:%S"
LOCAL_DATETIME_PRINT_FORMAT = "%Y-%m-%d %H:%M:%S"

DPI = 200
FPS = 20
CMAP = "RdBu_r"

FIT_PIXEL_STEP = 1
FIT_VALUE_MIN = None
FIT_VALUE_MAX = None
MIN_VALID_PIXELS = 100

# Only pixels inside this radius, measured from the image center, are used
# in the plane fit.
FIT_RADIUS_KM = 300.0

# Preprocessing before fitting the plane.
#
# Keep only one cleaning operation:
#   1. Build a broad Gaussian background.
#   2. Compute high_pass = original - Gaussian background.
#   3. Estimate high-pass scatter with a robust MAD sigma.
#   4. Flag unusually sharp pixels.
#   5. Optionally dilate the mask to include a small halo.
#   6. Replace only flagged pixels with the Gaussian-background value.
#
# Unflagged pixels remain exactly unchanged.
APPLY_STAR_RESIDUAL_FILTER = True

BACKGROUND_GAUSSIAN_SIGMA_KM = 40.0
BACKGROUND_GAUSSIAN_TRUNCATE = 3.0
STAR_RESIDUAL_MAD_SIGMA = 2.5
STAR_MASK_DILATION_KM = 4.0

ARROW_COLOR = "yellow"
CENTER_MARKER_COLOR = "black"
CENTER_MARKER_SIZE = 20
ARROW_LENGTH_PERCENTILE = 95
ARROW_MAX_FRACTION_OF_IMAGE = 0.35

# Diagnostic comparison arrows use the same scale for raw and preprocessed
# within each selected frame.
DIAGNOSTIC_ARROW_MAX_FRACTION_OF_IMAGE = 0.28

# Removed component shown in the animation:
#     removed component = original frame - preprocessed frame
# This shows what the Gaussian-background outlier replacement removed.
REMOVED_COMPONENT_CMAP = "RdBu_r"
REMOVED_COMPONENT_SCALE_PERCENTILE = 99.0
REMOVED_COMPONENT_SCALE_SAMPLE_FRAMES = 12

# Optional static diagnostic figures for tuning one day.
# Each selected frame compares the original image, Gaussian background,
# high-pass residual, detected mask, removed component, final preprocessed,
# and slope vectors before/after preprocessing.
MAKE_PREPROCESSING_DIAGNOSTICS = True

# Explicit frame numbers to inspect.
DIAGNOSTIC_FRAME_INDICES = [0, 30, 60, 90, 120, 150, 180, 210]

# Also inspect the frames nearest these elapsed times in minutes.
DIAGNOSTIC_TIMES_MIN = [30.0, 90.0, 150.0, 210.0, 270.0, 330.0]

TILT_VECTOR_PLOT_STEP = 10
VECTOR_PLOT_PADDING_FRACTION = 0.12

TILT_PLOT_START_TIME_MIN = 90.0
TILT_PLOT_END_TIME_MIN = 500.0

HOURLY_MARKER_SIZE = 55
MIDNIGHT_MARKER_SIZE = 130
HOURLY_LABEL_OFFSET_POINTS = (6, 6)

# On the whole-time 2-D plot, every hour is marked, but fewer labels are drawn.
WHOLE_TIME_LABEL_EVERY_N_HOURS = 8
WHOLE_TIME_COLORBAR_EVERY_N_HOURS = 2

# Separate sizes for the whole-range and zoomed 2-D tilt plots.
WHOLE_TIME_TILT_FIGSIZE = (18, 9)
ZOOM_TIME_TILT_FIGSIZE = (14, 7.5)
VECTOR_3D_FIGSIZE = (13, 7.5)

# Thin colorbars for the vector plots.
VECTOR_COLORBAR_FRACTION = 0.022
VECTOR_COLORBAR_PAD = 0.025
VECTOR_COLORBAR_ASPECT = 35

# For the hourly-average tilt plots, only include local clock hours from
# 21:00 through 03:00 (inclusive), i.e. 9 PM to 3 AM.
HOURLY_AVERAGE_START_HOUR = 21
HOURLY_AVERAGE_END_HOUR = 3


# ==================================================
# Time helpers
# ==================================================
def parse_pkl_start_datetime(date_start):
    """Parse data[7][1] = [date, start_time] as a local datetime."""
    if (
        not isinstance(date_start, (list, tuple, np.ndarray))
        or len(date_start) < 2
    ):
        raise ValueError(
            "Expected data[7][1] to contain [date, start_time], "
            f"but got: {date_start!r}"
        )

    combined_text = f"{str(date_start[0]).strip()} {str(date_start[1]).strip()}"

    accepted_formats = (
        "%m_%d_%Y %H:%M:%S",
        "%m_%d_%Y %H:%M",
        "%m-%d-%Y %H:%M:%S",
        "%m-%d-%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    )

    for date_format in accepted_formats:
        try:
            return datetime.strptime(combined_text, date_format)
        except ValueError:
            pass

    raise ValueError(
        "Could not parse the PKL date/start time: "
        f"{date_start!r}. Add its format to accepted_formats."
    )


def local_datetime_from_elapsed_minutes(start_datetime, elapsed_minutes):
    return start_datetime + timedelta(minutes=float(elapsed_minutes))


def make_hour_boundaries(start_datetime, time_values):
    """Return exact whole local hours within the elapsed-time range."""
    time_values = np.asarray(time_values, dtype=np.float64)
    finite_values = time_values[np.isfinite(time_values)]

    if finite_values.size == 0:
        return np.array([], dtype=object), np.array([], dtype=np.float64)

    minimum_elapsed = float(np.nanmin(finite_values))
    maximum_elapsed = float(np.nanmax(finite_values))

    minimum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, minimum_elapsed
    )
    maximum_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, maximum_elapsed
    )

    first_hour = minimum_datetime.replace(
        minute=0, second=0, microsecond=0
    )
    if first_hour < minimum_datetime:
        first_hour += timedelta(hours=1)

    hour_datetimes = []
    current_hour = first_hour
    while current_hour <= maximum_datetime:
        hour_datetimes.append(current_hour)
        current_hour += timedelta(hours=1)

    elapsed_hour_minutes = np.array(
        [
            (hour_datetime - start_datetime).total_seconds() / 60.0
            for hour_datetime in hour_datetimes
        ],
        dtype=np.float64,
    )

    return np.array(hour_datetimes, dtype=object), elapsed_hour_minutes


def format_hour_tick(start_datetime, elapsed_minutes):
    local_datetime = local_datetime_from_elapsed_minutes(
        start_datetime, elapsed_minutes
    )
    if local_datetime.hour == 0:
        local_label = "Midnight\n00:00"
    else:
        local_label = local_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)

    return f"{elapsed_minutes:.0f} min\n{local_label}"


def apply_hourly_elapsed_local_labels_to_colorbar(
    colorbar,
    start_datetime,
    time_values,
    label_every_n_hours=1,
):
    hour_datetimes, hour_elapsed_minutes = make_hour_boundaries(
        start_datetime, time_values
    )

    if hour_elapsed_minutes.size == 0:
        finite_values = np.asarray(time_values, dtype=np.float64)
        finite_values = finite_values[np.isfinite(finite_values)]
        if finite_values.size == 0:
            return

        hour_elapsed_minutes = np.unique(
            np.array(
                [
                    float(np.nanmin(finite_values)),
                    float(np.nanmax(finite_values)),
                ]
            )
        )
        hour_datetimes = np.array(
            [
                local_datetime_from_elapsed_minutes(start_datetime, value)
                for value in hour_elapsed_minutes
            ],
            dtype=object,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    keep = np.array(
        [
            (hour_datetime.hour == 0)
            or (hour_datetime.hour % label_every_n_hours == 0)
            for hour_datetime in hour_datetimes
        ],
        dtype=bool,
    )

    if np.any(keep):
        hour_elapsed_minutes = hour_elapsed_minutes[keep]

    colorbar.set_ticks(hour_elapsed_minutes)
    colorbar.set_ticklabels(
        [
            format_hour_tick(start_datetime, elapsed_minutes)
            for elapsed_minutes in hour_elapsed_minutes
        ]
    )
    colorbar.set_label("Elapsed time / local time")


def find_nearest_frame_indices_for_hours(
    selected_indices,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    target_hour_datetimes, target_hour_minutes = make_hour_boundaries(
        start_datetime, times_minutes[selected_indices]
    )

    if target_hour_minutes.size == 0:
        return np.array([], dtype=int), np.array([], dtype=object)

    marker_indices = []

    for target_datetime in target_hour_datetimes:
        differences = np.array(
            [
                abs(
                    (
                        local_datetimes[frame_index] - target_datetime
                    ).total_seconds()
                )
                for frame_index in selected_indices
            ],
            dtype=np.float64,
        )
        marker_indices.append(selected_indices[int(np.argmin(differences))])

    marker_indices = np.asarray(marker_indices, dtype=int)

    keep = np.concatenate(
        [
            np.array([True]),
            marker_indices[1:] != marker_indices[:-1],
        ]
    )

    return marker_indices[keep], target_hour_datetimes[keep]


def add_hourly_markers_2d(
    ax,
    selected_indices,
    x_values,
    y_values,
    local_datetimes,
    start_datetime,
    times_minutes,
    label_every_n_hours=1,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.2,
            label="1-hour marks",
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            zorder=8,
        )

    label_every_n_hours = max(int(label_every_n_hours), 1)

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        is_midnight = target_datetime.hour == 0
        should_label = (
            is_midnight
            or target_datetime.hour % label_every_n_hours == 0
        )

        if not should_label:
            continue

        label_text = (
            "Midnight\n00:00"
            if is_midnight
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.annotate(
            label_text,
            xy=(x_values[frame_index], y_values[frame_index]),
            xytext=HOURLY_LABEL_OFFSET_POINTS,
            textcoords="offset points",
            fontsize=8,
            fontweight="bold" if is_midnight else "normal",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=9,
        )


def add_hourly_markers_3d(
    ax,
    selected_indices,
    x_values,
    y_values,
    z_values,
    local_datetimes,
    start_datetime,
    times_minutes,
):
    marker_indices, target_hour_datetimes = (
        find_nearest_frame_indices_for_hours(
            selected_indices,
            local_datetimes,
            start_datetime,
            times_minutes,
        )
    )

    if marker_indices.size == 0:
        return

    regular_mask = np.array(
        [target_datetime.hour != 0 for target_datetime in target_hour_datetimes],
        dtype=bool,
    )
    midnight_mask = ~regular_mask

    if np.any(regular_mask):
        regular_indices = marker_indices[regular_mask]
        ax.scatter(
            x_values[regular_indices],
            y_values[regular_indices],
            z_values[regular_indices],
            s=HOURLY_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors="black",
            linewidths=1.0,
            label="1-hour marks",
            depthshade=False,
            zorder=7,
        )

    if np.any(midnight_mask):
        midnight_indices = marker_indices[midnight_mask]
        ax.scatter(
            x_values[midnight_indices],
            y_values[midnight_indices],
            z_values[midnight_indices],
            s=MIDNIGHT_MARKER_SIZE,
            marker="*",
            color="black",
            edgecolors="white",
            linewidths=0.8,
            label="Midnight",
            depthshade=False,
            zorder=8,
        )

    for frame_index, target_datetime in zip(
        marker_indices, target_hour_datetimes
    ):
        label_text = (
            "Midnight\n00:00"
            if target_datetime.hour == 0
            else target_datetime.strftime(LOCAL_TIME_LABEL_FORMAT)
        )

        ax.text(
            x_values[frame_index],
            y_values[frame_index],
            z_values[frame_index],
            f"  {label_text}",
            fontsize=8,
            zorder=9,
        )


# ==================================================
# Data and fitting helpers
# ==================================================
def load_wave_cube(pkl_file):
    with open(pkl_file, "rb") as file:
        data = pickle.load(file)

    try:
        label = data[5][0]
        wave_cube = np.asarray(data[5][1], dtype=np.float64)
        date_start = data[7][1]
    except (IndexError, TypeError) as exc:
        raise ValueError(
            "The PKL does not contain the expected data[5][1] perturbation "
            "cube and data[7][1] date/start time."
        ) from exc

    if wave_cube.ndim != 3:
        raise ValueError(
            "Expected perturbation data with shape (time, y, x), "
            f"but got {wave_cube.shape}."
        )

    return label, wave_cube, date_start


def robust_sigma_from_mad(values):
    """
    Estimate a Gaussian-equivalent sigma from the median absolute deviation.
    Returns NaN when the estimate cannot be formed.
    """
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    median_value = np.median(values)
    mad = np.median(np.abs(values - median_value))

    if not np.isfinite(mad) or mad <= 0:
        return np.nan

    return 1.4826 * mad


def preprocess_frame_for_plane_fit(frame, fit_radius_mask, return_steps=False):
    """
    Remove compact sharp residual peaks using a Gaussian-background detector.

    Only detected outlier pixels are changed. The rest of the image is left
    exactly as it was in the raw frame.
    """
    frame = np.asarray(frame, dtype=np.float64)
    finite = np.isfinite(frame)

    steps = {
        "original": frame.copy(),
        "gaussian_background": np.full_like(frame, np.nan, dtype=np.float64),
        "high_pass": np.full_like(frame, np.nan, dtype=np.float64),
        "star_mask": np.zeros_like(frame, dtype=bool),
        "removed": np.zeros_like(frame, dtype=np.float64),
        "preprocessed": frame.copy(),
    }

    if not APPLY_STAR_RESIDUAL_FILTER or BACKGROUND_GAUSSIAN_SIGMA_KM <= 0:
        return (frame.copy(), steps) if return_steps else frame.copy()

    statistics_mask = finite & fit_radius_mask
    if np.count_nonzero(statistics_mask) < MIN_VALID_PIXELS:
        return (frame.copy(), steps) if return_steps else frame.copy()

    sigma_y_pixels = BACKGROUND_GAUSSIAN_SIGMA_KM / DY_KM
    sigma_x_pixels = BACKGROUND_GAUSSIAN_SIGMA_KM / DX_KM

    # Normalized Gaussian filtering prevents NaNs/missing pixels from
    # artificially depressing the local background.
    filled_values = np.where(finite, frame, 0.0)
    finite_weights = finite.astype(np.float64)

    blurred_values = gaussian_filter(
        filled_values,
        sigma=(sigma_y_pixels, sigma_x_pixels),
        mode="nearest",
        truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
    )
    blurred_weights = gaussian_filter(
        finite_weights,
        sigma=(sigma_y_pixels, sigma_x_pixels),
        mode="nearest",
        truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
    )

    gaussian_background = np.full_like(frame, np.nan, dtype=np.float64)
    good_weight = blurred_weights > 1.0e-8
    gaussian_background[good_weight] = (
        blurred_values[good_weight] / blurred_weights[good_weight]
    )

    high_pass = frame - gaussian_background
    hp_values = high_pass[statistics_mask]
    hp_center = float(np.median(hp_values))
    hp_sigma = robust_sigma_from_mad(hp_values)

    star_mask = np.zeros_like(frame, dtype=bool)

    if np.isfinite(hp_sigma) and hp_sigma > 0:
        star_mask = (
            finite
            & np.isfinite(gaussian_background)
            & (
                np.abs(high_pass - hp_center)
                > STAR_RESIDUAL_MAD_SIGMA * hp_sigma
            )
        )

        if STAR_MASK_DILATION_KM is not None and STAR_MASK_DILATION_KM > 0:
            dilation_pixels = max(
                int(round(STAR_MASK_DILATION_KM / min(DX_KM, DY_KM))),
                1,
            )
            star_mask = binary_dilation(
                star_mask,
                iterations=dilation_pixels,
            ) & finite

    preprocessed = frame.copy()
    preprocessed[star_mask] = gaussian_background[star_mask]
    removed = frame - preprocessed

    steps["gaussian_background"] = gaussian_background
    steps["high_pass"] = high_pass
    steps["star_mask"] = star_mask
    steps["removed"] = removed
    steps["preprocessed"] = preprocessed.copy()

    if return_steps:
        return preprocessed, steps

    return preprocessed


def solve_plane_least_squares(x_values, y_values, z_values):
    """Return plane coefficients a, b, c for z = a*x + b*y + c."""
    design_matrix = np.column_stack(
        [
            x_values,
            y_values,
            np.ones(x_values.size, dtype=np.float64),
        ]
    )

    coefficients, _, _, _ = np.linalg.lstsq(
        design_matrix,
        z_values,
        rcond=None,
    )

    return coefficients, design_matrix



def fit_plane_to_image(
    frame_for_fit,
    x_grid_km,
    y_grid_km,
    fit_radius_mask,
):
    """
    Fit z = a*x + b*y + c directly to the supplied image.

    No preprocessing or iterative robust rejection is performed here.
    This makes it possible to use exactly the same fitting method for the
    raw and preprocessed images.
    """
    frame_sample = frame_for_fit[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    x_sample = x_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    y_sample = y_grid_km[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]
    radius_sample = fit_radius_mask[::FIT_PIXEL_STEP, ::FIT_PIXEL_STEP]

    valid = (
        radius_sample
        & np.isfinite(frame_sample)
        & np.isfinite(x_sample)
        & np.isfinite(y_sample)
    )

    if FIT_VALUE_MIN is not None:
        valid &= frame_sample >= FIT_VALUE_MIN
    if FIT_VALUE_MAX is not None:
        valid &= frame_sample <= FIT_VALUE_MAX

    valid_count = int(np.count_nonzero(valid))
    if valid_count < MIN_VALID_PIXELS:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, valid_count

    x_valid = x_sample[valid]
    y_valid = y_sample[valid]
    z_valid = frame_sample[valid]

    coefficients, design_matrix = solve_plane_least_squares(
        x_valid,
        y_valid,
        z_valid,
    )

    a, b, c = coefficients
    residuals = z_valid - design_matrix @ coefficients
    rmse = np.sqrt(np.mean(residuals**2))
    slope = np.hypot(a, b)
    angle_deg = (
        (np.degrees(np.arctan2(b, a)) + 180.0) % 360.0
    ) - 180.0

    return a, b, c, slope, angle_deg, rmse, valid_count


def calculate_vector_limits(x_values, y_values, padding_fraction):
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = np.asarray(x_values)[valid]
    y_values = np.asarray(y_values)[valid]

    if x_values.size == 0:
        return (-1.0, 1.0), (1.0, -1.0)

    x_min = np.nanmin(x_values)
    x_max = np.nanmax(x_values)
    y_min = np.nanmin(y_values)
    y_max = np.nanmax(y_values)

    x_span = x_max - x_min
    y_span = y_max - y_min

    if not np.isfinite(x_span) or x_span <= 0:
        x_span = max(abs(x_min), abs(x_max), 1.0e-6)
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = max(abs(y_min), abs(y_max), 1.0e-6)

    x_padding = padding_fraction * x_span
    y_padding = padding_fraction * y_span

    return (
        (x_min - x_padding, x_max + x_padding),
        (y_max + y_padding, y_min - y_padding),
    )


def safe_relative_output_name(pkl_file):
    """Create a unique folder name from the PKL's path relative to INPUT_ROOT."""
    try:
        relative_path = pkl_file.relative_to(INPUT_ROOT).with_suffix("")
        parts = relative_path.parts
    except ValueError:
        parts = (pkl_file.stem,)

    return "__".join(parts)


# ==================================================
# Plot helper
# ==================================================

# ==================================================
# Plot helpers
# ==================================================
def _draw_tilt_vector_2d_panel(
    ax,
    time_mask,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    panel_title,
    x_limits,
    y_limits,
    label_every_n_hours=1,
):
    selected_all = np.flatnonzero(time_mask)
    scatter = None

    if selected_all.size > 0:
        u_plot = x_slope_per_100km[selected_all]
        v_plot = y_slope_per_100km[selected_all]
        time_plot = times_minutes[selected_all]

        ax.plot(u_plot, v_plot, linewidth=0.8, alpha=0.30)

        scatter = ax.scatter(
            u_plot,
            v_plot,
            c=time_plot,
            cmap="viridis",
            s=26,
            zorder=3,
        )

        vector_indices = selected_all[::TILT_VECTOR_PLOT_STEP]
        if vector_indices.size > 0:
            ax.quiver(
                np.zeros(vector_indices.size),
                np.zeros(vector_indices.size),
                x_slope_per_100km[vector_indices],
                y_slope_per_100km[vector_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        add_hourly_markers_2d(
            ax=ax,
            selected_indices=selected_all,
            x_values=x_slope_per_100km,
            y_values=y_slope_per_100km,
            local_datetimes=local_datetimes,
            start_datetime=start_local_datetime,
            times_minutes=times_minutes,
            label_every_n_hours=label_every_n_hours,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(panel_title)
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    return scatter


def save_tilt_vector_2d_comparison_plot(
    raw_time_mask,
    preprocessed_time_mask,
    output_file,
    title_text,
    x_limits,
    y_limits,
    raw_x_slope_per_100km,
    raw_y_slope_per_100km,
    preprocessed_x_slope_per_100km,
    preprocessed_y_slope_per_100km,
    times_minutes,
    local_datetimes,
    start_local_datetime,
    label_every_n_hours=1,
    colorbar_every_n_hours=1,
    figure_size=(14, 18),
):
    """
    Save a three-panel tilt-vector comparison:

        Top:    Raw
        Middle: Preprocessed
        Bottom: Difference = Preprocessed - Raw

    Raw and preprocessed use identical X/Y limits.

    The difference panel uses its own tighter symmetric X/Y limits so that
    small preprocessing-induced changes are visible.
    """
    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=figure_size,
    )

    scatter_raw = _draw_tilt_vector_2d_panel(
        ax=axes[0],
        time_mask=raw_time_mask,
        x_slope_per_100km=raw_x_slope_per_100km,
        y_slope_per_100km=raw_y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Raw data",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    scatter_pre = _draw_tilt_vector_2d_panel(
        ax=axes[1],
        time_mask=preprocessed_time_mask,
        x_slope_per_100km=preprocessed_x_slope_per_100km,
        y_slope_per_100km=preprocessed_y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title="Preprocessed",
        x_limits=x_limits,
        y_limits=y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    # --------------------------------------------------
    # Difference = preprocessed - raw
    # --------------------------------------------------
    difference_x = (
        preprocessed_x_slope_per_100km
        - raw_x_slope_per_100km
    )
    difference_y = (
        preprocessed_y_slope_per_100km
        - raw_y_slope_per_100km
    )

    difference_time_mask = (
        raw_time_mask
        & preprocessed_time_mask
        & np.isfinite(difference_x)
        & np.isfinite(difference_y)
    )

    difference_indices = np.flatnonzero(difference_time_mask)

    # Use one symmetric scale for both X and Y so direction/magnitude in the
    # difference panel are not visually distorted.
    if difference_indices.size > 0:
        difference_abs_values = np.concatenate(
            [
                np.abs(difference_x[difference_indices]),
                np.abs(difference_y[difference_indices]),
            ]
        )

        difference_limit = np.nanpercentile(
            difference_abs_values,
            99.0,
        )

        if not np.isfinite(difference_limit) or difference_limit <= 0:
            difference_limit = np.nanmax(difference_abs_values)

        if not np.isfinite(difference_limit) or difference_limit <= 0:
            difference_limit = 1.0e-6

        difference_limit *= 1.15
    else:
        difference_limit = 1.0e-6

    # Y is plotted with the same image-coordinate convention as the other
    # tilt-vector panels: positive Y points downward.
    difference_x_limits = (
        -difference_limit,
        +difference_limit,
    )
    difference_y_limits = (
        +difference_limit,
        -difference_limit,
    )

    scatter_difference = _draw_tilt_vector_2d_panel(
        ax=axes[2],
        time_mask=difference_time_mask,
        x_slope_per_100km=difference_x,
        y_slope_per_100km=difference_y,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        panel_title=(
            "Difference: Preprocessed - Raw\n"
            r"$\Delta x = x_{\mathrm{pre}} - x_{\mathrm{raw}},\quad "
            r"\Delta y = y_{\mathrm{pre}} - y_{\mathrm{raw}}$"
        ),
        x_limits=difference_x_limits,
        y_limits=difference_y_limits,
        label_every_n_hours=label_every_n_hours,
    )

    axes[2].set_xlabel(
        "Difference in X slope "
        "(amplitude per 100 km; preprocessed - raw)"
    )
    axes[2].set_ylabel(
        "Difference in Y slope "
        "(amplitude per 100 km; preprocessed - raw)"
    )

    # Add summary statistics directly to the difference panel.
    if difference_indices.size > 0:
        median_dx = np.nanmedian(
            np.abs(difference_x[difference_indices])
        )
        median_dy = np.nanmedian(
            np.abs(difference_y[difference_indices])
        )
        max_dx = np.nanmax(
            np.abs(difference_x[difference_indices])
        )
        max_dy = np.nanmax(
            np.abs(difference_y[difference_indices])
        )

        axes[2].text(
            0.02,
            0.98,
            (
                f"median |Δx| = {median_dx:.6f}\n"
                f"median |Δy| = {median_dy:.6f}\n"
                f"max |Δx| = {max_dx:.6f}\n"
                f"max |Δy| = {max_dy:.6f}"
            ),
            transform=axes[2].transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                edgecolor="0.5",
                alpha=0.85,
            ),
            zorder=10,
        )

    # Use one shared time colorbar for all three panels.
    scatter_for_cbar = (
        scatter_difference
        if scatter_difference is not None
        else (
            scatter_pre
            if scatter_pre is not None
            else scatter_raw
        )
    )

    if scatter_for_cbar is not None:
        cbar = fig.colorbar(
            scatter_for_cbar,
            ax=axes,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )

        combined_time_mask = (
            raw_time_mask
            | preprocessed_time_mask
        )

        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            times_minutes[combined_time_mask],
            label_every_n_hours=colorbar_every_n_hours,
        )

    if label_every_n_hours <= 1:
        subtitle = (
            "Every whole local hour is marked and labeled; "
            "midnight is highlighted"
        )
    else:
        subtitle = (
            "Every whole local hour is marked; labels are shown every "
            f"{label_every_n_hours} hours; midnight is highlighted"
        )

    fig.suptitle(
        f"{title_text}\n"
        f"{subtitle}\n"
        "Raw and preprocessed use the same scale; "
        "difference uses a tighter symmetric scale",
        fontsize=14,
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.055,
        top=0.92,
        hspace=0.32,
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def local_hour_is_in_hourly_average_window(local_hour):
    """
    Return True when the local clock hour falls inside the requested nightly
    window. This supports windows that cross midnight, such as 21:00 to 03:00.
    """
    start_hour = int(HOURLY_AVERAGE_START_HOUR) % 24
    end_hour = int(HOURLY_AVERAGE_END_HOUR) % 24
    local_hour = int(local_hour) % 24

    if start_hour <= end_hour:
        return start_hour <= local_hour <= end_hour

    return (local_hour >= start_hour) or (local_hour <= end_hour)


def compute_hourly_average_rows(
    selected_indices,
    x_slope_per_100km,
    y_slope_per_100km,
    times_minutes,
    local_datetimes,
    input_file,
    output_folder,
    start_local_datetime,
):
    """
    Group the selected frame-center points by local clock hour and average
    the tilt-vector endpoint for each hour.
    """
    selected_indices = np.asarray(selected_indices, dtype=int)

    if selected_indices.size == 0:
        return []

    hour_groups = {}

    for frame_index in selected_indices:
        if not local_hour_is_in_hourly_average_window(
            local_datetimes[frame_index].hour
        ):
            continue

        hour_start = local_datetimes[frame_index].replace(
            minute=0,
            second=0,
            microsecond=0,
        )
        hour_groups.setdefault(hour_start, []).append(int(frame_index))

    rows = []

    night_label = start_local_datetime.strftime("%Y-%m-%d")

    for hour_start in sorted(hour_groups):
        frame_indices = np.asarray(hour_groups[hour_start], dtype=int)

        mean_x = float(np.nanmean(x_slope_per_100km[frame_indices]))
        mean_y = float(np.nanmean(y_slope_per_100km[frame_indices]))
        mean_elapsed_minutes = float(np.nanmean(times_minutes[frame_indices]))

        rows.append(
            {
                "input_file": str(input_file),
                "output_folder": str(output_folder),
                "start_local_datetime": start_local_datetime.isoformat(sep=" "),
                "night_label": night_label,
                "local_date": hour_start.strftime("%Y-%m-%d"),
                "hour_start_local_datetime": hour_start.isoformat(sep=" "),
                "hour_label": hour_start.strftime(LOCAL_TIME_LABEL_FORMAT),
                "is_midnight": int(hour_start.hour == 0),
                "mean_elapsed_minutes": mean_elapsed_minutes,
                "mean_x_slope_per_100km": mean_x,
                "mean_y_slope_per_100km": mean_y,
                "mean_slope_magnitude_per_100km": float(np.hypot(mean_x, mean_y)),
                "frame_count_in_average": int(frame_indices.size),
                "first_frame": int(frame_indices[0]),
                "last_frame": int(frame_indices[-1]),
            }
        )

    return rows


def save_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def load_hourly_average_csv(input_file):
    if not Path(input_file).exists():
        return []

    with open(input_file, "r", newline="") as file:
        return list(csv.DictReader(file))


def save_hourly_average_tilt_plot(
    hourly_rows,
    output_file,
    title_text,
    x_limits,
    y_limits,
    start_local_datetime,
    figure_size=(9, 8),
):
    """
    Save a 2-D tilt plot using one averaged point per selected local hour.
    """
    fig, ax = plt.subplots(figsize=figure_size)

    if hourly_rows:
        mean_x = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_y = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
            dtype=np.float64,
        )
        mean_elapsed = np.array(
            [float(row["mean_elapsed_minutes"]) for row in hourly_rows],
            dtype=np.float64,
        )

        ax.plot(
            mean_x,
            mean_y,
            linewidth=1.2,
            alpha=0.55,
            color="0.35",
            zorder=2,
        )

        scatter = ax.scatter(
            mean_x,
            mean_y,
            c=mean_elapsed,
            cmap="viridis",
            s=95,
            edgecolors="black",
            linewidths=0.8,
            zorder=3,
        )

        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=VECTOR_COLORBAR_PAD,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar,
            start_local_datetime,
            mean_elapsed,
            label_every_n_hours=1,
        )

        midnight_mask = np.array(
            [int(row["is_midnight"]) == 1 for row in hourly_rows],
            dtype=bool,
        )

        if np.any(midnight_mask):
            ax.scatter(
                mean_x[midnight_mask],
                mean_y[midnight_mask],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color="black",
                edgecolors="white",
                linewidths=0.8,
                label="Midnight",
                zorder=4,
            )

        for row, x_here, y_here in zip(hourly_rows, mean_x, mean_y):
            if int(row["is_midnight"]) == 1:
                label_text = "Midnight\n00:00"
                font_weight = "bold"
            else:
                label_text = row["hour_label"]
                font_weight = "normal"

            ax.annotate(
                label_text,
                xy=(x_here, y_here),
                xytext=HOURLY_LABEL_OFFSET_POINTS,
                textcoords="offset points",
                fontsize=8,
                fontweight=font_weight,
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white",
                    edgecolor="0.5",
                    alpha=0.85,
                ),
                zorder=5,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"{title_text}\n"
        "One averaged tilt-vector endpoint is shown for each local hour "
        f"from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def save_all_days_hourly_average_csv(hourly_rows, output_file):
    fieldnames = [
        "input_file",
        "output_folder",
        "start_local_datetime",
        "night_label",
        "local_date",
        "hour_start_local_datetime",
        "hour_label",
        "is_midnight",
        "mean_elapsed_minutes",
        "mean_x_slope_per_100km",
        "mean_y_slope_per_100km",
        "mean_slope_magnitude_per_100km",
        "frame_count_in_average",
        "first_frame",
        "last_frame",
    ]

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hourly_rows)


def save_all_days_hourly_average_plot(hourly_rows, output_file):
    """
    Plot the hourly averaged tilt points from all processed PKLs together.

    Each night/day is assigned its own color. The rows should already be filtered
    to the requested local-hour window.
    """
    if not hourly_rows:
        return

    x_values = np.array(
        [float(row["mean_x_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )
    y_values = np.array(
        [float(row["mean_y_slope_per_100km"]) for row in hourly_rows],
        dtype=np.float64,
    )

    x_limits, y_limits = calculate_vector_limits(
        x_values,
        y_values,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    rows_by_night = {}
    for row in hourly_rows:
        night_key = row.get("night_label", row["start_local_datetime"][:10])
        rows_by_night.setdefault(night_key, []).append(row)

    night_keys = sorted(rows_by_night.keys())

    if len(night_keys) <= 20:
        cmap = plt.get_cmap("tab20", max(len(night_keys), 1))
        colors = [cmap(i) for i in range(len(night_keys))]
    else:
        cmap = plt.get_cmap("nipy_spectral", max(len(night_keys), 1))
        colors = [cmap(i / max(len(night_keys) - 1, 1)) for i in range(len(night_keys))]

    for color, night_key in zip(colors, night_keys):
        night_rows = sorted(
            rows_by_night[night_key],
            key=lambda row: row["hour_start_local_datetime"],
        )

        x_here = np.array(
            [float(row["mean_x_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        y_here = np.array(
            [float(row["mean_y_slope_per_100km"]) for row in night_rows],
            dtype=np.float64,
        )
        midnight_mask_here = np.array(
            [int(row["is_midnight"]) == 1 for row in night_rows],
            dtype=bool,
        )

        ax.plot(
            x_here,
            y_here,
            linewidth=1.4,
            alpha=0.85,
            color=color,
            zorder=2,
            label=night_key,
        )

        ax.scatter(
            x_here,
            y_here,
            s=75,
            color=color,
            edgecolors="black",
            linewidths=0.5,
            alpha=0.95,
            zorder=3,
        )

        if np.any(midnight_mask_here):
            ax.scatter(
                x_here[midnight_mask_here],
                y_here[midnight_mask_here],
                s=MIDNIGHT_MARKER_SIZE,
                marker="*",
                color=color,
                edgecolors="black",
                linewidths=0.8,
                zorder=4,
            )

    ax.axhline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.8, alpha=0.6)
    ax.scatter([0.0], [0.0], s=30, c="black", marker="o", zorder=4)

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        "Hourly Averaged Tilt Points: All Days\n"
        f"Local-hour averages from {HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00; each day has a different color"
    )
    ax.set_xlabel("Mean X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Mean Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=8,
        framealpha=0.9,
        title="Day",
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


# ==================================================
# Preprocessing diagnostics
# ==================================================
def _symmetric_plot_limit(values, percentile=99.0, mask=None):
    values = np.asarray(values, dtype=np.float64)
    valid = np.isfinite(values)
    if mask is not None:
        valid &= mask
    if not np.any(valid):
        return 1.0
    vmax = np.nanpercentile(np.abs(values[valid]), percentile)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    return float(vmax)


def add_diagnostic_slope_vector(
    ax,
    a_here,
    b_here,
    vector_reference_slope,
    nx,
    ny,
):
    """
    Overlay one plane-slope vector on an image.

    vector_reference_slope is shared by the original/preprocessed diagnostic
    panels for the current frame, so the two arrow lengths are directly
    comparable.
    """
    if (
        not np.isfinite(a_here)
        or not np.isfinite(b_here)
        or not np.isfinite(vector_reference_slope)
        or vector_reference_slope <= 0
    ):
        return

    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0

    max_arrow_length_pix = (
        DIAGNOSTIC_ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)
    )

    arrow_u = (a_here / vector_reference_slope) * max_arrow_length_pix
    arrow_v = (b_here / vector_reference_slope) * max_arrow_length_pix

    ax.scatter(
        [center_x],
        [center_y],
        s=CENTER_MARKER_SIZE,
        c=CENTER_MARKER_COLOR,
        marker="o",
        zorder=6,
    )

    ax.quiver(
        [center_x],
        [center_y],
        [arrow_u],
        [arrow_v],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color=ARROW_COLOR,
        width=0.008,
        headwidth=4.5,
        headlength=6.0,
        headaxislength=5.5,
        zorder=7,
    )


def save_preprocessing_diagnostic_plot(
    frame_index,
    wave_cube,
    x_grid_km,
    y_grid_km,
    fit_radius_mask,
    raw_a_values,
    raw_b_values,
    preprocessed_a_values,
    preprocessed_b_values,
    times_minutes,
    local_datetimes,
    out_dir,
):
    """
    Save a compact diagnostic figure for the retained preprocessing method.

    Top row:
      1. Raw + raw slope vector
      2. Gaussian background
      3. High-pass = raw - background

    Bottom row:
      4. Detected replacement mask
      5. Removed component
      6. Final preprocessed + preprocessed slope vector
    """
    nt, ny, nx = wave_cube.shape
    if frame_index < 0 or frame_index >= nt:
        return

    original = np.asarray(wave_cube[frame_index], dtype=np.float64)
    preprocessed, steps = preprocess_frame_for_plane_fit(
        original,
        fit_radius_mask,
        return_steps=True,
    )

    raw_a = raw_a_values[frame_index]
    raw_b = raw_b_values[frame_index]
    pre_a = preprocessed_a_values[frame_index]
    pre_b = preprocessed_b_values[frame_index]

    raw_slope = np.hypot(raw_a, raw_b)
    pre_slope = np.hypot(pre_a, pre_b)

    comparison_slopes = np.array([raw_slope, pre_slope], dtype=np.float64)
    comparison_slopes = comparison_slopes[np.isfinite(comparison_slopes)]
    vector_reference_slope = (
        float(np.nanmax(comparison_slopes))
        if comparison_slopes.size > 0
        else 1.0
    )
    if not np.isfinite(vector_reference_slope) or vector_reference_slope <= 0:
        vector_reference_slope = 1.0

    main_vmax = _symmetric_plot_limit(
        np.concatenate(
            [
                original[np.isfinite(original)],
                preprocessed[np.isfinite(preprocessed)],
                steps["gaussian_background"][
                    np.isfinite(steps["gaussian_background"])
                ],
            ]
        ),
        percentile=99.0,
    )
    high_pass_vmax = _symmetric_plot_limit(
        steps["high_pass"],
        percentile=99.0,
        mask=fit_radius_mask,
    )
    removed_vmax = _symmetric_plot_limit(
        steps["removed"],
        percentile=REMOVED_COMPONENT_SCALE_PERCENTILE,
        mask=fit_radius_mask,
    )

    center_x = (nx - 1) / 2.0
    center_y = (ny - 1) / 2.0

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 11),
    )

    panels = [
        (
            axes[0, 0],
            original,
            (
                "Raw + slope vector\n"
                f"x = {100.0 * raw_a:+.5f}, "
                f"y = {100.0 * raw_b:+.5f}, "
                f"|s| = {100.0 * raw_slope:.5f} amp/100 km"
            ),
            CMAP,
            main_vmax,
            "Amplitude",
        ),
        (
            axes[0, 1],
            steps["gaussian_background"],
            f"Gaussian background\nsigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km",
            CMAP,
            main_vmax,
            "Background amplitude",
        ),
        (
            axes[0, 2],
            steps["high_pass"],
            "High-pass = raw - background",
            REMOVED_COMPONENT_CMAP,
            high_pass_vmax,
            "High-pass amplitude",
        ),
        (
            axes[1, 1],
            steps["removed"],
            "Removed component\n(raw - preprocessed)",
            REMOVED_COMPONENT_CMAP,
            removed_vmax,
            "Removed amplitude",
        ),
        (
            axes[1, 2],
            preprocessed,
            (
                "Final preprocessed + slope vector\n"
                f"x = {100.0 * pre_a:+.5f}, "
                f"y = {100.0 * pre_b:+.5f}, "
                f"|s| = {100.0 * pre_slope:.5f} amp/100 km"
            ),
            CMAP,
            main_vmax,
            "Amplitude",
        ),
    ]

    for ax, panel_data, title_text, cmap, vmax_here, cbar_label in panels:
        image = ax.imshow(
            panel_data,
            origin="upper",
            cmap=cmap,
            vmin=-vmax_here,
            vmax=vmax_here,
        )
        fit_circle = Ellipse(
            xy=(center_x, center_y),
            width=2.0 * FIT_RADIUS_KM / DX_KM,
            height=2.0 * FIT_RADIUS_KM / DY_KM,
            fill=False,
            edgecolor="black",
            linestyle="--",
            linewidth=1.1,
            alpha=0.85,
            zorder=4,
        )
        ax.add_patch(fit_circle)
        ax.set_title(title_text, fontsize=10.5)
        ax.set_xlabel("X pixel")
        ax.set_ylabel("Y pixel")
        cbar = fig.colorbar(
            image,
            ax=ax,
            fraction=0.040,
            pad=0.025,
            aspect=30,
        )
        cbar.set_label(cbar_label)

    mask_image = axes[1, 0].imshow(
        steps["star_mask"].astype(np.float64),
        origin="upper",
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )
    fit_circle = Ellipse(
        xy=(center_x, center_y),
        width=2.0 * FIT_RADIUS_KM / DX_KM,
        height=2.0 * FIT_RADIUS_KM / DY_KM,
        fill=False,
        edgecolor="red",
        linestyle="--",
        linewidth=1.1,
        alpha=0.85,
        zorder=4,
    )
    axes[1, 0].add_patch(fit_circle)
    axes[1, 0].set_title("Detected peak mask\nwhite = replaced pixels")
    axes[1, 0].set_xlabel("X pixel")
    axes[1, 0].set_ylabel("Y pixel")
    cbar_mask = fig.colorbar(
        mask_image,
        ax=axes[1, 0],
        fraction=0.040,
        pad=0.025,
        aspect=30,
    )
    cbar_mask.set_ticks([0.0, 1.0])
    cbar_mask.set_ticklabels(["0", "1"])
    cbar_mask.set_label("Mask value")

    add_diagnostic_slope_vector(
        axes[0, 0],
        raw_a,
        raw_b,
        vector_reference_slope,
        nx,
        ny,
    )
    add_diagnostic_slope_vector(
        axes[1, 2],
        pre_a,
        pre_b,
        vector_reference_slope,
        nx,
        ny,
    )

    local_time_text = local_datetimes[frame_index].strftime(
        LOCAL_TIME_EXACT_FORMAT
    )
    replaced_pixel_count = int(np.count_nonzero(steps["star_mask"]))
    inside_fit_count = int(
        np.count_nonzero(steps["star_mask"] & fit_radius_mask)
    )

    fig.suptitle(
        f"Gaussian-Background Peak Diagnostic | "
        f"Frame {frame_index} | "
        f"t = {times_minutes[frame_index]:.1f} min | "
        f"Local time = {local_time_text}\n"
        f"Background sigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km | "
        f"threshold = {STAR_RESIDUAL_MAD_SIGMA:g} MAD-sigma | "
        f"mask dilation = {STAR_MASK_DILATION_KM:g} km | "
        f"fit radius = {FIT_RADIUS_KM:g} km | "
        f"replaced = {replaced_pixel_count} total "
        f"({inside_fit_count} inside fit radius)",
        fontsize=13,
    )

    diagnostic_dir = out_dir / "preprocessing_diagnostics"
    diagnostic_dir.mkdir(parents=True, exist_ok=True)

    output_file = (
        diagnostic_dir
        / f"diagnostic_frame_{frame_index:04d}_"
          f"t_{times_minutes[frame_index]:07.1f}_min.png"
    )

    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.93))
    fig.savefig(output_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig)


def build_diagnostic_frame_indices(
    nt,
    times_minutes,
):
    """
    Combine requested frame numbers with frames nearest the requested
    elapsed times.
    """
    requested_indices = set()

    for frame_index in DIAGNOSTIC_FRAME_INDICES:
        frame_index = int(frame_index)

        if 0 <= frame_index < nt:
            requested_indices.add(frame_index)

    finite_time = np.isfinite(times_minutes)

    if np.any(finite_time):
        valid_indices = np.flatnonzero(
            finite_time
        )

        for target_time_min in DIAGNOSTIC_TIMES_MIN:
            nearest_position = int(
                np.argmin(
                    np.abs(
                        times_minutes[valid_indices]
                        - float(target_time_min)
                    )
                )
            )

            requested_indices.add(
                int(
                    valid_indices[
                        nearest_position
                    ]
                )
            )

    return sorted(requested_indices)


# ==================================================
# Process one PKL
# ==================================================
def process_one_pkl(pkl_file):
    print("\n" + "=" * 80)
    print(f"Reading: {pkl_file}")

    label, wave_cube, date_start = load_wave_cube(pkl_file)
    start_local_datetime = parse_pkl_start_datetime(date_start)

    if start_local_datetime.year not in TARGET_YEARS:
        print(
            f"Skipped: internal year {start_local_datetime.year} "
            f"is not in {sorted(TARGET_YEARS)}"
        )
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_year",
                "output_folder": "",
                "frame_count": "",
                "successful_fits": "",
                "hourly_average_csv": "",
                "message": f"Year {start_local_datetime.year} not selected",
            },
            [],
        )

    output_name = safe_relative_output_name(pkl_file)
    out_dir = OUTPUT_ROOT / str(start_local_datetime.year) / output_name
    out_dir.mkdir(parents=True, exist_ok=True)

    npz_file = out_dir / "plane_fit_results.npz"
    hourly_average_csv_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )
    hourly_average_plot_file = (
        out_dir
        / f"plane_tilt_vector_2d_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    required_existing_files = [
        npz_file,
        hourly_average_csv_file,
        hourly_average_plot_file,
    ]

    if SKIP_IF_RESULTS_EXIST and all(file.exists() for file in required_existing_files):
        print(f"Skipped existing results in: {out_dir}")
        hourly_rows = load_hourly_average_csv(hourly_average_csv_file)
        return (
            {
                "input_file": str(pkl_file),
                "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
                "status": "skipped_existing",
                "output_folder": str(out_dir),
                "frame_count": wave_cube.shape[0],
                "successful_fits": "",
                "hourly_average_csv": str(hourly_average_csv_file),
                "message": "All required result files already exist",
            },
            hourly_rows,
        )

    nt, ny, nx = wave_cube.shape
    print(f"Loaded entry: {label}")
    print(f"Perturbation cube shape: {wave_cube.shape}")
    print(
        "PKL interval start datetime: "
        f"{start_local_datetime.strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    frame_numbers = np.arange(nt, dtype=int)

    # data[7][1] is the beginning of the first frame interval.
    # Each fitted point is assigned to the center of its interval.
    times_seconds = (frame_numbers + 0.5) * DT_SECONDS
    times_minutes = times_seconds / 60.0

    local_datetimes = np.array(
        [
            local_datetime_from_elapsed_minutes(
                start_local_datetime,
                elapsed_minutes,
            )
            for elapsed_minutes in times_minutes
        ],
        dtype=object,
    )

    local_datetime_iso = np.array(
        [
            local_datetime.isoformat(sep=" ")
            for local_datetime in local_datetimes
        ],
        dtype=str,
    )

    print(
        "First frame-center datetime: "
        f"{local_datetimes[0].strftime(LOCAL_DATETIME_PRINT_FORMAT)}"
    )

    x_1d_km = (
        np.arange(nx, dtype=np.float64) - (nx - 1) / 2.0
    ) * DX_KM
    y_1d_km = (
        np.arange(ny, dtype=np.float64) - (ny - 1) / 2.0
    ) * DY_KM
    x_grid_km, y_grid_km = np.meshgrid(x_1d_km, y_1d_km)

    radial_distance_km = np.hypot(x_grid_km, y_grid_km)
    fit_radius_mask = radial_distance_km <= FIT_RADIUS_KM
    fit_radius_pixel_count = int(np.count_nonzero(fit_radius_mask))

    print(
        f"Plane-fit region: radius <= {FIT_RADIUS_KM:.1f} km "
        f"({fit_radius_pixel_count} pixels before other validity filters)"
    )
    if APPLY_STAR_RESIDUAL_FILTER:
        print(
            "Spatial preprocessing: Gaussian-background peak replacement "
            f"(sigma = {BACKGROUND_GAUSSIAN_SIGMA_KM:g} km, "
            f"threshold = {STAR_RESIDUAL_MAD_SIGMA:g} MAD-sigma, "
            f"mask dilation = {STAR_MASK_DILATION_KM:g} km)"
        )
    else:
        print("Spatial preprocessing: disabled")

    # Raw plane-fit results
    raw_a_values = np.full(nt, np.nan, dtype=np.float64)
    raw_b_values = np.full(nt, np.nan, dtype=np.float64)
    raw_c_values = np.full(nt, np.nan, dtype=np.float64)
    raw_slope_values = np.full(nt, np.nan, dtype=np.float64)
    raw_angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    raw_rmse_values = np.full(nt, np.nan, dtype=np.float64)
    raw_valid_pixel_counts = np.zeros(nt, dtype=int)

    # Preprocessed plane-fit results
    a_values = np.full(nt, np.nan, dtype=np.float64)
    b_values = np.full(nt, np.nan, dtype=np.float64)
    c_values = np.full(nt, np.nan, dtype=np.float64)
    slope_values = np.full(nt, np.nan, dtype=np.float64)
    angle_values_deg = np.full(nt, np.nan, dtype=np.float64)
    rmse_values = np.full(nt, np.nan, dtype=np.float64)
    valid_pixel_counts = np.zeros(nt, dtype=int)

    print("Fitting planes to raw and preprocessed frames...")

    for frame_index in range(nt):
        raw_frame = wave_cube[frame_index]

        (
            raw_a_values[frame_index],
            raw_b_values[frame_index],
            raw_c_values[frame_index],
            raw_slope_values[frame_index],
            raw_angle_values_deg[frame_index],
            raw_rmse_values[frame_index],
            raw_valid_pixel_counts[frame_index],
        ) = fit_plane_to_image(
            raw_frame,
            x_grid_km,
            y_grid_km,
            fit_radius_mask,
        )

        preprocessed_frame = preprocess_frame_for_plane_fit(
            raw_frame,
            fit_radius_mask,
        )

        (
            a_values[frame_index],
            b_values[frame_index],
            c_values[frame_index],
            slope_values[frame_index],
            angle_values_deg[frame_index],
            rmse_values[frame_index],
            valid_pixel_counts[frame_index],
        ) = fit_plane_to_image(
            preprocessed_frame,
            x_grid_km,
            y_grid_km,
            fit_radius_mask,
        )

        if (
            frame_index == 0
            or frame_index == nt - 1
            or (frame_index + 1) % 25 == 0
        ):
            print(f"  Processed frame {frame_index + 1}/{nt}")

    successful_fits = int(
        np.count_nonzero(
            np.isfinite(slope_values) & np.isfinite(angle_values_deg)
        )
    )

    # ----------------------------------------------
    # Optional preprocessing / fit diagnostics
    # ----------------------------------------------
    if MAKE_PREPROCESSING_DIAGNOSTICS:
        diagnostic_indices = build_diagnostic_frame_indices(
            nt,
            times_minutes,
        )

        print(
            "Saving preprocessing diagnostic plots for frames: "
            + ", ".join(
                str(index)
                for index in diagnostic_indices
            )
        )

        for diagnostic_frame_index in diagnostic_indices:
            save_preprocessing_diagnostic_plot(
                frame_index=diagnostic_frame_index,
                wave_cube=wave_cube,
                x_grid_km=x_grid_km,
                y_grid_km=y_grid_km,
                fit_radius_mask=fit_radius_mask,
                raw_a_values=raw_a_values,
                raw_b_values=raw_b_values,
                preprocessed_a_values=a_values,
                preprocessed_b_values=b_values,
                times_minutes=times_minutes,
                local_datetimes=local_datetimes,
                out_dir=out_dir,
            )

    # ----------------------------------------------
    # Save CSV and NPZ
    # ----------------------------------------------
    csv_file = out_dir / "plane_fit_results.csv"

    with open(csv_file, "w", newline="") as file:
        writer = csv.writer(file)

        writer.writerow(
            [
                "frame",
                "frame_center_time_seconds",
                "frame_center_time_minutes",
                "local_datetime",
                "local_time",
                "raw_slope_x_amplitude_per_km",
                "raw_slope_y_amplitude_per_km",
                "raw_plane_intercept",
                "raw_slope_magnitude_amplitude_per_km",
                "raw_increase_direction_deg",
                "raw_fit_rmse",
                "raw_valid_pixel_count",
                "preprocessed_slope_x_amplitude_per_km",
                "preprocessed_slope_y_amplitude_per_km",
                "preprocessed_plane_intercept",
                "preprocessed_slope_magnitude_amplitude_per_km",
                "preprocessed_increase_direction_deg",
                "preprocessed_fit_rmse",
                "preprocessed_valid_pixel_count",
            ]
        )

        for frame_index in range(nt):
            writer.writerow(
                [
                    frame_numbers[frame_index],
                    times_seconds[frame_index],
                    times_minutes[frame_index],
                    local_datetime_iso[frame_index],
                    local_datetimes[frame_index].strftime(
                        LOCAL_TIME_EXACT_FORMAT
                    ),
                    raw_a_values[frame_index],
                    raw_b_values[frame_index],
                    raw_c_values[frame_index],
                    raw_slope_values[frame_index],
                    raw_angle_values_deg[frame_index],
                    raw_rmse_values[frame_index],
                    raw_valid_pixel_counts[frame_index],
                    a_values[frame_index],
                    b_values[frame_index],
                    c_values[frame_index],
                    slope_values[frame_index],
                    angle_values_deg[frame_index],
                    rmse_values[frame_index],
                    valid_pixel_counts[frame_index],
                ]
            )

    np.savez(
        npz_file,
        input_file=str(pkl_file),
        frame=frame_numbers,
        time_seconds=times_seconds,
        time_minutes=times_minutes,
        time_reference="frame_center",
        frame_duration_seconds=DT_SECONDS,
        fit_radius_km=FIT_RADIUS_KM,
        preprocessing_method="gaussian_background_outlier_replacement",
        background_gaussian_sigma_km=BACKGROUND_GAUSSIAN_SIGMA_KM,
        background_gaussian_truncate=BACKGROUND_GAUSSIAN_TRUNCATE,
        star_residual_filter_enabled=APPLY_STAR_RESIDUAL_FILTER,
        star_residual_mad_sigma=STAR_RESIDUAL_MAD_SIGMA,
        star_mask_dilation_km=STAR_MASK_DILATION_KM,
        local_datetime=local_datetime_iso,
        start_local_datetime=start_local_datetime.isoformat(sep=" "),

        raw_slope_x_amplitude_per_km=raw_a_values,
        raw_slope_y_amplitude_per_km=raw_b_values,
        raw_plane_intercept=raw_c_values,
        raw_slope_magnitude_amplitude_per_km=raw_slope_values,
        raw_increase_direction_deg=raw_angle_values_deg,
        raw_fit_rmse=raw_rmse_values,
        raw_valid_pixel_count=raw_valid_pixel_counts,

        preprocessed_slope_x_amplitude_per_km=a_values,
        preprocessed_slope_y_amplitude_per_km=b_values,
        preprocessed_plane_intercept=c_values,
        preprocessed_slope_magnitude_amplitude_per_km=slope_values,
        preprocessed_increase_direction_deg=angle_values_deg,
        preprocessed_fit_rmse=rmse_values,
        preprocessed_valid_pixel_count=valid_pixel_counts,

        # Backward-compatible aliases: these are PREPROCESSED values.
        slope_x_amplitude_per_km=a_values,
        slope_y_amplitude_per_km=b_values,
        plane_intercept=c_values,
        slope_magnitude_amplitude_per_km=slope_values,
        increase_direction_deg_wrapped_minus180_to_180_image_coordinates=angle_values_deg,
        fit_rmse=rmse_values,
        valid_pixel_count=valid_pixel_counts,
    )

    # ----------------------------------------------
    # Angle plot
    # ----------------------------------------------
    fig_angle, ax_angle = plt.subplots(figsize=(11, 5))

    ax_angle.plot(
        frame_numbers,
        raw_angle_values_deg,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    ax_angle.plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
        label="Preprocessed",
    )

    ax_angle.set_title("Plane-fit Increasing Direction vs Frame")
    ax_angle.set_xlabel("Frame")
    ax_angle.set_ylabel(
        "Direction of perturbation increase "
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks(np.arange(-180, 181, 45))
    ax_angle.grid(True, alpha=0.3)
    ax_angle.legend()

    fig_angle.tight_layout()
    angle_plot_file = out_dir / "plane_angle_vs_frame.png"
    fig_angle.savefig(angle_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_angle)

    # ----------------------------------------------
    # Slope plot
    # ----------------------------------------------
    fig_slope, ax_slope = plt.subplots(figsize=(11, 5))

    ax_slope.plot(
        frame_numbers,
        raw_slope_values,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    ax_slope.plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
        label="Preprocessed",
    )

    ax_slope.set_title("Plane-fit Slope Magnitude vs Frame")
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Slope magnitude (amplitude per km)")
    ax_slope.grid(True, alpha=0.3)
    ax_slope.legend()

    fig_slope.tight_layout()
    slope_plot_file = out_dir / "plane_slope_vs_frame.png"
    fig_slope.savefig(slope_plot_file, dpi=DPI, bbox_inches="tight")
    plt.close(fig_slope)

    # ----------------------------------------------
    # Combined angle/slope comparison
    # ----------------------------------------------
    fig_combined, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 8),
        sharex=True,
    )

    axes[0].plot(
        frame_numbers,
        raw_angle_values_deg,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    axes[0].plot(
        frame_numbers,
        angle_values_deg,
        linewidth=1.3,
        label="Preprocessed",
    )
    axes[0].set_title("Plane-fit Direction and Slope: Raw vs Preprocessed")
    axes[0].set_ylabel(
        "Increase direction\n"
        "(-180° to 180°; 0° right, 90° down, -90° up)"
    )
    axes[0].set_ylim(-180, 180)
    axes[0].set_yticks(np.arange(-180, 181, 45))
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        frame_numbers,
        raw_slope_values,
        linewidth=1.0,
        alpha=0.65,
        label="Raw",
    )
    axes[1].plot(
        frame_numbers,
        slope_values,
        linewidth=1.3,
        label="Preprocessed",
    )
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Slope magnitude\n(amplitude per km)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig_combined.tight_layout()
    combined_plot_file = out_dir / "plane_angle_and_slope_vs_frame.png"
    fig_combined.savefig(
        combined_plot_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_combined)

    # ----------------------------------------------
    # Physical tilt vectors: raw vs preprocessed
    # ----------------------------------------------
    raw_x_slope_per_100km = 100.0 * raw_a_values
    raw_y_slope_per_100km = 100.0 * raw_b_values

    x_slope_per_100km = 100.0 * a_values
    y_slope_per_100km = 100.0 * b_values

    raw_finite_vector = (
        np.isfinite(raw_x_slope_per_100km)
        & np.isfinite(raw_y_slope_per_100km)
        & np.isfinite(times_minutes)
    )
    preprocessed_finite_vector = (
        np.isfinite(x_slope_per_100km)
        & np.isfinite(y_slope_per_100km)
        & np.isfinite(times_minutes)
    )

    raw_tilt_plot_time_mask = (
        raw_finite_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )
    preprocessed_tilt_plot_time_mask = (
        preprocessed_finite_vector
        & (times_minutes >= TILT_PLOT_START_TIME_MIN)
        & (times_minutes <= TILT_PLOT_END_TIME_MIN)
    )

    whole_combined_x = np.concatenate(
        [
            raw_x_slope_per_100km[raw_finite_vector],
            x_slope_per_100km[preprocessed_finite_vector],
        ]
    )
    whole_combined_y = np.concatenate(
        [
            raw_y_slope_per_100km[raw_finite_vector],
            y_slope_per_100km[preprocessed_finite_vector],
        ]
    )

    whole_vector_x_limits, whole_vector_y_limits = calculate_vector_limits(
        whole_combined_x,
        whole_combined_y,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    zoom_combined_x = np.concatenate(
        [
            raw_x_slope_per_100km[raw_tilt_plot_time_mask],
            x_slope_per_100km[preprocessed_tilt_plot_time_mask],
        ]
    )
    zoom_combined_y = np.concatenate(
        [
            raw_y_slope_per_100km[raw_tilt_plot_time_mask],
            y_slope_per_100km[preprocessed_tilt_plot_time_mask],
        ]
    )

    zoom_vector_x_limits, zoom_vector_y_limits = calculate_vector_limits(
        zoom_combined_x,
        zoom_combined_y,
        VECTOR_PLOT_PADDING_FRACTION,
    )

    # ----------------------------------------------
    # 3-D tilt-vector-vs-time comparison
    # top = raw, bottom = preprocessed
    # ----------------------------------------------
    fig_tilt_3d = plt.figure(figsize=(14, 15))
    ax_raw_3d = fig_tilt_3d.add_subplot(211, projection="3d")
    ax_pre_3d = fig_tilt_3d.add_subplot(212, projection="3d")

    comparison_3d = [
        (
            ax_raw_3d,
            raw_tilt_plot_time_mask,
            raw_x_slope_per_100km,
            raw_y_slope_per_100km,
            "Raw data",
        ),
        (
            ax_pre_3d,
            preprocessed_tilt_plot_time_mask,
            x_slope_per_100km,
            y_slope_per_100km,
            "Preprocessed",
        ),
    ]

    scatter_for_3d_cbar = None

    for (
        ax_here,
        mask_here,
        x_here_all,
        y_here_all,
        panel_title,
    ) in comparison_3d:
        selected_indices = np.flatnonzero(mask_here)

        if selected_indices.size > 0:
            x_here = x_here_all[selected_indices]
            y_here = y_here_all[selected_indices]
            t_here = times_minutes[selected_indices]

            ax_here.plot(
                x_here,
                y_here,
                t_here,
                linewidth=1.0,
                alpha=0.55,
            )

            scatter_here = ax_here.scatter(
                x_here,
                y_here,
                t_here,
                c=t_here,
                cmap="viridis",
                s=18,
                depthshade=False,
            )
            scatter_for_3d_cbar = scatter_here

            add_hourly_markers_3d(
                ax=ax_here,
                selected_indices=selected_indices,
                x_values=x_here_all,
                y_values=y_here_all,
                z_values=times_minutes,
                local_datetimes=local_datetimes,
                start_datetime=start_local_datetime,
                times_minutes=times_minutes,
            )

            hour_datetimes_3d, hour_elapsed_3d = make_hour_boundaries(
                start_local_datetime,
                times_minutes[selected_indices],
            )
            if hour_elapsed_3d.size > 0:
                ax_here.set_zticks(hour_elapsed_3d)
                ax_here.set_zticklabels(
                    [
                        (
                            "Midnight\n00:00"
                            if hour_datetime.hour == 0
                            else hour_datetime.strftime(
                                LOCAL_TIME_LABEL_FORMAT
                            )
                        )
                        + f"\n({elapsed_minutes:.0f} min)"
                        for hour_datetime, elapsed_minutes in zip(
                            hour_datetimes_3d,
                            hour_elapsed_3d,
                        )
                    ]
                )

        ax_here.set_xlim(*zoom_vector_x_limits)
        ax_here.set_ylim(*zoom_vector_y_limits)
        ax_here.set_box_aspect((1, 1, 1))
        ax_here.set_title(panel_title)
        ax_here.set_xlabel(
            "X slope (amplitude per 100 km; right positive)"
        )
        ax_here.set_ylabel(
            "Y slope (amplitude per 100 km; down positive)"
        )
        ax_here.set_zlabel("Elapsed time / local time")
        ax_here.grid(True, alpha=0.3)

        handles_here, _ = ax_here.get_legend_handles_labels()
        if handles_here:
            ax_here.legend()

    if scatter_for_3d_cbar is not None:
        combined_zoom_mask = (
            raw_tilt_plot_time_mask | preprocessed_tilt_plot_time_mask
        )
        cbar_3d = fig_tilt_3d.colorbar(
            scatter_for_3d_cbar,
            ax=[ax_raw_3d, ax_pre_3d],
            pad=0.05,
            shrink=0.72,
            fraction=VECTOR_COLORBAR_FRACTION,
            aspect=VECTOR_COLORBAR_ASPECT,
        )
        apply_hourly_elapsed_local_labels_to_colorbar(
            cbar_3d,
            start_local_datetime,
            times_minutes[combined_zoom_mask],
        )

    fig_tilt_3d.suptitle(
        "Plane-fit Tilt Vector vs Time: Raw vs Preprocessed\n"
        "Top = raw data; bottom = preprocessed; same X/Y limits",
        fontsize=14,
    )
    fig_tilt_3d.subplots_adjust(
        left=0.04,
        right=0.88,
        bottom=0.04,
        top=0.93,
        hspace=0.18,
    )

    tilt_vector_3d_file = out_dir / "plane_tilt_vector_3d_vs_time.png"
    fig_tilt_3d.savefig(
        tilt_vector_3d_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_tilt_3d)

    # ----------------------------------------------
    # 2-D whole-time and zoomed comparisons
    # ----------------------------------------------
    tilt_vector_2d_whole_file = (
        out_dir / "plane_tilt_vector_2d_whole_time_colored_by_time.png"
    )

    tilt_vector_2d_zoom_file = (
        out_dir
        / f"plane_tilt_vector_2d_zoom_{TILT_PLOT_START_TIME_MIN:.0f}_to_"
          f"{TILT_PLOT_END_TIME_MIN:.0f}_min_colored_by_time.png"
    )

    save_tilt_vector_2d_comparison_plot(
        raw_time_mask=raw_finite_vector,
        preprocessed_time_mask=preprocessed_finite_vector,
        output_file=tilt_vector_2d_whole_file,
        title_text="Plane-fit Tilt Vector Endpoints: Whole Time Range",
        x_limits=whole_vector_x_limits,
        y_limits=whole_vector_y_limits,
        raw_x_slope_per_100km=raw_x_slope_per_100km,
        raw_y_slope_per_100km=raw_y_slope_per_100km,
        preprocessed_x_slope_per_100km=x_slope_per_100km,
        preprocessed_y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=WHOLE_TIME_LABEL_EVERY_N_HOURS,
        colorbar_every_n_hours=WHOLE_TIME_COLORBAR_EVERY_N_HOURS,
        figure_size=(18, 21),
    )

    save_tilt_vector_2d_comparison_plot(
        raw_time_mask=raw_tilt_plot_time_mask,
        preprocessed_time_mask=preprocessed_tilt_plot_time_mask,
        output_file=tilt_vector_2d_zoom_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: "
            f"{TILT_PLOT_START_TIME_MIN:.0f}–"
            f"{TILT_PLOT_END_TIME_MIN:.0f} Minutes"
        ),
        x_limits=zoom_vector_x_limits,
        y_limits=zoom_vector_y_limits,
        raw_x_slope_per_100km=raw_x_slope_per_100km,
        raw_y_slope_per_100km=raw_y_slope_per_100km,
        preprocessed_x_slope_per_100km=x_slope_per_100km,
        preprocessed_y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        start_local_datetime=start_local_datetime,
        label_every_n_hours=1,
        colorbar_every_n_hours=1,
        figure_size=(14, 18),
    )

    # New: hourly-averaged zoomed tilt points
    hourly_average_rows = compute_hourly_average_rows(
        selected_indices=np.flatnonzero(preprocessed_finite_vector),
        x_slope_per_100km=x_slope_per_100km,
        y_slope_per_100km=y_slope_per_100km,
        times_minutes=times_minutes,
        local_datetimes=local_datetimes,
        input_file=pkl_file,
        output_folder=out_dir,
        start_local_datetime=start_local_datetime,
    )

    save_hourly_average_csv(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_csv_file,
    )

    if hourly_average_rows:
        hourly_average_x = np.array(
            [
                float(row["mean_x_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )
        hourly_average_y = np.array(
            [
                float(row["mean_y_slope_per_100km"])
                for row in hourly_average_rows
            ],
            dtype=np.float64,
        )

        hourly_average_x_limits, hourly_average_y_limits = (
            calculate_vector_limits(
                hourly_average_x,
                hourly_average_y,
                VECTOR_PLOT_PADDING_FRACTION,
            )
        )
    else:
        hourly_average_x_limits = (-1.0, 1.0)
        hourly_average_y_limits = (1.0, -1.0)

    save_hourly_average_tilt_plot(
        hourly_rows=hourly_average_rows,
        output_file=hourly_average_plot_file,
        title_text=(
            "Plane-fit Tilt Vector Endpoints: Hourly Averages\n"
            f"{HOURLY_AVERAGE_START_HOUR:02d}:00–"
            f"{HOURLY_AVERAGE_END_HOUR:02d}:00 Local Time"
        ),
        x_limits=hourly_average_x_limits,
        y_limits=hourly_average_y_limits,
        start_local_datetime=start_local_datetime,
        figure_size=ZOOM_TIME_TILT_FIGSIZE,
    )

    # ----------------------------------------------
    # Animation: raw vs preprocessed
    # ----------------------------------------------
    saved_animation_file = ""

    if MAKE_ANIMATION:
        all_slopes_for_scale = np.concatenate(
            [
                raw_slope_values[np.isfinite(raw_slope_values)],
                slope_values[np.isfinite(slope_values)],
            ]
        )

        if all_slopes_for_scale.size > 0:
            slope_ref = np.nanpercentile(
                all_slopes_for_scale,
                ARROW_LENGTH_PERCENTILE,
            )
        else:
            slope_ref = 1.0

        if not np.isfinite(slope_ref) or slope_ref <= 0:
            slope_ref = 1.0

        max_arrow_length_pix = ARROW_MAX_FRACTION_OF_IMAGE * min(nx, ny)

        raw_arrow_u = np.zeros(nt, dtype=np.float64)
        raw_arrow_v = np.zeros(nt, dtype=np.float64)
        pre_arrow_u = np.zeros(nt, dtype=np.float64)
        pre_arrow_v = np.zeros(nt, dtype=np.float64)

        finite_raw_ab = np.isfinite(raw_a_values) & np.isfinite(raw_b_values)
        finite_pre_ab = np.isfinite(a_values) & np.isfinite(b_values)

        raw_arrow_u[finite_raw_ab] = (
            raw_a_values[finite_raw_ab] / slope_ref
        ) * max_arrow_length_pix
        raw_arrow_v[finite_raw_ab] = (
            raw_b_values[finite_raw_ab] / slope_ref
        ) * max_arrow_length_pix

        pre_arrow_u[finite_pre_ab] = (
            a_values[finite_pre_ab] / slope_ref
        ) * max_arrow_length_pix
        pre_arrow_v[finite_pre_ab] = (
            b_values[finite_pre_ab] / slope_ref
        ) * max_arrow_length_pix

        vmax = np.nanpercentile(np.abs(wave_cube), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        vmin = -vmax

        preprocessed_frame_0 = preprocess_frame_for_plane_fit(
            wave_cube[0],
            fit_radius_mask,
        )

        fig_anim, (ax_raw, ax_pre) = plt.subplots(
            nrows=1,
            ncols=2,
            figsize=(13, 6),
        )

        im_raw = ax_raw.imshow(
            wave_cube[0],
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )
        im_pre = ax_pre.imshow(
            preprocessed_frame_0,
            origin="upper",
            cmap=CMAP,
            vmin=vmin,
            vmax=vmax,
        )

        cbar_raw = fig_anim.colorbar(
            im_raw,
            ax=ax_raw,
            fraction=0.046,
            pad=0.04,
        )
        cbar_raw.set_label("Perturbation amplitude")

        cbar_pre = fig_anim.colorbar(
            im_pre,
            ax=ax_pre,
            fraction=0.046,
            pad=0.04,
        )
        cbar_pre.set_label("Perturbation amplitude")

        center_x = (nx - 1) / 2.0
        center_y = (ny - 1) / 2.0

        for ax_here in (ax_raw, ax_pre):
            fit_outline = Ellipse(
                xy=(center_x, center_y),
                width=2.0 * FIT_RADIUS_KM / DX_KM,
                height=2.0 * FIT_RADIUS_KM / DY_KM,
                fill=False,
                edgecolor="black",
                linestyle="--",
                linewidth=1.2,
                alpha=0.8,
                zorder=4,
            )
            ax_here.add_patch(fit_outline)
            ax_here.scatter(
                [center_x],
                [center_y],
                s=CENTER_MARKER_SIZE,
                c=CENTER_MARKER_COLOR,
                marker="o",
            )
            ax_here.set_xlabel("X pixel")
            ax_here.set_ylabel("Y pixel")

        raw_quiver = ax_raw.quiver(
            [center_x],
            [center_y],
            [raw_arrow_u[0]],
            [raw_arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        pre_quiver = ax_pre.quiver(
            [center_x],
            [center_y],
            [pre_arrow_u[0]],
            [pre_arrow_v[0]],
            angles="xy",
            scale_units="xy",
            scale=1,
            color=ARROW_COLOR,
            width=0.008,
            headwidth=4.5,
            headlength=6.0,
            headaxislength=5.5,
        )

        ax_raw.set_title("Raw data + raw slope vector")
        ax_pre.set_title("Preprocessed + preprocessed slope vector")

        title = fig_anim.suptitle("")

        raw_info = ax_raw.text(
            0.02,
            0.98,
            "",
            transform=ax_raw.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        pre_info = ax_pre.text(
            0.02,
            0.98,
            "",
            transform=ax_pre.transAxes,
            va="top",
            ha="left",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.8,
                edgecolor="black",
            ),
        )

        def update(frame_index):
            raw_frame = wave_cube[frame_index]
            preprocessed_frame = preprocess_frame_for_plane_fit(
                raw_frame,
                fit_radius_mask,
            )

            im_raw.set_data(raw_frame)
            im_pre.set_data(preprocessed_frame)

            raw_quiver.set_UVC(
                [raw_arrow_u[frame_index]],
                [raw_arrow_v[frame_index]],
            )
            pre_quiver.set_UVC(
                [pre_arrow_u[frame_index]],
                [pre_arrow_v[frame_index]],
            )

            local_time_text = local_datetimes[frame_index].strftime(
                LOCAL_TIME_EXACT_FORMAT
            )
            title.set_text(
                f"Raw vs Preprocessed Plane-fit Vector | "
                f"Frame {frame_index}/{nt - 1} | "
                f"t = {times_minutes[frame_index]:.1f} min | "
                f"Local time = {local_time_text}"
            )

            raw_info.set_text(
                f"Direction = {raw_angle_values_deg[frame_index]:+.1f}°\n"
                f"|s| = {raw_slope_values[frame_index]:.4g} amp/km\n"
                f"a = {raw_a_values[frame_index]:.4g}\n"
                f"b = {raw_b_values[frame_index]:.4g}"
            )
            pre_info.set_text(
                f"Direction = {angle_values_deg[frame_index]:+.1f}°\n"
                f"|s| = {slope_values[frame_index]:.4g} amp/km\n"
                f"a = {a_values[frame_index]:.4g}\n"
                f"b = {b_values[frame_index]:.4g}"
            )

            return (
                im_raw,
                im_pre,
                raw_quiver,
                pre_quiver,
                title,
                raw_info,
                pre_info,
            )

        ani = FuncAnimation(
            fig_anim,
            update,
            frames=nt,
            interval=1000 / FPS,
            blit=False,
        )

        fig_anim.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))

        animation_mp4_file = out_dir / "plane_fit_vector_animation.mp4"
        animation_gif_file = out_dir / "plane_fit_vector_animation.gif"

        try:
            writer = FFMpegWriter(
                fps=FPS,
                bitrate=3000,
            )
            ani.save(
                animation_mp4_file,
                writer=writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_mp4_file)
        except Exception as exc:
            print(
                "Could not save MP4 with ffmpeg. "
                "Falling back to GIF."
            )
            print(f"Reason: {exc}")

            gif_writer = PillowWriter(fps=FPS)
            ani.save(
                animation_gif_file,
                writer=gif_writer,
                dpi=DPI,
            )
            saved_animation_file = str(animation_gif_file)

        plt.close(fig_anim)

    print(f"Finished: {pkl_file.name}")
    print(f"Successful fits: {successful_fits}/{nt}")
    print(f"Output folder: {out_dir}")

    return (
        {
            "input_file": str(pkl_file),
            "internal_start_datetime": start_local_datetime.isoformat(sep=" "),
            "status": "processed",
            "output_folder": str(out_dir),
            "frame_count": nt,
            "successful_fits": successful_fits,
            "hourly_average_csv": str(hourly_average_csv_file),
            "message": (
                f"Animation: {saved_animation_file}"
                if MAKE_ANIMATION
                else "Animation disabled"
            ),
        },
        hourly_average_rows,
    )


# ==================================================
# Batch driver
# ==================================================
def discover_pkl_files():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(
            f"Input folder does not exist:\n{INPUT_ROOT}"
        )

    iterator = (
        INPUT_ROOT.rglob("*.pkl")
        if RECURSIVE_SEARCH
        else INPUT_ROOT.glob("*.pkl")
    )

    files = []
    output_root_resolved = OUTPUT_ROOT.resolve()

    for file_path in iterator:
        try:
            file_resolved = file_path.resolve()
            if (
                file_resolved == output_root_resolved
                or output_root_resolved in file_resolved.parents
            ):
                continue
        except OSError:
            pass

        files.append(file_path)

    return sorted(files)


def write_batch_summary(summary_rows):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary_file = OUTPUT_ROOT / "batch_processing_summary.csv"

    fieldnames = [
        "input_file",
        "internal_start_datetime",
        "status",
        "output_folder",
        "frame_count",
        "successful_fits",
        "hourly_average_csv",
        "message",
    ]

    with open(summary_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    return summary_file


def main():
    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"Input root: {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Target years: {sorted(TARGET_YEARS)}")
    print(f"Animations enabled: {MAKE_ANIMATION}")
    print(
        "Preprocessing diagnostics enabled: "
        f"{MAKE_PREPROCESSING_DIAGNOSTICS}"
    )

    # ==================================================
    # Single-file mode
    # ==================================================
    if PROCESS_SINGLE_FILE:
        single_file = Path(
            SINGLE_PKL_FILE
        ).expanduser()

        print("\n" + "=" * 80)
        print("RUN MODE: SINGLE FILE")
        print(f"Selected PKL: {single_file}")

        if not single_file.exists():
            raise FileNotFoundError(
                "PROCESS_SINGLE_FILE is True, but the selected PKL "
                "does not exist:\n"
                f"{single_file}\n\n"
                "Change SINGLE_PKL_FILE near the top of the script."
            )

        summary_rows = []

        try:
            (
                summary_row,
                hourly_rows,
            ) = process_one_pkl(
                single_file
            )

            summary_rows.append(
                summary_row
            )

        except Exception as exc:
            print(f"FAILED: {single_file}")
            print(
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(single_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            summary_file = write_batch_summary(
                summary_rows
            )
            print(
                f"Summary saved: {summary_file}"
            )

            if not CONTINUE_ON_ERROR:
                raise

            return

        finally:
            plt.close("all")
            gc.collect()

        summary_file = write_batch_summary(
            summary_rows
        )

        print("\n" + "=" * 80)
        print("Single-file processing finished.")
        print(f"Summary: {summary_file}")

        if (
            summary_rows
            and summary_rows[0]["status"] == "processed"
        ):
            print(
                "Output folder: "
                f"{summary_rows[0]['output_folder']}"
            )

        print(
            "\nWhen you are satisfied with this day's preprocessing and "
            "plane fit, set PROCESS_SINGLE_FILE = False to run the batch."
        )

        return

    # ==================================================
    # Batch mode
    # ==================================================
    print("\n" + "=" * 80)
    print("RUN MODE: BATCH")

    pkl_files = discover_pkl_files()

    print(
        f"PKL files found: {len(pkl_files)}"
    )

    if not pkl_files:
        print("No PKL files were found.")
        return

    summary_rows = []
    all_days_hourly_average_rows = []

    for (
        file_number,
        pkl_file,
    ) in enumerate(
        pkl_files,
        start=1,
    ):
        print(
            f"\nBatch file {file_number}/{len(pkl_files)}: "
            f"{pkl_file.name}"
        )

        try:
            (
                summary_row,
                hourly_rows,
            ) = process_one_pkl(
                pkl_file
            )

            summary_rows.append(
                summary_row
            )

            all_days_hourly_average_rows.extend(
                hourly_rows
            )

        except Exception as exc:
            print(f"FAILED: {pkl_file}")
            print(
                f"{type(exc).__name__}: {exc}"
            )
            traceback.print_exc()

            summary_rows.append(
                {
                    "input_file": str(pkl_file),
                    "internal_start_datetime": "",
                    "status": "failed",
                    "output_folder": "",
                    "frame_count": "",
                    "successful_fits": "",
                    "hourly_average_csv": "",
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

            if not CONTINUE_ON_ERROR:
                summary_file = write_batch_summary(
                    summary_rows
                )
                print(
                    f"Partial summary saved: {summary_file}"
                )
                raise

        finally:
            plt.close("all")
            gc.collect()

        write_batch_summary(
            summary_rows
        )

    summary_file = write_batch_summary(
        summary_rows
    )

    all_days_hourly_average_csv_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.csv"
    )

    all_days_hourly_average_plot_file = (
        OUTPUT_ROOT
        / f"all_days_hourly_average_points_"
          f"{HOURLY_AVERAGE_START_HOUR:02d}to"
          f"{HOURLY_AVERAGE_END_HOUR:02d}.png"
    )

    save_all_days_hourly_average_csv(
        all_days_hourly_average_rows,
        all_days_hourly_average_csv_file,
    )

    save_all_days_hourly_average_plot(
        all_days_hourly_average_rows,
        all_days_hourly_average_plot_file,
    )

    processed_count = sum(
        row["status"] == "processed"
        for row in summary_rows
    )

    skipped_count = sum(
        str(row["status"]).startswith(
            "skipped"
        )
        for row in summary_rows
    )

    failed_count = sum(
        row["status"] == "failed"
        for row in summary_rows
    )

    print("\n" + "=" * 80)
    print("Batch finished.")
    print(f"Processed: {processed_count}")
    print(f"Skipped:   {skipped_count}")
    print(f"Failed:    {failed_count}")
    print(f"Summary:   {summary_file}")

    print(
        "All-days hourly-average CSV:  "
        f"{all_days_hourly_average_csv_file}"
    )

    print(
        "All-days hourly-average plot: "
        f"{all_days_hourly_average_plot_file}"
    )

    print(
        "Hourly-average local-hour window: "
        f"{HOURLY_AVERAGE_START_HOUR:02d}:00 to "
        f"{HOURLY_AVERAGE_END_HOUR:02d}:00"
    )


if __name__ == "__main__":
    main()


Input root: /Users/anhphan/Downloads/pkl
Output root: /Users/anhphan/Downloads/pkl/BLO_plane_fit_batch_outputs
Target years: [2024, 2025, 2026]
Animations enabled: True
Preprocessing diagnostics enabled: True

RUN MODE: SINGLE FILE
Selected PKL: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl

Reading: /Users/anhphan/Downloads/pkl/processed_data_full_03_25_2025.pkl
Loaded entry: Perturbation Images
Perturbation cube shape: (280, 1000, 1000)
PKL interval start datetime: 2025-03-25 20:29:27
First frame-center datetime: 2025-03-25 20:30:27
Plane-fit region: radius <= 300.0 km (282792 pixels before other validity filters)
Spatial preprocessing: Gaussian-background peak replacement (sigma = 40 km, threshold = 2.5 MAD-sigma, mask dilation = 4 km)
Fitting planes to raw and preprocessed frames...
  Processed frame 1/280
  Processed frame 25/280
  Processed frame 50/280
  Processed frame 75/280
  Processed frame 100/280
  Processed frame 125/280
  Processed frame 150/280
  Proce